In [ ]:
# === Importação de Biliotecas ===
import os
import gc
import math
import time
import math
import time
#import tqdm
from tqdm.notebook import tqdm
from sklearn.metrics import auc
import zipfile
import subprocess
import numpy as np
import pandas as pd
from fpdf import FPDF
import seaborn as sns
import miceforest as mf
import matplotlib.pyplot as plt
from scipy.stats import bootstrap
from sklearn.metrics import log_loss
import matplotlib.patches as mpatches
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, balanced_accuracy_score, roc_auc_score, roc_curve, confusion_matrix

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"X does not have valid feature names, but .* was fitted with feature names",
    category=UserWarning,
)

import os
os.environ["PYTHONWARNINGS"] = (
     "ignore:X does not have valid feature names, but .* was fitted with feature names:UserWarning"
 )

In [ ]:
from joblib import load

# === Parametros Genéricos ===
# === Definição de Parâmetros - Obrigatório ser informados pelo usuario
BASE_PATH = "SEPSIS_SHAP_RANDOM" 
FILE_IN = Path('../SEPSIS/data/processed/exams.csv')

os.makedirs(BASE_PATH, exist_ok=True)

METHOD = "" #Informado na célula de cada método

direcao_esperada = {
    # -------------------------------------------------
    # 1) Variáveis clínicas / laboratoriais (fisiológicas)
    # -------------------------------------------------

    # Peso de admissão: relação com sepse é bem indireta / não monotônica (obesidade, caquexia, etc.)
    "Admission Weight (Kg)": 0,

    # Marcadores metabólicos / ácido-base:
    "Anion gap": +1,          # gap aniônico ↑ geralmente indica acidose metabólica → pior quadro / maior gravidade
    "HCO3 (serum)": -1,       # bicarbonato ↑ costuma indicar menor acidose → melhor quadro (↓ prob. sepse grave)

    # Função renal / catabolismo:
    "BUN": +1,                # ureia ↑ → disfunção renal / catabolismo → pior prognóstico
    "Creatinine (serum)": +1, # creatinina ↑ → lesão renal aguda / crônica → maior gravidade

    # Eletrólitos e minerais:
    "Calcium non-ionized": -1,    # em sepse costuma cair; cálcio mais alto tende a ser menos grave (bem aproximado)
    "Chloride (serum)": 0,        # associação com sepse é muito dependente de reposição volêmica/fluídos
    "Magnesium": 0,               # hipo/hipermagnesemia → relação complexa, não estritamente monotônica
    "Phosphorous": 0,             # idem: muito dependente de função renal / nutrição
    "Potassium (serum)": 0,       # relação nitidamente em U (hipo e hiper são ruins) → melhor marcar 0
    "Sodium (serum)": 0,          # hipo/hipernatremia → ambos prejudiciais → direção não monotônica

    # Glicose:
    "Glucose (serum)": +1,        # hiperglicemia de estresse / diabetes → em geral associada a piores desfechos

    # Hemograma / coagulação:
    "Hemoglobin": 0,              # anemia vs. hemoconcentração → relação bem complexa, evitamos direção fixa
    "INR": +1,                    # INR ↑ → coagulopatia / disfunção hepática → pior prognóstico
    "Platelet Count": -1,         # plaquetas ↓ são típicas na sepse; mais plaquetas tende a ser melhor
    "WBC": +1,                    # pensando “de normal para leucocitose” → inflamação/infeção ↑ (U-shape com leucopenia, mas usamos +1 como aproximação)

    # Sinais vitais:
    "Heart Rate": +1,             # taquicardia → critério SIRS / sepse
    "Respiratory Rate": +1,       # taquipneia → critério de sepse / gravidade respiratória
    "Temperature Fahrenheit": +1, # febre é típico de infecção (sabendo que hipotermia também é grave, mas mantemos +1 como tendência)

    # Pressão arterial:
    "Non Invasive Blood Pressure systolic": -1,   # quanto maior (até normal), menor prob. choque séptico
    "Non Invasive Blood Pressure diastolic": -1,  # idem (↓ PA diastólica → pior perfusão)

    # Oxigenação:
    "O2 saturation pulseoxymetry": -1,    # saturação ↑ → melhor oxigenação → menor gravidade
    "SpO2 Desat Limit": 0,               # limite de desaturação é típico de configuração / contexto, não fisiologia direta

    # Coagulação (tempo parcial de tromboplastina):
    "PTT": +1,                # PTT ↑ → coagulopatia / CID → pior prognóstico

    # Idade:
    "anchor_age": +1,         # idade ↑ → maior risco de infecção/sepse e piores desfechos

    # -------------------------------------------------
    # 2) Variáveis de raça / sexo — NÃO impor direção
    # -------------------------------------------------
    # Por questões éticas e de justiça algorítmica, é mais seguro
    # não impor direção causal nesses atributos (deixar 0) e usar
    # directional soundness só em variáveis clínicas.

    #"race_black": 0,
    #"race_hispanic/latino": 0,
    #"race_other": 0,
    #"race_unknown": 0,
    #"race_white": 0,

    "gender_M": 0,            # diferença de risco entre sexos existe em estudos, mas é melhor não tratar
                              # como “mais é pior” ou “mais é melhor” de forma rígida nesse tipo de métrica.
}


PATH_CSV = Path('../SEPSIS/model_reports/random_forest/basico/csv/')
#X_test_basic_full.csv
#X_train_basic_full.csv


X_train = pd.read_csv(PATH_CSV / 'X_train_basic_full.csv')  
X_test = pd.read_csv(PATH_CSV / 'X_test_basic_full.csv')  

X_train = X_train.drop(X_train.columns[0], axis=1)
X_test = X_test.drop(X_test.columns[0], axis=1)
X_train.rename(columns={"stay_id": "id"}, inplace=True)
X_test.rename(columns={"stay_id": "id"}, inplace=True)  

X = pd.concat([X_train, X_test], ignore_index=True).sort_values(by="id").reset_index(drop=True)
X_ = X.copy()

EXCLUDE_COLS = ['stay_id', 'y_pred', 'y_proba', 'y','septic']
EXCLUDE_COLS = ['id','ID', 'patient_id', 'pid','septic','stay_id', 'race_white', 'race_unknown', 'race_black', 'race_hispanic/latino', 'race_other', 'y','y_proba', 'y_pred']
model = load("../SEPSIS/common/models/random_forest/v1/rf_basic_ts10_rs42.pkl") 

# ======================================================================
# 1. PREPARAÇÃO INICIAL E CONFIGURAÇÃO
# ======================================================================

# Identifica as colunas que DEVEM ser processadas
FEATURES = [c for c in X.columns if c not in EXCLUDE_COLS]

# Seleciona apenas as features relevantes para o processamento
X_processed_raw = X[FEATURES].copy()
X_train_processed_raw = X_train[FEATURES].copy()

# Detecta tipos de dados nas features selecionadas
num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(X_processed_raw[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

# ======================================================================
# 2. PIPELINE DE PRÉ-PROCESSAMENTO
# ======================================================================

# Pipelines para dados numéricos e categóricos (exatamente os mesmos do seu código)
num_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
])
cat_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1
    ))
])

# ColumnTransformer para aplicar os pipelines corretos a cada tipo de coluna
preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop" # Garante que apenas as colunas em FEATURES sejam mantidas
)

# ======================================================================
# 3. APLICAÇÃO DO PRÉ-PROCESSAMENTO
# ======================================================================

# Aplicar o fit_transform se este for o conjunto de treino
# Se você já treinou seu preprocessador em outro lugar, use apenas .transform()

# Exemplo usando fit_transform no seu DataFrame X de entrada:
X_ = preprocess.fit_transform(X_processed_raw)
X_train_ = preprocess.transform(X_train_processed_raw)

# O resultado X_ é um array numpy com seus dados prontos,
# com todas as nulos tratados e categóricas transformadas em números.

try:
    # sklearn >= 1.0 geralmente tem isso
    col_names = preprocess.get_feature_names_out()
    # opcional: remover prefixos "num__" / "cat__"
    col_names = [name.split("__", 1)[1] if "__" in name else name
                 for name in col_names]
except Exception:
    # fallback simples: como não houve expansão (ordinal), nº de colunas é o mesmo
    col_names = num_cols + cat_cols

# 1) DataFrames só com as features processadas
X_df = pd.DataFrame(X_, columns=col_names, index=X_processed_raw.index)
X_train_df = pd.DataFrame(X_train_, columns=col_names, index=X_train_processed_raw.index)

X_df = X_df[model.feature_names_in_]

# 2) Anexar id e y preservando alinhamento pelo índice
#    (X e X_train são os DataFrames "brutos", com id / y)
X_df = X_df.copy()
X_df["id"] = X.loc[X_df.index, "id"].values
if "y" in X.columns:
    X_df["y"] = X.loc[X_df.index, "y"].values

X_train_df = X_train_df.copy()
X_train_df["id"] = X_train.loc[X_train_df.index, "id"].values
if "y" in X_train.columns:
    X_train_df["y"] = X_train.loc[X_train_df.index, "y"].values

# 3) Colocar id e y nas primeiras posições, features depois
feature_cols = [c for c in X_df.columns if c not in ("id", "y")]
X_df = X_df[["id", "y"] + feature_cols]

# garantir que X_train_df tenha EXATAMENTE as mesmas features na mesma ordem
X_train_df = X_train_df[["id", "y"] + feature_cols]

# (Opcional) pequenas sanidades:
assert list(X_df.columns) == ["id", "y"] + feature_cols
assert list(X_train_df.columns) == ["id", "y"] + feature_cols

#ajsute final 
X_ = X_df.copy()

ENTRADAS_SELECIONADAS = pd.read_csv(Path('../SEPSIS/clusters_reports/random_forest/csv/entradas_selecionadas_final_id_only.csv'))
ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS.T.squeeze().to_list() 



In [ ]:
# =================== SHAP — seleção por ID e features do treino ===================

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import math
import os
import gc
from joblib import Parallel, delayed

# ---------------- CONFIG PARA BATCH / PARALELISMO ----------------
N_JOBS_SHAP     = 10        # 1 = sequencial, >1 = paraleliza batches com loky
BATCH_SIZE_SHAP = 250     # nº de linhas por batch de SHAP em X_plot / X_feat_sel
BATCH_SIZE_SHAP_SEL = 35
# ---------------- FUNÇÃO AUXILIAR: SHAP EM BATCHES ----------------
def shap_values_in_batches(explainer, X_data, pos_idx, 
                           batch_size=1000, n_jobs=1):
    """
    Calcula SHAP em batches para X_data, retornando a matriz
    (n_amostras, n_features) da classe positiva (pos_idx).
    """
    n = X_data.shape[0]
    idx_batches = [
        (start, min(start + batch_size, n))
        for start in range(0, n, batch_size)
    ]

    def _compute_batch(start, end):
        X_batch = X_data.iloc[start:end]
        sv_batch = explainer.shap_values(X_batch)

        if isinstance(sv_batch, (list, tuple)):
            M_batch = np.asarray(sv_batch[pos_idx], dtype=float)
        elif isinstance(sv_batch, np.ndarray) and sv_batch.ndim == 3:
            M_batch = sv_batch[:, :, pos_idx].astype(float)
        elif isinstance(sv_batch, np.ndarray) and sv_batch.ndim == 2:
            M_batch = sv_batch.astype(float)
        else:
            if hasattr(shap, "Explanation") and isinstance(sv_batch, shap.Explanation):
                M_batch = np.asarray(sv_batch.values, dtype=float)
            else:
                raise RuntimeError("Formato de shap_values em batch não reconhecido.")
        return M_batch

    if n_jobs == 1:
        mats = []
        for (s, e) in idx_batches:
            M = _compute_batch(s, e)
            mats.append(M)
            gc.collect()
    else:
        mats = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            batch_size=1
        )(delayed(_compute_batch)(s, e) for (s, e) in idx_batches)
        gc.collect()

    return np.vstack(mats)


# --- 1) Selecionar subconjunto por ID (preserva ordem de ENTRADAS_SELECIONADAS) ---
if "id" not in X_test.columns:
    raise RuntimeError("Coluna 'id' ausente em X_test.")

ids = pd.Index(ENTRADAS_SELECIONADAS)
X_sel = X[X["id"].isin(ids)].copy()
X_sel["__order__"] = pd.Categorical(X_sel["id"], categories=list(ids), ordered=True)
X_sel = X_sel.sort_values("__order__").drop(columns="__order__")

# --- 2) y_sel alinhado por ID (usa a própria coluna 'y' do X) ---
if "y" not in X.columns:
    raise RuntimeError("Coluna 'y' não encontrada em X.")
y_all = X[["id", "y"]].copy()
y_sel = y_all[y_all["id"].isin(ids)].copy()
y_sel["__order__"] = pd.Categorical(y_sel["id"], categories=list(ids), ordered=True)
y_sel = y_sel.sort_values("__order__").drop(columns="__order__")
y_sel_vec = y_sel["y"].astype(int).to_numpy()

# --- 3) Features iguais às do treino (ordem e nomes) ---
try:
    feat_cols = list(model.feature_names_in_)
except Exception:
    feat_cols = [c for c in X.columns if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(X[c])]

X_feat_sel  = X_sel[feat_cols].copy()
X_feat_test = X[feat_cols].copy()   # se X_test for menor, pode trocar por X_test[feat_cols]

# --- 4) Índice da classe positiva e Explainer shap_tree_explainer(probability) ---
try:
    classes = list(getattr(model, "classes_", [0, 1]))
except Exception:
    classes = [0, 1]
idx_pos = classes.index(1) if 1 in classes else int(np.argmax(classes))

shap_tree_explainer = shap.TreeExplainer(model, data=X_feat_sel, model_output="probability")

# --- 5) SHAP no subset (background = X_feat_sel) EM BATCHES + JOBS ---
# antes era: sv_sel = shap_tree_explainer.shap_values(X_feat_sel)
#            shap_mat_sel, base_value = _extract_mat_and_base(...)
# agora usamos a função em batch e pegamos o base_value direto do explainer

# batch_size não precisa ser grande aqui (615 instâncias); uso o mínimo com limite:
batch_sel = min(BATCH_SIZE_SHAP, BATCH_SIZE_SHAP_SEL)

shap_mat_sel = shap_values_in_batches(
    shap_tree_explainer,
    X_feat_sel,
    pos_idx=idx_pos,
    batch_size=batch_sel,
    n_jobs=N_JOBS_SHAP
)

# base_value (intercepto) vem do expected_value do explainer
ev = shap_tree_explainer.expected_value
if isinstance(ev, (list, np.ndarray)):
    base_value = ev[idx_pos]
else:
    base_value = ev

# --- 6) DataFrame final + reconciliação ---
df_shap = pd.DataFrame(shap_mat_sel, columns=feat_cols)

if isinstance(base_value, (np.ndarray, pd.Series, list)) and np.ndim(base_value) == 1:
    df_shap["intercepto"] = np.asarray(base_value, dtype=float)
else:
    df_shap["intercepto"] = float(base_value)

df_shap["soma_SHAP"]   = df_shap[feat_cols].sum(axis=1) + df_shap["intercepto"]
df_shap["prob_modelo"] = model.predict_proba(X_feat_sel)[:, idx_pos]
df_shap["erro_aprox"]  = (df_shap["prob_modelo"] - df_shap["soma_SHAP"]).abs()

# ID e rótulo humano
df_shap.insert(0, "id", X_sel["id"].to_numpy())
df_shap["tamanho_real"] = pd.Series(y_sel_vec).map({1: "grande", 0: "pequeno"})

# Exporta
os.makedirs(BASE_PATH, exist_ok=True)
out_xlsx = os.path.join(BASE_PATH, "interpretacao_shap_rf_SELECIONADAS_bg_Xsel.xlsx")
df_shap.to_excel(out_xlsx, index=False)
print(f"✅ Gerado: {out_xlsx}")

# --- 7) BAR (mean |SHAP|) no subset selecionado ---
mean_abs = np.nanmean(np.abs(shap_mat_sel), axis=0)
order    = np.argsort(mean_abs)[::-1]
top_bar  = order[: min(20, len(order))]
plt.figure(figsize=(10, max(6, 0.45*len(top_bar))))
plt.barh(np.arange(len(top_bar)), mean_abs[top_bar][::-1])
plt.yticks(np.arange(len(top_bar)), np.array(feat_cols)[top_bar][::-1], fontsize=10)
plt.xlabel("mean(|SHAP|)")
plt.title("Importância global (subset selecionado)")
plt.tight_layout(); plt.show()

# --- 8) DEFINIR UMA AMOSTRA DO TESTE PARA PLOTS ---
N_PLOT = 5000  # pode ser 2000, 3000... ajuste conforme recurso
if len(X_feat_test) > N_PLOT:
    X_plot = X_feat_test.sample(N_PLOT, random_state=42)
else:
    X_plot = X_feat_test

# --- 8) SHAP PARA X_plot EM BATCHES (com loky / gc) ---
shap_mat_plot = shap_values_in_batches(
    shap_tree_explainer,
    X_plot,
    pos_idx=idx_pos,
    batch_size=BATCH_SIZE_SHAP,
    n_jobs=N_JOBS_SHAP
)

# --- 8) BEESWARM COM AMOSTRA ---
plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_mat_plot,
    X_plot,
    plot_type="dot",
    max_display=min(20, shap_mat_plot.shape[1]),
    show=True
)

# --- 9) DEPENDENCE mini-múltiplos (top-9) USANDO A MESMA AMOSTRA ---
mean_abs_dep = np.nanmean(np.abs(shap_mat_plot), axis=0)
order_dep    = np.argsort(mean_abs_dep)[::-1][:9]
top_feats    = list(np.array(feat_cols)[order_dep])

proba = model.predict_proba(X_plot)[:, idx_pos]
yhat  = (proba >= 0.50).astype(int)

n = len(top_feats)
ncols = 3
nrows = int(math.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.3*nrows), squeeze=False)
COL_LOW, COL_HIGH = "#0072B2", "#D55E00"

for k, feat in enumerate(top_feats):
    r, c = divmod(k, ncols)
    ax = axes[r, c]
    j = X_plot.columns.get_loc(feat)
    x = np.asarray(X_plot.iloc[:, j], dtype=float)
    y = np.asarray(shap_mat_plot[:, j], dtype=float)
    colors = np.where(yhat == 1, COL_HIGH, COL_LOW)
    ax.scatter(x, y, s=9, c=colors, alpha=0.55, edgecolors="none")
    ax.axhline(0, color="#999", lw=0.8)
    ax.axvline(np.nanmedian(x), color="#ddd", lw=0.8, ls=":")
    ax.set_title(feat, fontsize=11, pad=4)
    ax.set_xlabel(feat)
    ax.set_ylabel(f"SHAP de {feat}")
    ax.grid(False)

for k in range(n, nrows * ncols):
    r, c = divmod(k, ncols)
    axes[r, c].axis("off")

handles = [
    Patch(color=COL_LOW,  label="Baixo risco (p<0.50)"),
    Patch(color=COL_HIGH, label="Alto risco (p≥0.50)")
]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.98, 0.98), frameon=True)
fig.tight_layout()
plt.show()


In [ ]:
# === Shapley Interpretabilidade [Análises Complementares do SHAP] ===
# Requisitos já definidos no passo SHAP:
#   df_shap, shap_mat_sel, X_sel, X_feat_sel, feat_cols, idx_pos

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import bootstrap
from sklearn.metrics import r2_score
import seaborn as sns
import shap

# ---------- 0) Amarrações ----------
# >>> Usar o SHAP já calculado (shap_mat_sel), SEM recalcular
shap_vals = shap_mat_sel.copy()        # (n_sel, n_features)
feat_cols = list(X_feat_sel.columns)   # garantir alinhamento
assert shap_vals.shape[1] == len(feat_cols), \
    "shap_mat_sel e feat_cols com dimensões incompatíveis."

# ---------- 1) Histograma do erro SHAP vs Modelo ----------
plt.figure(figsize=(10, 5))
plt.hist(df_shap["erro_aprox"], bins=20, edgecolor='black')
plt.title(f"Erro de Aproximação do SHAP vs. Modelo (erro médio = {df_shap['erro_aprox'].mean():.4f})")
plt.xlabel("Erro de Aproximação")
plt.ylabel("Frequência")
plt.grid(True)
plt.tight_layout()
plt.show()

# ---------- 2) Covariância + IC 95% ----------
X_cov  = df_shap[feat_cols]
y_real = df_shap['prob_modelo']

covariancias = []
for feature in X_cov.columns:
    x = X_cov[feature].values
    y = y_real.values

    cov = np.cov(x, y)[0, 1]

    def cov_func(x, y):
        return np.cov(x, y)[0, 1]

    ic = bootstrap((x, y), cov_func, confidence_level=0.95,
                   n_resamples=1000, method='percentile')

    covariancias.append({
        'feature': feature,
        'covariancia': cov,
        'IC_low': ic.confidence_interval.low,
        'IC_high': ic.confidence_interval.high
    })

df_cov = pd.DataFrame(covariancias)

# ---------- 3) Gráfico com IC 95% ----------
df_cov_sorted = df_cov.sort_values(by="covariancia", ascending=False)
err_lower = np.abs(df_cov_sorted["covariancia"] - df_cov_sorted["IC_low"])
err_upper = np.abs(df_cov_sorted["IC_high"] - df_cov_sorted["covariancia"])

plt.figure(figsize=(12, 6))
plt.bar(df_cov_sorted["feature"], df_cov_sorted["covariancia"],
        yerr=[err_lower, err_upper],
        capsize=5, color="skyblue", edgecolor="black")
plt.xticks(rotation=90)
plt.ylabel("Covariância")
plt.title("Covariância das Features com Probabilidade (IC 95%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# ---------- 4) Dispersão com R² ----------
real  = df_shap["prob_modelo"]
aprox = df_shap["soma_SHAP"]
r2 = r2_score(real, aprox)

plt.figure(figsize=(6, 6))
sns.regplot(x=real, y=aprox, line_kws={"color": "red"})
plt.xlabel("Probabilidade do Modelo")
plt.ylabel("Aproximação SHAP")
plt.title(f"Dispersão SHAP x Modelo\nR² = {r2:.4f}")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# ---------- 5) RMSE Visual ----------
erros = aprox.values - real.values
rmse = np.sqrt(np.mean(erros ** 2))

plt.figure(figsize=(10, 6))
plt.scatter(real, aprox, color="blue", label="Aproximações")
plt.plot([0, 1], [0, 1], color="green", linestyle="--", label="Ideal")
for i in range(len(real)):
    plt.plot([real.iloc[i], real.iloc[i]], [real.iloc[i], aprox.iloc[i]], color="red", alpha=0.4)
plt.title(f"Dispersão SHAP vs Modelo com RMSE Visual\nRMSE = {rmse:.4f}")
plt.xlabel("Probabilidade do Modelo")
plt.ylabel("Aproximação SHAP")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

# ---------- 6) WATERFALL para TODAS as instâncias ----------
def limpar_waterfall(ax=None):
    ax = ax or plt.gca()
    ax.grid(False)
    ax.set_axisbelow(False)
    for gl in ax.get_xgridlines() + ax.get_ygridlines():
        gl.set_visible(False)
    for ln in list(ax.lines):
        ln.set_visible(False)
    for p in ax.patches:
        try:
            p.set_linewidth(0.0)
            p.set_edgecolor(p.get_facecolor())
        except Exception:
            pass
    for col in ax.collections:
        try:
            col.set_linewidth(0.0)
            col.set_edgecolor(col.get_facecolor())
        except Exception:
            pass
    ax.set_facecolor("white")
    for sp in ax.spines.values():
        sp.set_visible(False)

mpl.rcParams['axes.grid'] = False
mpl.rcParams['patch.antialiased'] = False
mpl.rcParams['patch.edgecolor'] = 'none'

def _base_for_row(i):
    base_col = df_shap["intercepto"].iloc[i]
    try:
        return float(base_col) if np.ndim(base_col) == 0 else float(np.asarray(base_col).item())
    except Exception:
        return float(base_col)

explanations = []
for i in range(len(X_sel)):
    explanations.append(
        shap.Explanation(
            values        = shap_vals[i],
            base_values   = _base_for_row(i),
            data          = X_feat_sel.iloc[i].values,
            feature_names = feat_cols,
        )
    )

# Desenhar todas (pode ser lento)
for idx_plot, exp_i in enumerate(explanations):
    plt.figure(figsize=(6, 4.5))
    shap.plots.waterfall(exp_i, max_display=12, show=False)
    limpar_waterfall()
    try:
        inst_id = X_sel["id"].iloc[idx_plot]
        plt.gcf().suptitle(f"Instância local {idx_plot} | id {inst_id}",
                           fontsize=10, y=0.98)
    except Exception:
        plt.gcf().suptitle(f"Instância local {idx_plot}", fontsize=10, y=0.98)
    plt.tight_layout()
    plt.show()


In [ ]:
# ===  Shapley Interpretabilidade [Funções de Calculo das Métricas]===
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from lime.lime_tabular import LimeTabularExplainer
import re
import numpy as np
from sklearn.metrics import mean_squared_error
from typing import Optional, Union, Iterable
from scipy.stats import kendalltau, spearmanr, pearsonr
import shap
import pandas as pd 
import time 

def calcular_ic95_percentual(valores):
    # Média e desvio padrão
    media = np.mean(valores)
    std = np.std(valores, ddof=1)
    n = len(valores)
    # Erro padrão
    erro_padrao = std / np.sqrt(n)
    # IC 95% com distribuição normal (Z=1.96)
    ic_low = media - 1.96 * erro_padrao
    ic_high = media + 1.96 * erro_padrao
    return ic_low, ic_high

def formatar_ic_percentual(valor_central, limite_inferior, limite_superior):
    return f"{valor_central:.1f}% [{limite_inferior:.1f}%, {limite_superior:.1f}%]"


# ------------------------- utilidades fidelity -------------------------

def _ensure_row_df_fidelity(x_row, cols):
    """Aceita Series/1D array/1xD DataFrame e devolve DataFrame 1xD com colunas=cols."""
    if isinstance(x_row, pd.DataFrame):
        return x_row[cols].iloc[[0]]
    if isinstance(x_row, pd.Series):
        return pd.DataFrame([x_row.values], columns=cols)
    a = np.asarray(x_row, dtype=float).ravel()
    return pd.DataFrame([a], columns=cols)

def _phi_from_shap_output_fidelity(out, explainer, pos_idx):
    """
    Normaliza os formatos do SHAP para vetor (d,) da classe pos_idx.
    """
    # list/tuple por classe
    if isinstance(out, (list, tuple)):
        return np.asarray(out[pos_idx], float).ravel()
    # Explanation novo
    try:
        if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
            V = np.asarray(out.values, float)
            return V.reshape(-1)  # 1xD -> D
    except Exception:
        pass
    # np.ndarray 3D (n,d,k) ou 2D (n,d)
    A = np.asarray(out)
    if A.ndim == 3:
        return A[0, :, pos_idx].astype(float).ravel()
    if A.ndim == 2:
        if A.shape[0] == 1:   # 1xD
            return A[0].astype(float).ravel()
        raise ValueError(f"SHAP 2D ambíguo {A.shape}; forneça apenas 1 instância.")
    raise ValueError("Formato de shap_values não reconhecido.")

def _to_target_fidelity(prob, target):
    """Converte prob -> target ('prob' ou 'logit')."""
    EPS = 1e-12
    p = np.clip(prob, EPS, 1 - EPS)
    if target == "logit":
        return np.log(p / (1 - p))
    return p

def _from_target_fidelity(val, target):
    """Converte target -> probabilidade."""
    if target == "logit":
        # sigmoid
        return 1.0 / (1.0 + np.exp(-val))
    return np.clip(val, 1e-12, 1 - 1e-12)

# ------------------------- versão completa (Yeh et al., 2019) -------------------------

def calcula_fidelity_shapley(
    *,
    x_row_df,                 # 1xD DataFrame (ou Series/array; será normalizado)
    model,                    # modelo com predict_proba
    X_ref: pd.DataFrame,      # para baseline/limites
    background: pd.DataFrame = None,  # background SHAP (recomendado: X_train ou X_sel)
    drop_cols=('classe','y'),
    pos_class=1,
    n_masks: int = 2048,      # nº de máscaras para a expectativa
    mask_rate: float = 0.5,   # prob. de desligar cada feature
    baseline: str | dict | np.ndarray = "mean",  # 'mean'|'median'|'zero'|'background'|dict|np.array
    target: str = "prob",     # 'prob' ou 'logit'
    metric: str = "r2",       # 'r2' (canônica tipo R²) ou 'logloss' (skill score)
    random_state: int = 42,
    return_details: bool = True
):
    """
    Fidelity de SHAP no espírito de Yeh et al. (2019):
      - Com metric='r2': R² entre Δ_real e Δ_hat (= I^T φ).
      - Com metric='logloss': skill score de logloss entre f(x-I) real vs aproximado por φ.
    Retorna: (percentual, absoluto[, details])
    """
    rng = np.random.default_rng(random_state)
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_fidelity(x_row_df, feat_cols)

    # classe positiva (índice robusto)
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # Explainer e φ(x)
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")
    out = expl.shap_values(x1[feat_cols])
    phi = _phi_from_shap_output_fidelity(out, expl, pos_idx)  # (d,)

    # f(x) (na escala alvo)
    fx_prob = float(model.predict_proba(x1[feat_cols])[:, pos_idx][0])
    fx_t    = float(_to_target_fidelity(fx_prob, target))

    # baseline vetor
    if isinstance(baseline, np.ndarray):
        base_vec = baseline.astype(float).ravel()
        assert base_vec.size == len(feat_cols)
    elif isinstance(baseline, dict):
        base_vec = np.array([float(baseline.get(c, X_ref[c].astype(float).mean())) for c in feat_cols], float)
    elif baseline == "median":
        base_vec = X_ref[feat_cols].median().astype(float).values
    elif baseline == "zero":
        base_vec = np.zeros(len(feat_cols), float)
    elif baseline == "background":
        base_vec = None  # amostrar por linha a cada máscara
    else:  # 'mean' (default)
        base_vec = X_ref[feat_cols].mean().astype(float).values

    # amostras
    d = len(feat_cols)
    residuals, delta_real, delta_hat = [], [], []
    ll_terms, ll_naive = [], []
    for _ in range(int(n_masks)):
        # máscara I ~ Bernoulli(mask_rate)^d, garantindo pelo menos 1 desligada
        I = rng.binomial(1, mask_rate, size=d).astype(bool)
        if not I.any():
            I[rng.integers(0, d)] = True

        # x_{-I}: substitui colunas mascaradas pela baseline
        if isinstance(base_vec, np.ndarray):
            xm = x1[feat_cols].to_numpy().copy()
            xm[0, I] = base_vec[I]
        else:
            # baseline='background' -> amostra 1 linha do bg a cada máscara
            b = bg.sample(n=1, random_state=rng.integers(0, 2**31 - 1)).to_numpy()
            xm = x1[feat_cols].to_numpy().copy()
            xm[0, I] = b[0, I]
        xm_df = pd.DataFrame(xm, columns=feat_cols)

        # f(x_{-I})
        p_minus = float(model.predict_proba(xm_df)[:, pos_idx][0])
        ft_minus = float(_to_target_fidelity(p_minus, target))

        # Δ real e Δ chapéu
        d_real = fx_t - ft_minus
        d_hat  = float(np.dot(I.astype(float), phi))

        residuals.append((d_hat - d_real))
        delta_real.append(d_real)
        delta_hat.append(d_hat)

        if metric.lower() == "logloss":
            p_hat_minus = float(_from_target_fidelity(fx_t - d_hat, target))
            # LL(p_true, p_hat)
            eps = 1e-12
            p_true = np.clip(p_minus, eps, 1 - eps)
            p_pred = np.clip(p_hat_minus, eps, 1 - eps)
            ll = -(p_true*np.log(p_pred) + (1-p_true)*np.log(1-p_pred))
            ll_terms.append(ll)
            # naive: prever p_hat_minus = f(x) (sem “ligar/desligar” nada)
            ll_naive.append(-(p_true*np.log(fx_prob) + (1-p_true)*np.log(1-fx_prob)))

    residuals = np.asarray(residuals, float)
    delta_real = np.asarray(delta_real, float)
    delta_hat  = np.asarray(delta_hat,  float)

    if metric.lower() == "r2":
        mse = float(np.mean(residuals**2))
        var = float(np.var(delta_real))
        r2  = 1.0 if var < 1e-12 else (1.0 - (mse / var))
        r2  = float(np.clip(r2, 0.0, 1.0))
        out_main = (100*r2, r2)  # percent, abs
        details = {
            "metric": "r2",
            "mse": mse, "var_delta": var,
            "fx_prob": fx_prob, "phi": phi,
            "delta_real": delta_real, "delta_hat": delta_hat,
        }
    else:  # 'logloss'
        ll_mean    = float(np.mean(ll_terms)) if ll_terms else np.nan
        ll_mean_nv = float(np.mean(ll_naive)) if ll_naive else np.nan
        # skill score (>=0 melhor)
        skill = float(np.clip(1.0 - (ll_mean / (ll_mean_nv + 1e-12)), 0.0, 1.0))
        out_main = (100*skill, skill)  # percent, abs (skill)
        details = {
            "metric": "logloss",
            "ll_mean": ll_mean,
            "ll_naive": ll_mean_nv,
            "fx_prob": fx_prob, "phi": phi,
            "delta_real": delta_real, "delta_hat": delta_hat,
        }

    if return_details:
        return (*out_main, details)
    return out_main

def calcula_infidelity(
    model,
    x_row,                      # np.array 1D da instância (mesma ordem de feature_names)
    feature_names,              # lista de nomes das colunas
    pos_class=1,                # rótulo da classe positiva em model.classes_
    target="prob",              # "prob" ou "logit"
    phi=None,                   # vetor SHAP da instância (d,). Se None, será calculado.
    background=None,            # DataFrame para TreeExplainer (se phi=None) e p/ baseline
    baseline="median",          # "median" | "mean" | np.ndarray (1,d)
    n_masks=2048,               # nº de subconjuntos I
    p_mask=0.5,                 # prob de “desligar” uma feature em cada máscara
    random_state=42,
    lower_better: bool = False  # <<< quando True, retorna % como "menor=melhor"
):
    """
    Infidelity no sentido de Yeh et al. (2019):
      - infidelity_abs = E[(Δpred-Δreal)^2]      (menor = melhor)
      - infidelity_pct = 100*max(0, R²_Δ)        (maior = melhor)
      - se lower_better=True: devolve 100 - infidelity_pct.
    """
    import numpy as np
    import pandas as pd
    import shap

    EPS = 1e-12
    rng = np.random.default_rng(random_state)
    d = len(feature_names)
    x_row = np.asarray(x_row, dtype=float).reshape(1, -1)

    # ---- 0) índice robusto da classe positiva ----
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # ---- 1) Prob/logit helpers
    def _to_prob(df_mat):
        proba = model.predict_proba(pd.DataFrame(df_mat, columns=feature_names))[:, pos_idx]
        return np.clip(proba, EPS, 1 - EPS)

    def _to_target(p):
        if target.lower() == "logit":
            return np.log(p/(1.0 - p))
        return p

    # ---- 2) SHAP φ(x) (se não veio pronto)
    if phi is None:
        if background is None:
            raise ValueError("Forneça 'phi' ou 'background' para calcular SHAP.")
        mo = "probability" if target.lower() == "prob" else "log_odds"
        expl = shap.TreeExplainer(model, data=background, model_output=mo)
        sv = expl.shap_values(pd.DataFrame(x_row, columns=feature_names))
        # pega classe positiva
        if isinstance(sv, (list, tuple)):
            phi = np.asarray(sv[pos_idx], float).reshape(-1)
        elif hasattr(shap, "Explanation") and isinstance(sv, shap.Explanation):
            phi = np.asarray(sv.values, float).reshape(-1)
        elif isinstance(sv, np.ndarray) and sv.ndim == 3:
            phi = sv[:, :, pos_idx].reshape(-1)
        elif isinstance(sv, np.ndarray) and sv.ndim == 2:
            phi = sv.reshape(-1)
        else:
            raise RuntimeError("Formato de shap_values não reconhecido.")
    else:
        phi = np.asarray(phi, float).reshape(-1)

    # ---- 3) Vetor baseline (para construir x - I)
    if isinstance(baseline, str):
        if background is None:
            raise ValueError("Para baseline='median'/'mean', forneça 'background'.")
        if baseline.lower() == "median":
            b = np.asarray(background[feature_names].median().values, float)
        elif baseline.lower() == "mean":
            b = np.asarray(background[feature_names].mean().values, float)
        else:
            raise ValueError("baseline string deve ser 'median' ou 'mean'.")
    else:
        b = np.asarray(baseline, float).reshape(-1)
        if b.shape[0] != d:
            raise ValueError("baseline ndarray deve ter dimensão (d,) igual a feature_names.")

    # ---- 4) Amostra de máscaras I (0=mantém, 1=desliga→baseline)
    M = rng.binomial(1, p_mask, size=(n_masks, d)).astype(bool)
    # evita máscara toda-zero (sem perturbação)
    zero_rows = np.where(M.sum(axis=1) == 0)[0]
    for r in zero_rows:
        j = rng.integers(0, d)
        M[r, j] = True

    # ---- 5) x - I (substitui features mascaradas por baseline)
    Z = np.repeat(x_row, repeats=n_masks, axis=0)
    Z_masked = np.where(M, b.reshape(1, -1), Z)

    # ---- 6) Δ_real = f_t(x) - f_t(x - I)
    fx_t   = _to_target(_to_prob(x_row))[0]
    fz_t   = _to_target(_to_prob(Z_masked))
    delta_real = fx_t - fz_t  # (n_masks,)

    # ---- 7) Δ_pred = I^T φ(x)
    delta_pred = M @ phi  # (n_masks,)

    # ---- 8) Infidelity (expectation ≈ média amostral)
    resid = delta_pred - delta_real
    infid_abs = float(np.mean(resid**2))

    mu_dr = float(np.mean(delta_real))
    sse   = float(np.sum((delta_real - delta_pred)**2))
    sst   = float(np.sum((delta_real - mu_dr)**2)) + EPS
    r2    = 1.0 - sse/sst
    infid_pct = float(max(0.0, r2) * 100.0)

    # versão "menor=melhor" em % (apenas transformação linear)
    infid_mm_pct = float(100.0 - infid_pct)

    details = {
        "phi": phi,
        "baseline": b,
        "masks_used": M.shape[0],
        "mean_mask_size": float(M.sum(axis=1).mean()),
        "fx_target": fx_t,
        "delta_real_mean": float(delta_real.mean()),
        "delta_pred_mean": float(delta_pred.mean()),
        "infidelity_pct": infid_pct,      # mantém no details para rastreio
        "infidelity_mm_pct": infid_mm_pct # % com "menor=melhor"
    }

    # Retorno retrocompatível
    if lower_better:
        return infid_mm_pct, infid_abs, details
    else:
        return infid_pct, infid_abs, details

def _to_target_scale_faithfulness(prob: np.ndarray, target: str = "logit", eps: float = 1e-12) -> np.ndarray:
    prob = np.asarray(prob, float).clip(eps, 1 - eps)
    if str(target).lower() == "logit":
        return np.log(prob / (1.0 - prob))
    return prob  # "prob"

def _shap_values_one_row_faithfulness(
    explainer: shap.Explainer,
    x_df: pd.DataFrame,
    pos_class: int
) -> np.ndarray:
    """
    Extrai o vetor 1D de SHAP para uma única instância (classe positiva).
    Compatível com diferentes versões do SHAP.
    """
    sv = explainer.shap_values(x_df)  # pode ser list, ndarray 2D/3D, Explanation…
    # list por classe
    if isinstance(sv, (list, tuple)):
        return np.asarray(sv[pos_class], float).reshape(-1)[-x_df.shape[1]:]
    # Explanation
    if hasattr(shap, "Explanation") and isinstance(sv, shap.Explanation):
        return np.asarray(sv.values, float).reshape(-1)
    # ndarray
    arr = np.asarray(sv)
    if arr.ndim == 3:
        return arr[0, :, pos_class].astype(float)  # (n=1, d, n_classes)
    if arr.ndim == 2:
        return arr[0, :].astype(float)
    raise RuntimeError("Formato de shap_values não reconhecido para 1 amostra.")

def calcula_faithfulness_shap(
    x_row: Union[pd.Series, pd.DataFrame, np.ndarray],
    *,
    model,                              # precisa ter .predict_proba
    X_ref: pd.DataFrame,                # DataFrame com as MESMAS colunas do treino
    pos_class: int = 1,
    shap_explainer: Optional[shap.Explainer] = None,
    background: Optional[pd.DataFrame] = None,   # se None usa X_ref
    target: str = "logit",              # "logit" (recomendado) ou "prob"
    corr: str = "kendall",              # "kendall" | "spearman" | "pearson"
    ref_strategy: str = "background",   # "background" | "mean" | "median" | "zero" | {"col":valor}
    n_ref_samples: int = 128,           # usado apenas se ref_strategy="background"
    random_state: int = 42,
    use_abs: bool = False,              # correlaciona |phi_i| com |Δ_i|
    return_details: bool = False
):
    """
    Faithfulness_SHAP(x) = corr( φ_i(x), f_t(x) − E[f_t(x_{-i})] )
    - f_t = saída do modelo na escala 'prob' ou 'logit'
    - x_{-i}: x com a feature i substituída por baseline (média/mediana/zero ou amostras do background)
    Retorna: (faith_percent, faith_abs, corr_raw [, details])
    """
    EPS = 1e-12

    # --- prepara x como DataFrame com mesmas colunas do X_ref ---
    feat_cols = list(X_ref.columns)
    if isinstance(x_row, pd.Series):
        x_df = pd.DataFrame([x_row[feat_cols].values], columns=feat_cols)
    elif isinstance(x_row, pd.DataFrame):
        x_df = x_row[feat_cols].iloc[:1].copy()
    else:
        x_df = pd.DataFrame(np.asarray(x_row, float).reshape(1, -1), columns=feat_cols)

    # --- probabilidades & alvo na escala escolhida ---
    fx_prob = float(model.predict_proba(x_df)[:, pos_class][0])
    fx_t    = float(_to_target_scale_faithfulness(np.array([fx_prob]), target)[0])

    # --- vetor de SHAP da instância ---
    if shap_explainer is None:
        # default: usa background (ou X_ref) para coerência interventional
        bg = background if background is not None else X_ref
        shap_explainer = shap.TreeExplainer(model, data=bg, model_output="probability")
    phi = _shap_values_one_row_faithfulness(shap_explainer, x_df, pos_class)  # shape (d,)

    # --- deltas reais ao "desligar" cada feature ---
    rng = np.random.default_rng(random_state)
    deltas = np.empty(len(feat_cols), dtype=float)

    if isinstance(ref_strategy, dict):
        # um valor por coluna
        ref_vals = {c: float(ref_strategy.get(c, X_ref[c].astype(float).mean())) for c in feat_cols}
    elif ref_strategy == "mean":
        ref_vals = X_ref.astype(float).mean().to_dict()
    elif ref_strategy == "median":
        ref_vals = X_ref.astype(float).median().to_dict()
    elif ref_strategy == "zero":
        ref_vals = {c: 0.0 for c in feat_cols}
    else:
        ref_vals = None  # "background" ⇒ usa amostras

    # background para amostragem (interventional), se pedido
    BG = background if background is not None else X_ref

    for j, col in enumerate(feat_cols):
        if ref_strategy == "background":
            # média de f_t(x com col=j substituída por valores do background)
            idx = rng.integers(0, len(BG), size=n_ref_samples)
            repl = BG.iloc[idx, j].to_numpy(dtype=float)
            batch = np.repeat(x_df.values, n_ref_samples, axis=0)
            batch[:, j] = repl
            fz_prob = model.predict_proba(pd.DataFrame(batch, columns=feat_cols))[:, pos_class]
            fz_t    = _to_target_scale_faithfulness(fz_prob, target)
            deltas[j] = fx_t - float(np.mean(fz_t))
        else:
            xm = x_df.copy()
            xm[col] = ref_vals[col]
            fz_prob = float(model.predict_proba(xm)[:, pos_class][0])
            fz_t    = float(_to_target_scale_faithfulness(np.array([fz_prob]), target)[0])
            deltas[j] = fx_t - fz_t

    # --- correlação (possivelmente em módulo) ---
    a = np.abs(phi) if use_abs else phi
    b = np.abs(deltas) if use_abs else deltas

    # se uma das séries é constante, correlação = 0
    if np.allclose(a, a[0]) or np.allclose(b, b[0]):
        corr_raw, pval = 0.0, np.nan
    else:
        if corr == "spearman":
            corr_raw, pval = spearmanr(a, b)
        elif corr == "pearson":
            corr_raw, pval = pearsonr(a, b)
        else:  # "kendall"
            corr_raw, pval = kendalltau(a, b, variant='b')
        if not np.isfinite(corr_raw):
            corr_raw, pval = 0.0, np.nan

    faith_abs     = float(np.clip((corr_raw + 1.0)/2.0, 0.0, 1.0))  # [0,1], maior=melhor
    faith_percent = float(f"{faith_abs*100:.3f}")

    if not return_details:
        return faith_percent, faith_abs, corr_raw

    details = {
        "phi": phi,
        "deltas": deltas,
        "feat_cols": feat_cols,
        "corr_type": corr,
        "corr_raw": corr_raw,
        "p_value": pval,
        "target": target,
        "ref_strategy": ref_strategy,
        "n_ref_samples": n_ref_samples if ref_strategy=="background" else None,
    }
    return faith_percent, faith_abs, corr_raw, details

# def calcula_simplicity_shap(
#     phi_row,                       # vetor 1D de shap values da instância (alinhado a feature_names)
#     feature_names=None,            # lista/Index com nomes das colunas (opcional, p/ details)
#     tau=0.05,                      # limiar em % da massa L1 (0.05 ou 5)
#     return_details=False
# ):
#     EPS = 1e-12
#     phi = np.asarray(phi_row, dtype=float).ravel()
#     d   = int(phi.size)

#     # massa L1 e participação relativa de cada feature
#     abs_phi = np.abs(phi)
#     l1_sum  = float(abs_phi.sum())

#     # caso degenerado: todos os SHAP ~ 0
#     if l1_sum <= EPS or d == 0:
#         k_tau = 0
#         simplicity_abs = 1.0
#         simplicity_pct = 100.0
#         complexity_pct = 100.0 - simplicity_pct  # <<< menor = melhor
#         if not return_details:
#             return k_tau, complexity_pct
#         feat = feature_names if feature_names is not None else [f"f{i}" for i in range(d)]
#         details = {
#             "feature_table": pd.DataFrame({
#                 "feature": feat, "phi": phi, "abs_phi": abs_phi,
#                 "p_l1": np.zeros_like(abs_phi), "above_tau_pct": np.zeros_like(abs_phi, bool)
#             }),
#             "tau_input": float(tau),
#             "tau_eff_fraction": float(tau/100.0 if tau > 1 else tau),
#             "d_bar": d,
#             "l1_sum": l1_sum,
#             "complexity_k": k_tau,
#             "simplicity_abs": simplicity_abs,
#             "simplicity_percent": simplicity_pct,
#             "complexity_percent": complexity_pct,   # <<< novo (menor = melhor)
#         }
#         return k_tau, complexity_pct, details

#     p_l1 = abs_phi / l1_sum
#     tau_eff = float(tau/100.0 if tau > 1 else tau)

#     mask_above = p_l1 > tau_eff               # use >= se preferir empates no limiar
#     k_tau = int(mask_above.sum())

#     # Simplicity(x) = 1 - k_tau / d  (maior = melhor)
#     simplicity_abs = 1.0 - (k_tau / d)
#     simplicity_pct = float(simplicity_abs * 100.0)
#     complexity_pct = 100.0 - simplicity_pct     # <<< menor = melhor

#     if not return_details:
#         return k_tau, complexity_pct

#     feat = feature_names if feature_names is not None else [f"f{i}" for i in range(d)]
#     rank = p_l1.argsort()[::-1].argsort() + 1
#     details = {
#         "feature_table": pd.DataFrame({
#             "feature": feat,
#             "phi": phi,
#             "abs_phi": abs_phi,
#             "p_l1": p_l1,
#             "above_tau_pct": mask_above,
#             "rank_p_l1": rank
#         }).sort_values("p_l1", ascending=False).reset_index(drop=True),
#         "tau_input": float(tau),
#         "tau_eff_fraction": tau_eff,
#         "d_bar": d,
#         "l1_sum": l1_sum,
#         "complexity_k": k_tau,
#         "simplicity_abs": float(simplicity_abs),
#         "simplicity_percent": float(simplicity_pct),
#         "complexity_percent": float(complexity_pct),  # <<< novo (menor = melhor)
#     }
#     return k_tau, complexity_pct, details
def calcula_simplicity_shap(
    phi_row,                       # vetor 1D de shap values da instância (alinhado a feature_names)
    feature_names=None,            # lista/Index com nomes das colunas (opcional, p/ details)
    tau=0.05,                      # limiar em % da massa L1 (0.05 ou 5)
    return_details=False
):
    EPS = 1e-12
    phi = np.asarray(phi_row, dtype=float).ravel()
    d   = int(phi.size)

    # massa L1 e participação relativa de cada feature
    abs_phi = np.abs(phi)
    l1_sum  = float(abs_phi.sum())

    # caso degenerado: todos os SHAP ~ 0
    if l1_sum <= EPS or d == 0:
        k_tau = 0
        simplicity_abs = 1.0
        simplicity_pct = 100.0                # MAIOR = melhor
        complexity_pct = 100.0 - simplicity_pct  # legado (MENOR = melhor)
        if not return_details:
            return k_tau, simplicity_pct       # <<== muda p/ simplicidade
        feat = feature_names if feature_names is not None else [f"f{i}" for i in range(d)]
        details = {
            "feature_table": pd.DataFrame({
                "feature": feat, "phi": phi, "abs_phi": abs_phi,
                "p_l1": np.zeros_like(abs_phi), "above_tau_pct": np.zeros_like(abs_phi, bool)
            }),
            "tau_input": float(tau),
            "tau_eff_fraction": float(tau/100.0 if tau > 1 else tau),
            "d_bar": d,
            "l1_sum": l1_sum,
            "complexity_k": k_tau,
            "simplicity_abs": simplicity_abs,
            "simplicity_percent": simplicity_pct,     # MAIOR = melhor
            "complexity_percent": complexity_pct,     # legado (MENOR = melhor)
        }
        return k_tau, simplicity_pct, details         # <<== muda p/ simplicidade

    p_l1 = abs_phi / l1_sum
    tau_eff = float(tau/100.0 if tau > 1 else tau)

    mask_above = p_l1 > tau_eff               # use >= se quiser empates no limiar
    k_tau = int(mask_above.sum())

    # Simplicity(x) = 1 - k_tau / d  (MAIOR = melhor)
    simplicity_abs = 1.0 - (k_tau / d)
    simplicity_pct = float(simplicity_abs * 100.0)
    complexity_pct = 100.0 - simplicity_pct       # legado (MENOR = melhor)

    if not return_details:
        return k_tau, simplicity_pct               # <<== muda p/ simplicidade

    feat = feature_names if feature_names is not None else [f"f{i}" for i in range(d)]
    rank = p_l1.argsort()[::-1].argsort() + 1
    details = {
        "feature_table": pd.DataFrame({
            "feature": feat,
            "phi": phi,
            "abs_phi": abs_phi,
            "p_l1": p_l1,
            "above_tau_pct": mask_above,
            "rank_p_l1": rank
        }).sort_values("p_l1", ascending=False).reset_index(drop=True),
        "tau_input": float(tau),
        "tau_eff_fraction": tau_eff,
        "d_bar": d,
        "l1_sum": l1_sum,
        "complexity_k": k_tau,
        "simplicity_abs": float(simplicity_abs),
        "simplicity_percent": float(simplicity_pct),   # MAIOR = melhor (principal)
        "complexity_percent": float(complexity_pct),   # legado (MENOR = melhor)
    }
    return k_tau, simplicity_pct, details            # <<== muda p/ simplicidade

def _normalize_corr(corr):
    """
    Normaliza o parâmetro 'corr' para um de {'spearman','kendall','pearson'}.
    Aceita vários apelidos; default = 'spearman' quando None ou desconhecido.
    """
    if corr is None:
        return "spearman"
    c = str(corr).strip().lower()
    aliases = {
        # Spearman
        "spearman": "spearman", "spear": "spearman", "spearmn": "spearman",
        "rank": "spearman", "s": "spearman",

        # Kendall (tau-b)
        "kendall": "kendall", "kendal": "kendall", "tau": "kendall",
        "tau-b": "kendall", "taub": "kendall", "k": "kendall",

        # Pearson
        "pearson": "pearson", "pearsonr": "pearson", "perarson": "pearson",
        "linear": "pearson", "p": "pearson", "corr": "pearson",
    }
    return aliases.get(c, "spearman")


def _phi_from_shap_output_consistency(shap_values_obj, explainer=None, pos_idx=1):
    
    """
    Converte a saída do SHAP (várias versões/formas) em um vetor 1D (d,)
    de valores SHAP para UMA instância, escolhendo a classe `pos_idx` se multiclass.

    Aceita:
      - shap.Explanation (v2+): .values com shape (d,), (1,d) ou (1,d,n_classes)
      - lista/tupla por classe: [arr_classe0, arr_classe1, ...]
      - ndarray 3D: (n, d, n_classes)
      - ndarray 2D: (1, d) ou (d, 1)
      - ndarray 1D: (d,)
    """
    # 1) Novo API: shap.Explanation
    try:
        if hasattr(shap, "Explanation") and isinstance(shap_values_obj, shap.Explanation):
            v = np.asarray(shap_values_obj.values, dtype=float)
            v = np.squeeze(v)
            if v.ndim == 1:               # (d,)
                return v.astype(float)
            if v.ndim == 2 and v.shape[0] == 1:   # (1,d)
                return v[0].astype(float)
            if v.ndim == 3 and v.shape[0] == 1:   # (1,d,n_classes)
                return v[0, :, pos_idx].astype(float)
    except Exception:
        pass

    # 2) Lista/Tupla por classe (versões antigas)
    if isinstance(shap_values_obj, (list, tuple)) and len(shap_values_obj) >= (pos_idx + 1):
        v = np.asarray(shap_values_obj[pos_idx], dtype=float)
        v = np.squeeze(v)
        if v.ndim == 1:               # (d,)
            return v
        if v.ndim == 2 and v.shape[0] == 1:  # (1,d)
            return v[0]
        if v.ndim == 2 and v.shape[0] > 1:
            raise ValueError(f"Esperada apenas 1 instância; recebido batch com shape {v.shape}.")
        return v

    # 3) Arrays puros
    if isinstance(shap_values_obj, np.ndarray):
        a = shap_values_obj
        if a.ndim == 1:                     # (d,)
            return a.astype(float)
        if a.ndim == 2:
            if a.shape[0] == 1:             # (1,d)
                return a[0].astype(float)
            if a.shape[1] == 1:             # (d,1)
                return a[:, 0].astype(float)
            raise ValueError(f"Ambiguidade no formato SHAP 2D {a.shape}; forneça apenas uma instância.")
        if a.ndim == 3:                      # (n, d, n_classes)
            if a.shape[0] == 1:
                return a[0, :, pos_idx].astype(float)
            raise ValueError(f"Esperado batch=1 em a.shape[0]; shape recebido: {a.shape}")

    raise RuntimeError("Formato de shap_values não reconhecido em _phi_from_shap_output.")
                                                                                                                                 
def calcula_consistency_shap_models(
    x_row_df,                 # DataFrame com 1 linha (mesmas colunas)
    X_ref,                    # DataFrame de referência p/ valores "mask"
    model_f,                  # modelo f
    model_fp,                 # modelo f'
    *, 
    pos_class=1,
    ref_strategy="mean",      # 'mean'|'median'|'zero'|'mode' ou dict col->valor
    corr="spearman",          # correlação entre Δφ e Δm
    background_f=None,        # background p/ SHAP(f)  (array/DF)   (opcional)
    background_fp=None,       # background p/ SHAP(f') (array/DF)   (opcional)
    rank_by="raw",            # normalmente "raw" (com sinal) aqui
    return_details=False
):
    """
    Mede a consistência do axioma: aumentos na contribuição marginal implicam
    aumentos nos valores de Shapley (e vice-versa), para UMA instância.
    Retorna: (consistency_percent, consistency_abs[, details])
    """
    corr = _normalize_corr(corr)

    # classes
    try:
        classes = list(getattr(model_f, "classes_", [0,1]))
    except Exception:
        classes = [0,1]
    pos_idx = classes.index(1) if 1 in classes else int(np.argmax(classes))

    # explainers / phi (f) e (f')
    bg_f  = background_f if background_f is not None else X_ref
    bg_fp = background_fp if background_fp is not None else X_ref
    expl_f  = shap.TreeExplainer(model_f,  data=bg_f,  model_output="probability")
    expl_fp = shap.TreeExplainer(model_fp, data=bg_fp, model_output="probability")

    out_f  = expl_f.shap_values(x_row_df)
    out_fp = expl_fp.shap_values(x_row_df)
    phi_f  = _phi_from_shap_output_consistency(out_f,  expl_f,  pos_idx)
    phi_fp = _phi_from_shap_output_consistency(out_fp, expl_fp, pos_idx)
    d = phi_f.size

    # valores de referência p/ "remover" feature
    if isinstance(ref_strategy, dict):
        ref_vals = {c: float(ref_strategy.get(c, X_ref[c].astype(float).mean())) for c in x_row_df.columns}
    else:
        if ref_strategy == 'median':
            ref_vals = X_ref[x_row_df.columns].astype(float).median().to_dict()
        elif ref_strategy == 'zero':
            ref_vals = {c: 0.0 for c in x_row_df.columns}
        elif ref_strategy == 'mode':
            ref_vals = {c: float(X_ref[c].mode(dropna=True)[0]) if not X_ref[c].mode(dropna=True).empty
                        else float(X_ref[c].astype(float).mean()) for c in x_row_df.columns}
        else:  # 'mean'
            ref_vals = X_ref[x_row_df.columns].astype(float).mean().to_dict()

    # f(x) e f(x\i) para f e f'
    fx_f  = float(model_f.predict_proba(x_row_df.values)[0, pos_idx])
    fx_fp = float(model_fp.predict_proba(x_row_df.values)[0, pos_idx])

    delta_m_f  = []
    delta_m_fp = []
    for col in x_row_df.columns:
        xm = x_row_df.copy()
        xm[col] = ref_vals[col]
        delta_m_f.append(fx_f  - float(model_f.predict_proba(xm.values)[0, pos_idx]))
        delta_m_fp.append(fx_fp - float(model_fp.predict_proba(xm.values)[0, pos_idx]))

    delta_m_f  = np.asarray(delta_m_f,  float)
    delta_m_fp = np.asarray(delta_m_fp, float)

    # diferenças entre modelos
    dphi = (np.abs(phi_fp) - np.abs(phi_f)) if str(rank_by).lower()=="abs" else (phi_fp - phi_f)
    dm   = delta_m_fp - delta_m_f

    # correlação Δφ vs Δm
    if np.allclose(dphi, dphi[0]) or np.allclose(dm, dm[0]):
        rho = 0.0
    else:
        if corr == "kendall":
            rho, _ = kendalltau(dphi, dm, variant="b")
        elif corr == "pearson":
            rho, _ = pearsonr(dphi, dm)
        else:
            rho, _ = spearmanr(dphi, dm)
        if not np.isfinite(rho):
            rho = 0.0

    consistency_abs = float(np.clip((rho + 1.0)/2.0, 0.0, 1.0))
    consistency_pct = float(f"{consistency_abs*100:.3f}")

    if not return_details:
        return consistency_pct, consistency_abs

    details = {
        "rho": float(rho),
        "corr_type": corr,
        "rank_by": str(rank_by).lower(),
        "phi_f":  phi_f,   "phi_fp": phi_fp,
        "delta_m_f": delta_m_f, "delta_m_fp": delta_m_fp,
        "dphi": dphi, "dm": dm, "d": int(d),
        "ref_vals": ref_vals,
    }
    return consistency_pct, consistency_abs, details


def _phi_from_shap_output_robustez(out, explainer, pos_idx):
    """
    Extrai um vetor 1D (d,) com os valores SHAP da classe positiva.
    Aceita: lista por classe, ndarray 3D (n,d,C), shap.Explanation ou ndarray 2D (1,d).
    """
    # lista/tupla por classe
    if isinstance(out, (list, tuple)):
        a = np.asarray(out[pos_idx], dtype=float)
        if a.ndim == 2 and a.shape[0] == 1:
            return a[0]
        if a.ndim == 1:
            return a
        raise ValueError(f"Esperado (1,d) na classe {pos_idx}, obtido {a.shape}.")
    # Explanation
    if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
        a = np.asarray(out.values, dtype=float)
        if a.ndim == 1:
            return a
        if a.ndim == 2 and a.shape[0] == 1:
            return a[0]
        raise ValueError(f"Formato Explanation inesperado: {a.shape}.")
    # ndarray 3D (n,d,C)
    if isinstance(out, np.ndarray) and out.ndim == 3:
        if out.shape[0] != 1:
            raise ValueError("Forneça apenas UMA instância (n=1) para calcular robustez.")
        return out[0, :, pos_idx].astype(float)
    # ndarray 2D (1,d)
    if isinstance(out, np.ndarray) and out.ndim == 2:
        if out.shape[0] == 1:
            return out[0].astype(float)
        raise ValueError("Ambiguidade (n,d) com n>1; envie apenas uma instância.")
    raise ValueError("Formato SHAP não reconhecido.")

def _ensure_row_df_robustez(x_row_df, feat_cols_eff):
    """Garante DataFrame 1×d com as colunas na ordem correta."""
    if isinstance(x_row_df, pd.Series):
        x_row_df = x_row_df.to_frame().T
    elif isinstance(x_row_df, np.ndarray):
        x_row_df = pd.DataFrame(x_row_df.reshape(1, -1), columns=feat_cols_eff)
    else:
        x_row_df = pd.DataFrame(x_row_df).copy()
    # reordena/filtra para feat_cols_eff
    return x_row_df.loc[:, feat_cols_eff].iloc[[0]]

def calcula_robustez_shap(
    *,
    x_row_df,                 # DataFrame 1×d (use X_sel.iloc[[pos_local]])
    model,                    # modelo treinado (RF, XGB, etc.)
    X_ref: pd.DataFrame,      # DataFrame de referência (p/ bounds e categorias)
    background: pd.DataFrame = None,  # background SHAP (ex.: X_sel ou X_train amostrado)
    drop_cols=('classe','y'),
    pos_class=1,
    n_trials: int = 300,
    epsilon: float = 0.05,
    epsilon_mode: str = "relative_range",   # 'relative_range' | 'absolute'
    norm: str = "l2",                       # 'l2' ou 'l1'
    strategy: str = "all",                  # 'all' (perturba todas) | 'one' (uma aleatória)
    features_to_test=None,                  # lista de colunas a perturbar (ou None=todas)
    cat_flip_prob: float = 0.10,
    random_state: int = 42,
    topk_delta: int = 8,
    score_mode: str = "relative",           # 'relative' | 'cosine' | 'exp'
    score_alpha: float = None,              # só p/ 'exp'
    higher_better: bool = False             # <<< NOVO: quando True, 1º retorno vira robust_score (↑ melhor)
):
    """
    Robustness_SHAP(x) = max_{||x'−x||<ε} || Φ(x) − Φ(x') ||  (Φ = vetor SHAP).

    Retornos:
      - Se higher_better=False (padrão):
          robust_max, robust_mean, robust_std, details
          (distâncias — menor = melhor)
      - Se higher_better=True:
          robust_score, robust_mean, robust_std, details
          (robust_score ∈ [0,100] — maior = melhor)
    """
    rng = np.random.default_rng(random_state)

    # ------ colunas efetivas ------
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    x_row_df = _ensure_row_df_robustez(x_row_df, feat_cols_eff)

    # ------ classe positiva como índice ------
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(1) if 1 in classes else int(np.argmax(classes))

    # ------ explainer SHAP ------
    bg = background if background is not None else X_ref[feat_cols_eff]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")

    # ------ SHAP base (Φ(x)) ------
    out0 = expl.shap_values(x_row_df[feat_cols_eff])
    phi0 = _phi_from_shap_output_robustez(out0, expl, pos_idx)  # (d,)

    # ------ tipos/bounds p/ perturbar ------
    num_idx, cat_idx = [], []
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            num_idx.append(j)
        else:
            cat_idx.append(j)

    bounds = {}
    for j in num_idx:
        col = feat_cols_eff[j]
        v = X_ref[col].to_numpy()
        bounds[j] = (float(np.nanmin(v)), float(np.nanmax(v)))

    # utilitário de perturbação
    def _perturb_row(x1: pd.DataFrame) -> pd.DataFrame:
        xprime = x1.copy()
        # quais colunas mexer
        targets = list(features_to_test) if features_to_test else list(feat_cols_eff)
        if strategy == "one":
            targets = [rng.choice(targets)]
        t_idx = [feat_cols_eff.index(c) for c in targets]

        # numéricas
        for j in [jj for jj in t_idx if jj in num_idx]:
            col = feat_cols_eff[j]
            lo, hi = bounds.get(j, (x1.iloc[0, j], x1.iloc[0, j]))
            if epsilon_mode == "relative_range":
                radius = epsilon * max(1e-12, hi - lo)
            else:  # 'absolute'
                radius = float(epsilon)
            val = float(x1.iloc[0, j]) + rng.uniform(-radius, radius)
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                val = min(max(val, lo), hi)
            xprime.iloc[0, j] = val

        # categóricas
        for j in [jj for jj in t_idx if jj in cat_idx]:
            if rng.random() < cat_flip_prob:
                col = feat_cols_eff[j]
                cats = pd.Series(X_ref[col].dropna().unique()).tolist()
                if len(cats) > 1:
                    cur = x1.iloc[0, j]
                    choices = [c for c in cats if c != cur]
                    if choices:
                        xprime.iloc[0, j] = rng.choice(choices)
        return xprime

    # ------ busca adversarial simples ------
    dists = []
    best = {"dist": -np.inf, "xprime": None, "phi_prime": None}
    for _ in range(int(n_trials)):
        x_p = _perturb_row(x_row_df)
        outp = expl.shap_values(x_p[feat_cols_eff])
        phi_p = _phi_from_shap_output_robustez(outp, expl, pos_idx)

        if norm == "l2":
            dist = float(np.linalg.norm(phi0 - phi_p))
        else:
            dist = float(np.linalg.norm(phi0 - phi_p, ord=1))
        dists.append(dist)

        if dist > best["dist"]:
            best["dist"] = dist
            best["xprime"] = x_p.copy()
            best["phi_prime"] = phi_p.copy()

    robust_max  = float(np.max(dists)) if dists else 0.0
    robust_mean = float(np.mean(dists)) if dists else 0.0
    robust_std  = float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0

    # top-K deltas de Φ
    delta = np.abs(phi0 - best["phi_prime"])
    topk_idx = np.argsort(-delta)[:min(topk_delta, delta.size)]
    top_weights_delta = [
        {
            "feature": feat_cols_eff[j],
            "phi_x": float(phi0[j]),
            "phi_xprime": float(best["phi_prime"][j]),
            "abs_delta": float(delta[j]),
        }
        for j in topk_idx
    ]

    # ---- escores (opcionais) ----
    eps = 1e-12
    if norm == "l2":
        base_norm = float(np.linalg.norm(phi0))
        p_norm    = float(np.linalg.norm(best["phi_prime"]))
    else:
        base_norm = float(np.linalg.norm(phi0, ord=1))
        p_norm    = float(np.linalg.norm(best["phi_prime"], ord=1))

    if score_mode == "relative":
        ratio = robust_max / (base_norm + eps)
        robust_score = 100.0 * max(0.0, 1.0 - min(1.0, float(ratio)))
    elif score_mode == "cosine":
        dot = float(np.dot(phi0, best["phi_prime"]))
        denom = (np.linalg.norm(phi0) * np.linalg.norm(best["phi_prime"]) + eps)
        cos_sim = float(dot / denom)
        robust_score = 100.0 * (cos_sim + 1.0) / 2.0
    elif score_mode == "exp":
        if score_alpha is None:
            med = float(np.median(dists)) if len(dists) else 0.0
            score_alpha = np.log(2.0) / (med + eps) if med > 0 else 1.0
        robust_score = 100.0 * float(np.exp(-score_alpha * robust_max))
    else:
        raise ValueError("score_mode deve ser 'relative', 'cosine' ou 'exp'.")

    details = {
        "norm": norm,
        "epsilon": float(epsilon),
        "epsilon_mode": epsilon_mode,
        "strategy": strategy,
        "n_trials": int(n_trials),
        "n_lime_repeats": 0,                 # compat. com seu make_robustness_row
        "tested_features": ",".join(features_to_test) if features_to_test else "ALL",
        "dists": dists,
        "top_weights_delta": top_weights_delta,
        "robust_score": float(robust_score),     # ↑ melhor
        "score_mode": score_mode,
        "score_alpha": score_alpha if score_mode == "exp" else None,
        "base_norm_phi0": base_norm,
        "phi_prime_norm": p_norm,
        "pos_class_index": int(pos_idx),
    }

    # --- Retorno: compatível por padrão; com higher_better=True, 1º retorno vira o score (↑ melhor)
    if higher_better:
        return robust_score, robust_mean, robust_std, details
    return robust_max, robust_mean, robust_std, details

def calcula_completeness_shap(
    *,
    x_row_df,                  # Series / 1xD DataFrame / 1D array
    model,                     # modelo com predict_proba
    X_ref: pd.DataFrame,       # para obter as colunas e (opcional) background
    background: pd.DataFrame=None,
    drop_cols=('classe','y'),
    pos_class=1,
    target: str = "prob",      # "prob" (padrão) ou "raw" (margem das árvores)
    return_details: bool = True
):
    """
    Completeness_SHAP(x) ∈ [0,100]: 100% quando f(x) = φ0 + Σ φ_i(x) (mesma escala dos SHAP).
    Retorna: (completeness_percent, erro_abs [, details])
    """
    # --- helpers internos (não conflitam com os seus) ---
    def _ensure_row_df_local(x_row, cols):
        if isinstance(x_row, pd.DataFrame):
            return x_row.loc[:, cols].iloc[[0]]
        if isinstance(x_row, pd.Series):
            return pd.DataFrame([x_row.values], columns=cols)
        return pd.DataFrame([np.asarray(x_row, float).ravel()], columns=cols)

    def _phi_from_out_local(out, explainer, pos_idx):
        # shap.Explanation
        try:
            if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
                v = np.asarray(out.values, float).squeeze()
                if v.ndim == 1: return v.astype(float)
                if v.ndim == 2 and v.shape[0] == 1: return v[0].astype(float)
        except Exception:
            pass
        # lista por classe
        if isinstance(out, (list, tuple)):
            return np.asarray(out[pos_idx], float).ravel()
        # arrays
        A = np.asarray(out)
        if A.ndim == 3:  # (1,d,C)
            return A[0, :, pos_idx].astype(float)
        if A.ndim == 2:  # (1,d)
            if A.shape[0] != 1:
                raise ValueError(f"Forneça apenas 1 instância, recebido {A.shape}")
            return A[0].astype(float)
        raise ValueError("Formato de shap_values não reconhecido.")

    def _base_from_out_local(out, explainer, pos_idx):
        # tenta vir do objeto Explanation
        try:
            if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
                b = np.asarray(out.base_values, float).squeeze()
                if b.ndim == 0: return float(b)
                if b.ndim == 1:
                    return float(b[pos_idx] if b.size > 1 else b[0])
        except Exception:
            pass
        # fallback: expected_value do explainer
        ev = getattr(explainer, "expected_value", None)
        if isinstance(ev, (list, np.ndarray)):
            return float(np.asarray(ev, float)[pos_idx])
        return float(ev)

    # --- preparação ---
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_local(x_row_df, feat_cols)

    # índice da classe positiva na saída de probas
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # escolhe o background e a escala do SHAP
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    model_output = "probability" if str(target).lower() == "prob" else "raw"

    expl = shap.TreeExplainer(model, data=bg, model_output=model_output)
    out  = expl.shap_values(x1[feat_cols])

    phi   = _phi_from_out_local(out, expl, pos_idx)           # (d,)
    phi_0 = _base_from_out_local(out, expl, pos_idx)          # escalar

    # f(x) na MESMA escala dos SHAP
    fx_prob = float(model.predict_proba(x1[feat_cols])[:, pos_idx][0])
    fx_t    = fx_prob if model_output == "probability" else np.log(np.clip(fx_prob,1e-12,1-1e-12) /
                                                                   np.clip(1-fx_prob,1e-12,1-1e-12))

    recon   = float(phi_0 + np.sum(phi))
    err_abs = float(abs(fx_t - recon))            # erro absoluto de reconstrução

    # score ∈ [0,100] (quanto maior, melhor). Em probabilidade, o pior erro possível é 1.
    denom   = 1.0 if model_output == "probability" else (abs(fx_t) + 1.0)  # escala simples p/ margem
    score   = 100.0 * max(0.0, 1.0 - err_abs/denom)

    if not return_details:
        return score, err_abs

    details = {
        "fx": fx_t,
        "phi0": phi_0,
        "sum_phi": float(np.sum(phi)),
        "recon": recon,
        "erro_abs": err_abs,
        "scale": model_output,
        "phi_vector": phi,
    }
    return score, err_abs, details

def calcula_selectivity_shap(
    *,
    x_row_df,                 # Series / 1xD DataFrame / 1D array (uma única instância)
    model,                    # modelo com predict_proba
    X_ref: pd.DataFrame,      # serve para colunas e para construir baseline
    background: pd.DataFrame=None,   # background do SHAP (ex.: X_train ou X_sel)
    drop_cols=('classe','y'),
    pos_class=1,
    K: int | None = None,     # nº máx de features consideradas (se None usa todas)
    baseline: str | dict | np.ndarray = "mean",  # 'mean'|'median'|'zero'|'background'|dict|array
    target: str = "prob",     # "prob" (recomendado) ou "logit" (converte para logit)
    clip_negatives: bool = True,      # se True, usa max(0, f(x)-f(x\S_k)) nas médias
    return_details: bool = True
):
    """
    Selectivity(x) = (1/K) * Σ_{k=1..K} [ f(x) − f(x\S_k) ],
      onde S_k são as k features mais importantes (|φ| desc).

    Retorna: (selectivity_percent, selectivity_abs [, details])
      - selectivity_abs = média das quedas (na escala escolhida)
      - selectivity_percent = 100 * selectivity_abs / (|f(x)| + eps)  (em prob., denom≈f(x))
    """
    EPS = 1e-12

    # ----- helpers locais (evita conflito com seus helpers) -----
    def _ensure_row_df_local(x_row, cols):
        if isinstance(x_row, pd.DataFrame):  return x_row.loc[:, cols].iloc[[0]]
        if isinstance(x_row, pd.Series):     return pd.DataFrame([x_row.values], columns=cols)
        return pd.DataFrame([np.asarray(x_row, float).ravel()], columns=cols)

    def _phi_from_out_local(out, explainer, pos_idx):
        try:  # shap.Explanation
            if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
                v = np.asarray(out.values, float).squeeze()
                if v.ndim == 1: return v.astype(float)
                if v.ndim == 2 and v.shape[0] == 1: return v[0].astype(float)
        except Exception:
            pass
        if isinstance(out, (list, tuple)):   # lista por classe
            return np.asarray(out[pos_idx], float).ravel()
        A = np.asarray(out)
        if A.ndim == 3:  # (1,d,C)
            return A[0, :, pos_idx].astype(float)
        if A.ndim == 2 and A.shape[0] == 1:  # (1,d)
            return A[0].astype(float)
        raise ValueError("Formato de shap_values não reconhecido para uma instância.")

    def _to_target_local(p):
        p = np.clip(float(p), EPS, 1 - EPS)
        if str(target).lower() == "logit":
            return float(np.log(p / (1 - p)))
        return p

    # ----- preparação -----
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_local(x_row_df, feat_cols)

    # índice da classe positiva
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # background + explainer (em probabilidade)
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")

    # φ(x) e ranking por |φ|
    out = expl.shap_values(x1[feat_cols])
    phi = _phi_from_out_local(out, expl, pos_idx)  # (d,)
    order = np.argsort(np.abs(phi))[::-1]          # índices das + importantes
    d = len(feat_cols)
    K_eff = int(d if K is None else max(1, min(K, d)))

    # f(x) na escala-alvo
    fx_prob = float(model.predict_proba(x1[feat_cols])[:, pos_idx][0])
    fx_t    = _to_target_local(fx_prob)

    # baseline vetor
    if isinstance(baseline, np.ndarray):
        base_vec = baseline.astype(float).ravel()
        assert base_vec.size == d
        def _fill_with_base(x_arr, idxs):
            x_arr[0, idxs] = base_vec[idxs];  return x_arr
    elif isinstance(baseline, dict):
        base_vec = np.array([float(baseline.get(c, X_ref[c].astype(float).mean())) for c in feat_cols], float)
        def _fill_with_base(x_arr, idxs):
            x_arr[0, idxs] = base_vec[idxs];  return x_arr
    elif str(baseline).lower() in ("median",):
        base_vec = X_ref[feat_cols].median().astype(float).values
        def _fill_with_base(x_arr, idxs):
            x_arr[0, idxs] = base_vec[idxs];  return x_arr
    elif str(baseline).lower() in ("zero",):
        base_vec = np.zeros(d, float)
        def _fill_with_base(x_arr, idxs):
            x_arr[0, idxs] = base_vec[idxs];  return x_arr
    elif str(baseline).lower() in ("background", "bg"):
        rng = np.random.default_rng(2025)
        def _fill_with_base(x_arr, idxs):
            b = bg.sample(n=1, random_state=rng.integers(0, 2**31-1)).to_numpy()
            x_arr[0, idxs] = b[0, idxs];  return x_arr
    else:  # 'mean' (default)
        base_vec = X_ref[feat_cols].mean().astype(float).values
        def _fill_with_base(x_arr, idxs):
            x_arr[0, idxs] = base_vec[idxs];  return x_arr

    # ----- aplica S_k e mede quedas -----
    drops = []
    f_x_minus_list = []
    top_features = []
    for k in range(1, K_eff + 1):
        idxs = order[:k]
        xm = x1[feat_cols].to_numpy().copy()
        xm = _fill_with_base(xm, idxs)
        xm_df = pd.DataFrame(xm, columns=feat_cols)

        p_minus = float(model.predict_proba(xm_df)[:, pos_idx][0])
        f_minus = _to_target_local(p_minus)
        drop    = fx_t - f_minus
        if clip_negatives:      # evita que oscilações positivas “anulem” a queda
            drop = max(0.0, drop)

        drops.append(drop)
        f_x_minus_list.append(f_minus)
        top_features.append([feat_cols[j] for j in idxs])

    drops = np.asarray(drops, float)
    selectivity_abs = float(np.mean(drops))                    # média das quedas
    denom = abs(fx_t) + EPS
    selectivity_percent = float(np.clip(100.0 * selectivity_abs / denom, 0.0, 100.0))

    if not return_details:
        return selectivity_percent, selectivity_abs

    details = {
        "fx": fx_t,
        "K_eff": K_eff,
        "order_by_absphi": [feat_cols[j] for j in order],
        "phi_sorted": phi[order],
        "drops_by_k": drops,                 # array com as quedas por k=1..K
        "f_x_minus_by_k": f_x_minus_list,    # f(x\S_k) por k
        "top_features_by_k": top_features,   # lista de listas com as top-k
        "baseline": "background" if baseline is None else str(baseline),
        "target": str(target).lower()
    }
    return selectivity_percent, selectivity_abs, details

def calcula_soundness_shap(
    *,
    x_row_df,                 # Series / 1xD DataFrame / 1D array (uma instância)
    model,                    # modelo com .predict_proba
    X_ref: pd.DataFrame,
    background: pd.DataFrame = None,
    drop_cols=('classe','y'),
    pos_class=1,
    # --- definição de U ---
    U_cols=None,              # usado quando U_mode='given'
    U_mode:str="given",       # 'given' | 'auto_local'
    tau:float=0.01,           # limiar p/ 'auto_local' (queda máxima em f(x))
    masking:str="mean_mode",  # 'mean_mode'|'median_mode'|'zero'|'sample'|'background'
    baseline_values:dict|None=None,  # overrides por coluna em 'auto_local'
    # --- escala do f(x) ---
    target:str="prob",        # 'prob' (recomendado) | 'logit'
    return_details:bool=True
):
    """
    Retorna (soundness_percent, soundness_abs [, details]).
    Quanto mais próximo de 1 (ou 100%), mais 'sound' (pouca massa |phi| em features irrelevantes).
    """
    EPS = 1e-12

    # ----- helpers (não conflitam com os seus) -----
    def _ensure_row_df_local(x_row, cols):
        if isinstance(x_row, pd.DataFrame):  return x_row.loc[:, cols].iloc[[0]]
        if isinstance(x_row, pd.Series):     return pd.DataFrame([x_row.values], columns=cols)
        return pd.DataFrame([np.asarray(x_row, float).ravel()], columns=cols)

    def _phi_from_out_local(out, explainer, pos_idx):
        try:
            if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
                v = np.asarray(out.values, float).squeeze()
                if v.ndim == 1: return v.astype(float)
                if v.ndim == 2 and v.shape[0] == 1: return v[0].astype(float)
        except Exception:
            pass
        if isinstance(out, (list, tuple)):
            return np.asarray(out[pos_idx], float).ravel()
        A = np.asarray(out)
        if A.ndim == 3:  # (1,d,C)
            return A[0, :, pos_idx].astype(float)
        if A.ndim == 2 and A.shape[0] == 1:
            return A[0].astype(float)
        raise ValueError("Formato SHAP não reconhecido para uma instância.")

    def _to_target_local(p):
        p = np.clip(float(p), EPS, 1 - EPS)
        if str(target).lower() == "logit":
            return float(np.log(p / (1 - p)))
        return p

    # ----- preparação -----
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_local(x_row_df, feat_cols)

    # índice da classe positiva
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # background + explainer (prob)
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")

    # φ(x) e massa total
    out = expl.shap_values(x1[feat_cols])
    phi = _phi_from_out_local(out, expl, pos_idx)           # (d,)
    abs_phi = np.abs(phi)
    denom = float(abs_phi.sum()) + EPS                      # Σ|phi_i|

    # f(x) na escala-alvo (apenas para 'auto_local')
    fx_prob = float(model.predict_proba(x1[feat_cols])[:, pos_idx][0])
    fx_t    = _to_target_local(fx_prob)

    # ----- definir U -----
    U_used = []   # nomes
    U_idx  = []   # índices
    auto_local_details = []

    if str(U_mode).lower() == "given":
        U_cols = list(U_cols) if U_cols is not None else []
        for c in U_cols:
            if c in feat_cols:
                U_used.append(c)
                U_idx.append(feat_cols.index(c))

    elif str(U_mode).lower() == "auto_local":
        # construir baselines por coluna
        if baseline_values is None:
            baseline_values = {}
        for c in feat_cols:
            if c in baseline_values: 
                continue
            s = X_ref[c]
            if pd.api.types.is_numeric_dtype(s):
                if masking == "median_mode":
                    baseline_values[c] = float(s.median())
                elif masking == "zero":
                    baseline_values[c] = 0.0
                elif masking == "sample":
                    vals = s.dropna().values
                    baseline_values[c] = float(vals[np.random.randint(len(vals))]) if len(vals) else float(s.mean())
                else:  # 'mean_mode' (default)
                    baseline_values[c] = float(s.mean())
            else:
                if masking == "sample":
                    vals = s.dropna().values
                    baseline_values[c] = vals[np.random.randint(len(vals))] if len(vals) else (s.mode().iloc[0] if not s.mode().empty else None)
                elif masking == "background":
                    # será amostrado da linha de background na hora de mascarar
                    baseline_values[c] = None
                else:  # moda
                    baseline_values[c] = s.mode().iloc[0] if not s.mode().empty else s.dropna().iloc[0]

        # testa feature a feature
        for j, col in enumerate(feat_cols):
            xm = x1.copy()
            if masking == "background":
                b = bg.sample(n=1).iloc[0, :]
                xm.iloc[0, j] = b[col]
                base_used = f"[bg]{b[col]}"
            else:
                xm.iloc[0, j] = baseline_values[col]
                base_used = baseline_values[col]

            p_mask = float(model.predict_proba(xm[feat_cols])[:, pos_idx][0])
            fx_mask = _to_target_local(p_mask)
            delta = abs(fx_t - fx_mask)
            auto_local_details.append({"feature": col, "delta_fx": float(delta), "baseline_used": base_used})
            if delta <= float(tau):
                U_used.append(col)
                U_idx.append(j)
    else:
        raise ValueError("U_mode deve ser 'given' ou 'auto_local'.")

    # ----- Soundness -----
    numer = float(abs_phi[U_idx].sum()) if U_idx else 0.0   # Σ_{j∈U} |phi_j|
    soundness_abs = 1.0 - (numer / denom)                   # ∈ [0,1]
    soundness_abs = float(np.clip(soundness_abs, 0.0, 1.0))
    soundness_percent = 100.0 * soundness_abs

    if not return_details:
        return soundness_percent, soundness_abs

    # detalhes
    weights_by_feature = [
        {"feature": feat_cols[i], "phi": float(phi[i]), "abs_phi": float(abs_phi[i])}
        for i in range(len(feat_cols))
    ]
    weights_by_feature.sort(key=lambda d: d["abs_phi"], reverse=True)

    details = {
        "fx": fx_t,
        "denom_sum_abs_phi": denom,
        "numer_sum_abs_phi_U": numer,
        "U_mode": U_mode,
        "U_cols_used": U_used,
        "auto_local_details": auto_local_details if U_mode=="auto_local" else None,
        "weights_by_feature": weights_by_feature,
    }
    return soundness_percent, soundness_abs, details

def _ensure_row_df_dir_soudness(x_row_df, feat_cols_eff):
    """
    Garante um DataFrame 1×d alinhado às colunas `feat_cols_eff`.
    Aceita: DataFrame (qualquer shape), Series, array 1D/2D.
    """
    # normaliza para DataFrame
    if isinstance(x_row_df, pd.DataFrame):
        xdf = x_row_df.copy()
    elif isinstance(x_row_df, pd.Series):
        xdf = x_row_df.to_frame().T
    else:
        arr = np.asarray(x_row_df)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        xdf = pd.DataFrame(arr, columns=feat_cols_eff if arr.shape[1] == len(feat_cols_eff) else None)

    # reordena/filtra e garante 1 linha
    if set(feat_cols_eff).issubset(set(xdf.columns)):
        xdf = xdf.loc[:, feat_cols_eff]
    else:
        # se vier sem rótulos compatíveis, força as colunas
        xdf = pd.DataFrame(xdf.to_numpy().reshape(1, -1), columns=feat_cols_eff)
    return xdf.iloc[[0]]

def _phi_from_shap_output_dir_soudness(out, explainer, pos_idx: int):
    """
    Extrai vetor 1D (d,) de valores SHAP da classe `pos_idx` para UMA instância.
    Aceita:
      - lista/tupla por classe: [arr_classe0, arr_classe1, ...]
      - shap.Explanation (v2+): .values com shape (d,), (1,d) ou (1,d,C)
      - ndarray 3D: (n, d, C)   (usa n=1)
      - ndarray 2D: (1, d) ou (d, 1)
      - ndarray 1D: (d,)
    """
    # 1) Novo API: shap.Explanation
    try:
        if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
            v = np.asarray(out.values, dtype=float)
            v = np.squeeze(v)
            if v.ndim == 1:                 # (d,)
                return v.astype(float)
            if v.ndim == 2 and v.shape[0] == 1:   # (1,d)
                return v[0].astype(float)
            if v.ndim == 3 and v.shape[0] == 1:   # (1,d,C)
                return v[0, :, pos_idx].astype(float)
    except Exception:
        pass

    # 2) Lista/tupla por classe (versões antigas)
    if isinstance(out, (list, tuple)):
        v = np.asarray(out[pos_idx], dtype=float)
        v = np.squeeze(v)
        if v.ndim == 1:               # (d,)
            return v
        if v.ndim == 2 and v.shape[0] == 1:  # (1,d)
            return v[0]
        raise ValueError(f"Esperada apenas 1 instância; recebido shape {v.shape}.")

    # 3) Arrays puros
    a = np.asarray(out)
    if a.ndim == 1:                    # (d,)
        return a.astype(float)
    if a.ndim == 2:
        if a.shape[0] == 1:            # (1,d)
            return a[0].astype(float)
        if a.shape[1] == 1:            # (d,1)
            return a[:, 0].astype(float)
        raise ValueError(f"Formato SHAP 2D ambíguo {a.shape}; forneça apenas uma instância.")
    if a.ndim == 3:                     # (n,d,C)
        if a.shape[0] == 1:
            return a[0, :, pos_idx].astype(float)
        raise ValueError(f"Esperado n=1 em a.shape[0]; shape {a.shape} recebido.")

    raise RuntimeError("Formato de shap_values não reconhecido em _phi_from_shap_output_dir_soudness.")

def _sign_from_weight_dir_soudness(w: float, abs_max: float, eps_rel: float = 0.01) -> int:
    """
    Converte um peso (φ_i) em {-1,0,+1}, tratando valores muito pequenos como neutros.
    eps_rel=0.01 => abaixo de 1% do |peso|max local conta como 0.
    """
    thr = eps_rel * (abs_max if abs_max > 0 else 1.0)
    if abs(w) <= thr:
        return 0
    return 1 if w > 0 else -1

def calcula_directional_soundness_shap(
    *,
    x_row_df,                 # Series / 1xD DataFrame / 1D array (uma instância)
    model,                    # modelo com .predict_proba
    X_ref: pd.DataFrame,
    background: pd.DataFrame = None,
    drop_cols=('classe','y'),
    pos_class=1,

    # --- como obter R e GT ---
    direcao_esperada: dict | None = None,   # {"feat": -1|0|+1} se R_mode="given"
    R_mode: str = "given",                  # "given" | "topk_shap" | "auto_local"
    topk: int = 10,                         # usado em "topk_shap"
    tau: float = 0.01,                      # usado em "auto_local"
    tau_mode: str = "abs",                  # "abs" | "rel_fx"
    masking: str = "mean_mode",             # "mean_mode"|"median_mode"|"zero"|"sample"|"background"
    baseline_values: dict | None = None,    # overrides por coluna

    # --- interpretação do sinal da explicação ---
    sign_eps_rel: float = 0.01,             # até 1% do |φ|max vira 0 (neutro)
    count_zero_matches: bool = True,        # se GT=0 e φ≈0 contam como acerto

    return_details: bool = True
):
    """
    Retorna: (score_percent, frac_raw, details)
      - score_percent (0–100) = 100 * frac_raw
      - frac_raw (0–1)        = fração de coincidências de sinal em R
    """
    EPS = 1e-12

    # --- helpers locais (reaproveitam sua convenção) ---
    def _mask_value_for(col: str):
        if baseline_values and col in baseline_values:
            return baseline_values[col]
        s = X_ref[col]
        if pd.api.types.is_numeric_dtype(s):
            if masking == "median_mode":
                return float(s.median())
            elif masking == "zero":
                return 0.0
            elif masking == "sample":
                vals = s.dropna().values
                return float(np.random.choice(vals)) if len(vals) else float(s.mean())
            else:  # "mean_mode"
                return float(s.mean())
        # categóricas
        if masking == "sample":
            vals = s.dropna().values
            if len(vals):
                return np.random.choice(vals)
        if masking == "background":
            # será usado linha a linha (abaixo)
            return None
        mode = s.mode()
        return mode.iloc[0] if not mode.empty else (s.dropna().iloc[0] if s.dropna().size else None)

    # --- espaço efetivo e formatos ---
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_dir_soudness(x_row_df, feat_cols)

    # índice da classe positiva
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # f(x) (prob)
    fx = float(model.predict_proba(x1[feat_cols])[:, pos_idx][0])

    # SHAP φ(x)
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")
    out = expl.shap_values(x1[feat_cols])
    phi = _phi_from_shap_output_dir_soudness(out, expl, pos_idx)  # (d,)
    abs_max_phi = float(np.max(np.abs(phi))) if phi.size else 0.0

    # --- definir R e GT_dir ---
    R_feats: list[str] = []
    GT_dir: dict[str, int] = {}

    if R_mode == "given":
        if not direcao_esperada:
            raise ValueError("R_mode='given' requer direcao_esperada={'feat': -1|0|+1}.")
        for feat, gt in direcao_esperada.items():
            if feat in feat_cols:
                R_feats.append(feat)
                GT_dir[feat] = int(np.sign(gt)) if gt in (-1, 0, 1) else int(np.sign(gt))

    elif R_mode == "topk_shap":
        ranked = np.argsort(-np.abs(phi))[:min(topk, len(feat_cols))]
        R_feats = [feat_cols[j] for j in ranked]
        # inferir direção esperada medindo efeito em f(x)
        for feat in R_feats:
            j = feat_cols.index(feat)
            xm = x1.copy()
            if masking == "background":
                b = bg.sample(1).iloc[0, :]
                xm.iloc[0, j] = b[feat]
            else:
                xm.iloc[0, j] = _mask_value_for(feat)
            fx_mask = float(model.predict_proba(xm[feat_cols])[:, pos_idx][0])
            delta = fx - fx_mask  # >0: presença/valor atual AUMENTA f(x)
            GT_dir[feat] = 1 if delta > 0 else (-1 if delta < 0 else 0)

    elif R_mode == "auto_local":
        for j, feat in enumerate(feat_cols):
            xm = x1.copy()
            if masking == "background":
                b = bg.sample(1).iloc[0, :]
                xm.iloc[0, j] = b[feat]
            else:
                xm.iloc[0, j] = _mask_value_for(feat)
            fx_mask = float(model.predict_proba(xm[feat_cols])[:, pos_idx][0])
            delta = fx - fx_mask
            thr = (tau * fx if str(tau_mode).lower() == "rel_fx" else tau)
            if abs(delta) > thr:
                R_feats.append(feat)
                GT_dir[feat] = 1 if delta > 0 else -1
    else:
        raise ValueError("R_mode deve ser 'given', 'topk_shap' ou 'auto_local'.")

    # --- avaliar coincidência de sinais ---
    matches = 0
    considered = 0
    per_feat = []

    for feat in R_feats:
        j = feat_cols.index(feat)
        s_hat = _sign_from_weight_dir_soudness(float(phi[j]), abs_max_phi, eps_rel=sign_eps_rel)  # {-1,0,+1}
        s_exp = GT_dir.get(feat, 0)

        if s_exp == 0 or s_hat == 0:
            ok = (s_exp == 0 and s_hat == 0 and count_zero_matches)
        else:
            ok = (s_exp == s_hat)

        considered += 1
        if ok:
            matches += 1

        per_feat.append({
            "feature": feat,
            "phi": float(phi[j]),
            "sign_shap": int(s_hat),
            "sign_expected": int(s_exp),
            "match": bool(ok),
        })

    frac = (matches / considered) if considered > 0 else np.nan
    score_pct = float(frac * 100.0) if considered > 0 else np.nan

    details = {
        "fx": fx,
        "R_mode": R_mode,
        "R_size": considered,
        "R_feats": R_feats,
        "topk": topk,
        "tau": tau,
        "tau_mode": tau_mode,
        "masking": masking,
        "sign_eps_rel": sign_eps_rel,
        "count_zero_matches": count_zero_matches,
        "per_feature": per_feat
    }

    if return_details:
        return score_pct, frac, details
    return score_pct, frac

# ===== Helpers (com sufixo _sstability) =====
import numpy as np, pandas as pd, shap
from typing import Sequence, Optional, Dict, Tuple, List

def _ensure_row_df_sstability(x_row_df, feat_cols_eff):
    """Garante DataFrame 1×d com colunas=feat_cols_eff (e somente 1 linha)."""
    if isinstance(x_row_df, pd.DataFrame):
        df = x_row_df.copy()
    elif isinstance(x_row_df, pd.Series):
        df = x_row_df.to_frame().T
    else:
        arr = np.asarray(x_row_df)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        df = pd.DataFrame(arr, columns=feat_cols_eff if arr.shape[1] == len(feat_cols_eff) else None)
    if set(feat_cols_eff).issubset(df.columns):
        df = df.loc[:, feat_cols_eff]
    else:
        df = pd.DataFrame(df.to_numpy().reshape(1, -1), columns=feat_cols_eff)
    return df.iloc[[0]]

def _phi_from_shap_output_sstability(out, explainer, pos_idx: int):
    """
    Converte saídas diversas do SHAP para vetor 1D (d,) da classe pos_idx.
    Aceita shap.Explanation, lista/tupla por classe, np.ndarray 1D/2D/3D.
    """
    try:
        if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
            v = np.asarray(out.values, float)
            v = np.squeeze(v)
            if v.ndim == 1:  return v.astype(float)
            if v.ndim == 2 and v.shape[0] == 1:  return v[0].astype(float)
            if v.ndim == 3 and v.shape[0] == 1:  return v[0, :, pos_idx].astype(float)
    except Exception:
        pass

    if isinstance(out, (list, tuple)):
        v = np.asarray(out[pos_idx], float)
        v = np.squeeze(v)
        if v.ndim == 1: return v
        if v.ndim == 2 and v.shape[0] == 1: return v[0]
        raise ValueError(f"Esperada 1 instância; recebido shape {v.shape}.")

    a = np.asarray(out)
    if a.ndim == 1: return a.astype(float)
    if a.ndim == 2:
        if a.shape[0] == 1: return a[0].astype(float)
        if a.shape[1] == 1: return a[:,0].astype(float)
        raise ValueError(f"Formato SHAP 2D ambíguo: {a.shape}.")
    if a.ndim == 3:
        if a.shape[0] == 1: return a[0, :, pos_idx].astype(float)
        raise ValueError(f"Esperado n=1 em a.shape[0]; recebido {a.shape}.")
    raise RuntimeError("Formato de shap_values não reconhecido (sstability).")

def _split_num_cat_sstability(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Tuple[List[int], List[int]]:
    num_idx, cat_idx = [], []
    for j, c in enumerate(feat_cols_eff):
        (num_idx if pd.api.types.is_numeric_dtype(X_ref[c]) else cat_idx).append(j)
    return num_idx, cat_idx

def _bounds_numeric_sstability(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str]) -> Dict[int, Tuple[float, float]]:
    b = {}
    for j, c in enumerate(feat_cols_eff):
        if pd.api.types.is_numeric_dtype(X_ref[c]):
            v = X_ref[c].to_numpy()
            b[j] = (float(np.nanmin(v)), float(np.nanmax(v)))
    return b

def _fit_quantile_edges_sstability(X_ref: pd.DataFrame, feat_cols_eff: Sequence[str], num_bins: int = 4) -> Dict[int, Optional[np.ndarray]]:
    edges_map = {}
    for j, c in enumerate(feat_cols_eff):
        s = X_ref[c]
        if not pd.api.types.is_numeric_dtype(s):
            edges_map[j] = None
            continue
        qs = np.linspace(0, 1, num_bins+1)
        edges = np.unique(np.quantile(s.dropna().values, qs))
        edges_map[j] = edges if len(edges) >= 2 else None
    return edges_map

def _bucket_index_sstability(val: float, edges: np.ndarray) -> int:
    idx = np.searchsorted(edges, val, side='right') - 1
    return int(np.clip(idx, 0, len(edges)-2))

def _dist_to_nearest_edge_sstability(val: float, edges: Optional[np.ndarray]) -> float:
    if edges is None or len(edges) < 2:
        return np.inf
    i = _bucket_index_sstability(val, edges)
    left, right = edges[i], edges[i+1]
    return float(min(abs(val-left), abs(right-val)))

# ===== Stability para SHAP =====
def calcula_stability_shap(
    *,
    x_row_df,                    # DataFrame/Series/array de 1 instância
    model,                       # modelo treinado (ex.: RandomForest)
    X_ref: pd.DataFrame,         # referencial p/ tipos, bounds, baselines
    background: pd.DataFrame = None,    # background SHAP (ex.: X_train/X_sel)
    drop_cols=('classe','y'),
    pos_class=1,                 # valor da classe positiva (ex.: 1)
    n_trials: int = 200,         # nº de vizinhos x+δ
    norm: str = "l2",            # 'l2' ou 'l1'

    # --- ruído p/ numéricas ---
    sigma_mode: str = "adaptive",  # 'adaptive' | 'relative_range' | 'absolute'
    sigma: float = 0.05,           # se 'relative_range' ou 'absolute'
    alpha: float = 0.02,           # 'adaptive': fração do range
    beta: float = 0.10,            # 'adaptive': fração do std
    num_bins: int = 4,             # p/ estimar “bordas” (evitar cruzar intervalos)
    avoid_bucket_crossing: bool = True,

    # --- ruído p/ categóricas ---
    p_flip: float = 0.05,          # prob. de flip categórico

    random_state: int = 42
):
    """
    Stability_SHAP(x) = E_δ[ || Φ(x) − Φ(x+δ) || ], δ ~ perturbação pequena.
    Retorna: stability_mean, stability_std, stability_score(0–100), details(dict).
    """
    rng = np.random.default_rng(random_state)

    # colunas efetivas e normalização do x
    feat_cols = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_sstability(x_row_df, feat_cols)

    # índice da classe positiva
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # Explainer SHAP e Φ(x)
    bg = background[feat_cols] if background is not None else X_ref[feat_cols]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")
    out0 = expl.shap_values(x1[feat_cols])
    phi0 = _phi_from_shap_output_sstability(out0, expl, pos_idx)  # (d,)

    # preparação para perturbações
    num_idx, cat_idx = _split_num_cat_sstability(X_ref, feat_cols)
    bounds   = _bounds_numeric_sstability(X_ref, feat_cols)
    edges_mp = _fit_quantile_edges_sstability(X_ref, feat_cols, num_bins=num_bins)

    # sigma por feature numérica
    sigma_map: Dict[int, float] = {}
    x_vec = x1.iloc[0].values
    for j in num_idx:
        lo, hi = bounds.get(j, (x_vec[j], x_vec[j]))
        span = max(0.0, hi - lo)
        stdj = float(X_ref.iloc[:, j].std(ddof=0)) if span > 0 else 0.0
        if sigma_mode == "relative_range":
            sigma_map[j] = sigma * (span if span > 0 else 1.0)
        elif sigma_mode == "absolute":
            sigma_map[j] = float(sigma)
        elif sigma_mode == "adaptive":
            dist_edge = _dist_to_nearest_edge_sstability(float(x_vec[j]), edges_mp.get(j))
            cands = []
            if span > 0: cands.append(alpha * span)
            if stdj > 0: cands.append(beta * stdj)
            if np.isfinite(dist_edge): cands.append(0.45 * dist_edge)
            sigma_map[j] = max(1e-9, min(cands)) if cands else 1e-3
        else:
            raise ValueError("sigma_mode deve ser 'adaptive', 'relative_range' ou 'absolute'.")

    # categorias: lista de valores possíveis
    cats_by_col = {}
    for j in cat_idx:
        col = feat_cols[j]
        vals = pd.Series(X_ref[col].dropna().unique()).tolist()
        cats_by_col[j] = vals

    # laço de amostras
    dists = []
    accum_abs_delta = np.zeros_like(phi0)
    bucket_cross_events = 0
    bucket_checks = 0

    for _ in range(int(n_trials)):
        xprime = x1.copy()

        # numéricas com jitter
        for j in num_idx:
            sig = sigma_map[j]
            noise = float(rng.normal(0.0, sig))
            cand = float(x_vec[j]) + noise

            edges = edges_mp.get(j)
            if edges is not None:
                bucket_checks += 1
                if _bucket_index_sstability(cand, edges) != _bucket_index_sstability(float(x_vec[j]), edges):
                    bucket_cross_events += 1
                if avoid_bucket_crossing:
                    i = _bucket_index_sstability(float(x_vec[j]), edges)
                    left, right = edges[i], edges[i+1]
                    eps = 1e-9
                    cand = min(max(cand, left+eps), right-eps)

            lo, hi = bounds.get(j, (cand, cand))
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                cand = min(max(cand, lo), hi)

            xprime.iat[0, j] = cand

        # categóricas com flip ocasional
        for j in cat_idx:
            if rng.random() < p_flip and len(cats_by_col.get(j, [])) > 1:
                cur = x1.iat[0, j]
                others = [c for c in cats_by_col[j] if c != cur]
                if others:
                    xprime.iat[0, j] = rng.choice(others)

        # SHAP(x') e distância
        outp = expl.shap_values(xprime[feat_cols])
        phip = _phi_from_shap_output_sstability(outp, expl, pos_idx)

        if norm == "l2":
            dist = float(np.linalg.norm(phi0 - phip))
        elif norm == "l1":
            dist = float(np.linalg.norm(phi0 - phip, ord=1))
        else:
            raise ValueError("norm deve ser 'l2' ou 'l1'.")
        dists.append(dist)
        accum_abs_delta += np.abs(phi0 - phip)

    # agregações
    stability_mean = float(np.mean(dists)) if dists else 0.0
    stability_std  = float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0

    base_norm = float(np.linalg.norm(phi0)) if norm == "l2" else float(np.linalg.norm(phi0, ord=1))
    eps = 1e-12
    ratio = stability_mean / (base_norm + eps)
    stability_score = 100.0 * max(0.0, 1.0 - min(1.0, ratio))   # 0–100 (maior = melhor)

    mean_abs_delta = accum_abs_delta / max(1, n_trials)
    top_idx = np.argsort(-mean_abs_delta)[:min(8, mean_abs_delta.size)]
    top_changes = [
        {"feature": feat_cols[j], "mean_abs_delta_phi": float(mean_abs_delta[j]), "phi0": float(phi0[j])}
        for j in top_idx
    ]

    details = {
        "norm": norm,
        "n_trials": int(n_trials),
        "sigma_mode": sigma_mode,
        "sigma": sigma,
        "alpha": alpha,
        "beta": beta,
        "num_bins": int(num_bins),
        "avoid_bucket_crossing": bool(avoid_bucket_crossing),
        "p_flip": float(p_flip),
        "n_lime_repeats": 0,                   # compat. com seu make_stability_row
        "base_norm_w0": base_norm,             # nome mantido p/ compatibilidade
        "base_norm_phi0": base_norm,           # nome mais descritivo
        "bucket_cross_rate": (bucket_cross_events / bucket_checks) if bucket_checks > 0 else 0.0,
        "sigma_map_summary": {
            "mean": float(np.mean(list(sigma_map.values()))) if sigma_map else 0.0,
            "min":  float(np.min(list(sigma_map.values())))  if sigma_map else 0.0,
            "max":  float(np.max(list(sigma_map.values())))  if sigma_map else 0.0,
        },
        "top_changes": top_changes,
        "dists": dists,
        "pos_class_index": int(pos_idx),
    }
    return stability_mean, stability_std, stability_score, details

def _ensure_row_df_ssufficiency(x_row_df, feat_cols_eff):
    """Normaliza para DataFrame 1×d alinhado a feat_cols_eff."""
    if isinstance(x_row_df, pd.DataFrame):
        df = x_row_df.copy()
    elif isinstance(x_row_df, pd.Series):
        df = x_row_df.to_frame().T
    else:
        arr = np.asarray(x_row_df)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        df = pd.DataFrame(arr, columns=feat_cols_eff if arr.shape[1] == len(feat_cols_eff) else None)
    if set(feat_cols_eff).issubset(df.columns):
        df = df.loc[:, feat_cols_eff]
    else:
        df = pd.DataFrame(df.to_numpy().reshape(1, -1), columns=feat_cols_eff)
    return df.iloc[[0]]

def _phi_from_shap_output_ssufficiency(out, explainer, pos_idx: int):
    """
    Converte várias saídas do SHAP para vetor 1D (d,) da classe pos_idx.
    """
    try:
        if hasattr(shap, "Explanation") and isinstance(out, shap.Explanation):
            v = np.asarray(out.values, float)
            v = np.squeeze(v)
            if v.ndim == 1:  return v.astype(float)
            if v.ndim == 2 and v.shape[0] == 1:  return v[0].astype(float)
            if v.ndim == 3 and v.shape[0] == 1:  return v[0, :, pos_idx].astype(float)
    except Exception:
        pass
    if isinstance(out, (list, tuple)):
        v = np.asarray(out[pos_idx], float)
        v = np.squeeze(v)
        if v.ndim == 1: return v
        if v.ndim == 2 and v.shape[0] == 1: return v[0]
        raise ValueError(f"Esperada 1 instância; recebido shape {v.shape}.")
    a = np.asarray(out)
    if a.ndim == 1: return a.astype(float)
    if a.ndim == 2:
        if a.shape[0] == 1: return a[0].astype(float)
        if a.shape[1] == 1: return a[:,0].astype(float)
        raise ValueError(f"Formato SHAP 2D ambíguo: {a.shape}.")
    if a.ndim == 3:
        if a.shape[0] == 1: return a[0, :, pos_idx].astype(float)
        raise ValueError(f"Esperado n=1 em a.shape[0]; recebido {a.shape}.")
    raise RuntimeError("Formato de shap_values não reconhecido (ssufficiency).")

def _build_baselines_ssufficiency(
    X_ref: pd.DataFrame,
    feat_cols_eff: Sequence[str],
    masking: str,
    baseline_values: Optional[Dict[str, object]],
    background_df: Optional[pd.DataFrame]
):
    """
    Decide o valor de máscara para cada coluna.
    masking: 'median_mode' (padrão) | 'mean_mode' | 'zero' | 'sample' | 'background_row'
    - 'background_row': será sorteada uma linha inteira depois (retorna callable por coluna).
    - 'sample' (por coluna): amostra 1 valor observado em X_ref.
    """
    baselines = {}
    masking = (masking or "median_mode").lower()

    for c in feat_cols_eff:
        # prioridade: baseline explícito do usuário
        if baseline_values and (c in baseline_values):
            baselines[c] = baseline_values[c]
            continue

        s = X_ref[c].dropna()
        is_num = pd.api.types.is_numeric_dtype(X_ref[c])

        if masking == "zero" and is_num:
            baselines[c] = 0.0
        elif masking in ("mean_mode", "median_mode"):
            if is_num:
                baselines[c] = float(s.mean()) if masking == "mean_mode" else float(s.median())
            else:
                baselines[c] = (s.mode().iloc[0] if not s.mode().empty else (s.iloc[0] if len(s) else None))
        elif masking == "sample":
            # uma função que amostra 1 valor da coluna quando chamada
            def _mk_sampler(col=s):
                def _fn():
                    return col.sample(1).iloc[0] if len(col) else None
                return _fn
            baselines[c] = _mk_sampler()
        elif masking == "background_row":
            # usa a linha (amostrada) do background: retorna callable para pegar o valor da coluna
            bg = background_df if background_df is not None else X_ref
            bg = bg[feat_cols_eff]
            def _mk_bg_sampler(col_name=c, bg_df=bg):
                def _fn():
                    r = bg_df.sample(1).iloc[0]
                    return r[col_name]
                return _fn
            baselines[c] = _mk_bg_sampler()
        else:
            # fallback
            if is_num:
                baselines[c] = float(s.median()) if len(s) else 0.0
            else:
                baselines[c] = (s.mode().iloc[0] if not s.mode().empty else (s.iloc[0] if len(s) else None))
    return baselines

def _mask_x_keep_S_ssufficiency(x_vec: np.ndarray,
                                feat_cols_eff: Sequence[str],
                                S_idx: set,
                                baselines: Dict[str, object]) -> np.ndarray:
    """
    Constrói x|S: mantém valores de x APENAS nas features de S; fora de S aplica baseline.
    Aceita callable como baseline (amostra dinâmica).
    """
    xS = x_vec.copy()
    for j, c in enumerate(feat_cols_eff):
        if j in S_idx:
            continue
        b = baselines[c]
        xS[j] = (b() if callable(b) else b)
    return xS

def make_sufficiency_debug_row_sufficiency(inst_id, details, decimals: int = 8):
    """
    Monta uma linha de debug para Sufficiency (SHAP).
    Espera o dicionário `details` retornado por `calcula_sufficiency_shap`.

    Campos extras incluídos para SHAP:
      - class_index: índice da classe usada (pos_idx)
      - phi_dense  : dict {feature -> phi_i(x)} arredondado
    """
    r = {"instancia": int(inst_id), "fx": round(float(details["fx"]), decimals)}

    # f(x|S_k) em colunas separadas
    for lbl, val in details.get("fx_Sk", {}).items():
        r[f"fx_S_{lbl}"] = round(float(val), decimals)

    # quedas f(x) - f(x|S_k)
    for lbl, val in details.get("drop_Sk", {}).items():
        r[f"drop_S_{lbl}"] = round(float(val), decimals)

    # metadados
    r["masking"]        = details.get("masking")
    r["ranking"]        = details.get("ranking")
    r["Ks"]             = details.get("Ks")
    r["topK_features"]  = details.get("topK_features")

    # extras SHAP
    if "class_index" in details:
        r["class_index"] = int(details["class_index"])
    if "phi_dense" in details:
        r["phi_dense"] = {k: round(float(v), decimals) for k, v in details["phi_dense"].items()}

    return r

def calcula_sufficiency_shap(
    *,
    x_row_df,                         # DataFrame/Series/array 1×d
    model,
    X_ref: pd.DataFrame,
    background: pd.DataFrame = None,  # recomendado: X_train ou X_sel
    drop_cols=('classe','y'),
    pos_class=1,
    percent_list: Sequence[float] = (0.01, 0.05, 0.10, 0.20),
    ranking: str = "abs",             # 'abs' por |phi|, 'signed' por phi
    masking: str = "median_mode",     # 'median_mode'|'mean_mode'|'zero'|'sample'|'background_row'
    baseline_values: Optional[Dict[str, object]] = None,
    f_is_proba: bool = True,
    random_state: int = 42
):
    """
    Implementa:  SUFF_K(x) = f_y(x) − f_y(x|S_K(x))
      - S_K(x): top-K features por |phi_i(x)| (ou por phi_i se ranking='signed').
      - x|S_K: mantém valores de x apenas em S_K e mascara o resto com um baseline.
    Retorna (scores_dict, details)
    """
    rng = np.random.default_rng(random_state)

    # espaço efetivo + formatos
    feat_cols_eff = [c for c in X_ref.columns if c not in set(drop_cols)]
    x1 = _ensure_row_df_ssufficiency(x_row_df, feat_cols_eff)

    # índice da classe positiva
    try:
        classes = list(getattr(model, "classes_", [0, 1]))
    except Exception:
        classes = [0, 1]
    pos_idx = classes.index(pos_class) if pos_class in classes else int(np.argmax(classes))

    # f(x)
    fx = float(model.predict_proba(x1[feat_cols_eff])[:, pos_idx][0])

    # SHAP: TreeExplainer (ajuste se o seu modelo não for de árvore)
    bg = background[feat_cols_eff] if background is not None else X_ref[feat_cols_eff]
    expl = shap.TreeExplainer(model, data=bg, model_output="probability")
    out = expl.shap_values(x1[feat_cols_eff])
    phi = _phi_from_shap_output_ssufficiency(out, expl, pos_idx)  # (d,)

    # ranking dos índices
    if ranking == "signed":
        order = np.argsort(-phi)
    else:
        order = np.argsort(-np.abs(phi))

    # baselines
    baselines = _build_baselines_ssufficiency(
        X_ref=X_ref, feat_cols_eff=feat_cols_eff,
        masking=masking, baseline_values=baseline_values,
        background_df=bg
    )

    # para cada %K, monta S_K e calcula f(x|S_K)
    d = len(feat_cols_eff)
    eps = 1e-12
    k_list, fx_Sk, drops, S_lists = [], [], [], []
    x_vec = x1.iloc[0].values

    for p in percent_list:
        K = int(np.ceil(max(1, float(p) * d)))
        top_idx = set(order[:K].tolist())
        xS = _mask_x_keep_S_ssufficiency(x_vec, feat_cols_eff, top_idx, baselines)
        fxS = float(model.predict_proba(pd.DataFrame([xS], columns=feat_cols_eff))[:, pos_idx][0])
        drop = fx - fxS

        k_list.append(K)
        fx_Sk.append(fxS)
        drops.append(drop)
        # nomes das features no Top-K (ordenadas por |phi| desc)
        S_lists.append([feat_cols_eff[j] for j in order[:K]])

    # normalização 0–100 (maior=melhor; queda menor => suficiência maior)
    scores = []
    for drop in drops:
        if f_is_proba:
            scores.append(100.0 * (1.0 - np.clip(drop / max(fx, eps), 0.0, 1.0)))
        else:
            scores.append(100.0 * (1.0 - np.clip(drop / (abs(fx) + eps), 0.0, 1.0)))

    aopc_raw = float(np.mean(drops)) if drops else 0.0
    if f_is_proba:
        aopc_score = 100.0 * (1.0 - np.clip(aopc_raw / max(fx, eps), 0.0, 1.0))
    else:
        aopc_score = 100.0 * (1.0 - np.clip(aopc_raw / (abs(fx) + eps), 0.0, 1.0))

    labels = [f"{int(p*100)}%" for p in percent_list]
    scores_dict = {lbl: float(sc) for lbl, sc in zip(labels, scores)}
    scores_dict["AOPC_suff(%)"] = float(aopc_score)
    scores_dict["AOPC_suff_raw"] = float(aopc_raw)

    details = {
        "fx": fx,
        "Ks": k_list,
        "percents": labels,
        "fx_Sk": {lbl: val for lbl, val in zip(labels, fx_Sk)},
        "drop_Sk": {lbl: val for lbl, val in zip(labels, drops)},
        "topK_features": {lbl: feats for lbl, feats in zip(labels, S_lists)},
        "phi_dense": {feat_cols_eff[i]: float(phi[i]) for i in range(d)},
        "masking": masking,
        "ranking": ranking,
        "class_index": pos_idx
    }
    return scores_dict, details

def make_sufficiency_row_sufficiency(
    inst_id,
    scores_dict: dict,
    percent_labels=("1%", "5%", "10%", "20%"),
    decimals=3
):
    """
    Monta uma linha-resumo para Sufficiency (SHAP).

    Parâmetros
    ----------
    inst_id : int
        Id/índice da instância.
    scores_dict : dict
        Dicionário retornado por `calcula_sufficiency_shap` com chaves como
        '1%', '5%', '10%', '20%', 'AOPC_suff(%)', 'AOPC_suff_raw'.
    percent_labels : tuple[str]
        Quais percentuais incluir nas colunas (na ordem desejada).
    decimals : int
        Casas decimais para os percentuais (em %).

    Retorno
    -------
    dict: linha pronta para DataFrame.
    """
    row = {"instancia": int(inst_id)}

    # colunas para percentuais escolhidos
    for k in percent_labels:
        if k in scores_dict and scores_dict[k] is not None:
            row[f"suff_{k}"] = round(float(scores_dict[k]), decimals)

    # AOPC (score em %) e valor bruto
    aopc_pct = scores_dict.get("AOPC_suff(%)", None)
    aopc_raw = scores_dict.get("AOPC_suff_raw", None)
    if aopc_pct is not None:
        row["suff_AOPC(%)"] = round(float(aopc_pct), decimals)
    if aopc_raw is not None:
        row["suff_AOPC_raw"] = round(float(aopc_raw), 6)

    return row


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_fidelity_shapley_diagnostics(details, titulo="Fidelity SHAP – diagnóstico", save_path=None):
    """
    details: dicionário retornado por calcula_fidelity_shapley (metric='r2' ou 'logloss')
             deve conter:
               - "delta_real": array-like
               - "delta_hat":  array-like
    titulo: título geral da figura
    save_path: se não for None, salva a figura nesse caminho
    """
    # --- extrai vetores ---
    delta_real = np.asarray(details.get("delta_real", []), dtype=float)
    delta_hat  = np.asarray(details.get("delta_hat",  []), dtype=float)

    # limpa NaNs/inf
    mask = np.isfinite(delta_real) & np.isfinite(delta_hat)
    delta_real = delta_real[mask]
    delta_hat  = delta_hat[mask]

    if delta_real.size == 0:
        raise ValueError("details['delta_real'] ou details['delta_hat'] está vazio ou só com NaN.")

    # resíduos
    resid = delta_hat - delta_real

    # estatísticas básicas
    mse   = float(np.mean(resid**2))
    var_y = float(np.var(delta_real))
    r2    = 1.0 - mse / (var_y + 1e-12)
    r2    = float(np.clip(r2, 0.0, 1.0))

    # regressão linear simples delta_hat ≈ a + b * delta_real
    x = delta_real
    y = delta_hat
    b1, b0 = np.polyfit(x, y, deg=1)  # y ≈ b0 + b1*x
    y_pred_line = b0 + b1 * x
    # R² da regressão
    ss_res = np.sum((y - y_pred_line)**2)
    ss_tot = np.sum((y - np.mean(y))**2) + 1e-12
    r2_reg = 1.0 - ss_res/ss_tot

    # ================= FIGURA =================
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), dpi=120)
    fig.suptitle(titulo, fontsize=16, fontweight="bold", y=0.97)

    # ---------------- (1) Scatter Δ_real vs Δ_pred + linha de regressão ----------------
    ax = axes[0, 0]
    ax.scatter(delta_real, delta_hat, s=12, alpha=0.5, edgecolor="none")
    # linha identidade
    lim_min = min(delta_real.min(), delta_hat.min())
    lim_max = max(delta_real.max(), delta_hat.max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=1.2, color="gray", label="Identidade")
    # linha de regressão
    xs_line = np.linspace(lim_min, lim_max, 100)
    ax.plot(xs_line, b0 + b1*xs_line, color="black", linewidth=1.4, label="Regressão linear")

    ax.set_xlabel("Δ_real  (f(x) − f(x_{−I}))")
    ax.set_ylabel("Δ_pred  (∑ φ_i em I)")
    ax.set_title(f"Scatter Δ_real vs Δ_pred\nR²(fidelity) = {r2:.3f} | R²(regressão) = {r2_reg:.3f}")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)

    # ---------------- (2) Heatmap 2D (densidade) ----------------
    ax = axes[0, 1]
    hb = ax.hexbin(
        delta_real,
        delta_hat,
        gridsize=40,
        cmap="viridis",
        mincnt=1
    )
    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", linewidth=1.0, color="white")
    ax.set_xlabel("Δ_real")
    ax.set_ylabel("Δ_pred")
    ax.set_title("Densidade de pontos (Δ_real vs Δ_pred)")
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Contagem")

    # ---------------- (3) Resíduo vs Δ_real ----------------
    ax = axes[1, 0]
    ax.scatter(delta_real, resid, s=10, alpha=0.5, edgecolor="none")
    ax.axhline(0.0, linestyle="--", linewidth=1.0, color="gray")
    ax.set_xlabel("Δ_real")
    ax.set_ylabel("Resíduo = Δ_pred − Δ_real")
    ax.set_title("Resíduos em função de Δ_real")
    ax.grid(alpha=0.3)

    # ---------------- (4) Histograma de resíduos ----------------
    ax = axes[1, 1]
    ax.hist(resid, bins=30, alpha=0.8, edgecolor="black")
    ax.axvline(0.0, linestyle="--", linewidth=1.0, color="gray")
    ax.set_xlabel("Resíduo")
    ax.set_ylabel("Frequência")
    ax.set_title(f"Distribuição dos resíduos\nMSE = {mse:.4f}")

    plt.tight_layout(rect=[0, 0.0, 1, 0.94])

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"✅ Figura de diagnostics de fidelity SHAP salva em: {save_path}")

    plt.show()


In [ ]:
N_JOBS = 4

In [ ]:
# ===  Shapley Interpretabilidade [Cálculos de Métricas]===
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
import time
import multiprocessing
from joblib import Parallel, delayed, parallel_backend
import psutil, os
from pathlib import Path  # <<< NOVO

np.random.seed(42)

# --- garantir que BASE_PATH seja Path (corrige erro do "/") ---
if 'BASE_PATH' in globals() and not isinstance(BASE_PATH, Path):
    BASE_PATH = Path(BASE_PATH)

# --- garantir espaço de features numéricas iguais ao treino ---
try:
    feat_cols = list(model.feature_names_in_)
except Exception:
    feat_cols = [c for c in X.columns if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(X[c])]

X_feat_sel   = X_sel[feat_cols].copy()
X_feat_train = X_train[feat_cols].copy() if set(feat_cols).issubset(X_train.columns) else X_train.filter(feat_cols).copy()
X_feat_test  = X[feat_cols].copy()

inicio = time.time()
SHAPLE_TABLE_METRICS         = []
SHAPLE_FIDELITY_VALUES       = []
SHAPLE_INFIDELITY_VALUES     = []
SHAPLE_FAITHFULLNESS_VALUES  = []
SHAPLE_CALIBRATION_VALUES    = []
SHAPLE_SELECTIVITY_VALUES    = []
SHAPLE_SOUNDNESS_VALUES      = []
SHAPLE_RELEVANCE_VALUES      = []
SHAPLE_SIMPLICITY_VALUES     = []
SHAPLE_ROBUSTENESS_VALUES    = []
SHAPLE_ROBUSTENESS_ADV_VALUES= []
SHAPLE_STABILITY_VALUES      = []
SHAPLE_MONOTONIA_VALUES      = []
SHAPLE_COMPLETENESS_VALUES   = []
SHAPLE_SUFFICIENCY_VALUES    = []
SHAPLE_CONSISTENCY_VALUES    = []
SHAPLE_CLASS_CONSISTENCY_VALUES = []
SHAPLE_DISCRIMINABILITY_VALUES =[]
SHAPLE_DISPARITY_VALUES = []
SHAPLE_SUFFICIENCY_1_VALUES  =  []
SHAPLE_SUFFICIENCY_5_VALUES  =  []
SHAPLE_SUFFICIENCY_10_VALUES = []
SHAPLE_SUFFICIENCY_20_VALUES = []
SHAPLE_DIR_SOUNDNESS_VALUES  = []

# --- helper seguro para converter valores para float (evita 'N/A', '0,123', etc.) ---
def _to_float_safe(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace('%', '').replace(',', '.')
    try:
        return float(s)
    except Exception:
        return np.nan

# --- mapeia id (em ENTRADAS_SELECIONADAS) -> índice local (em X_sel/df_shap/shap_mat)
map_global_to_local = {int(id_): li for li, id_ in enumerate(ENTRADAS_SELECIONADAS)}

rows      = []
rows_inf  = []
rows_faith = []
rows_simplicity = []
rows_consistency = []
rows_robustness  = []
rows_selectivity = []
rows_soundness = []
rows_dirsound = []
rows_stability = []
rows_suff = []
rows_suff_debug = []

PERC_LIST = (0.01, 0.05, 0.10, 0.20)  # 1%, 5%, 10%, 20%

# índice da classe positiva (coerente com o modelo)
classes = list(getattr(model, "classes_", [0, 1]))
idx_pos = classes.index(1) if 1 in classes else int(np.argmax(classes))

print(f'Tamanho de entradas selecionadas : {len(ENTRADAS_SELECIONADAS)}')


def _process_one_shap_instance(i, idx_instancia):
    """
    Processa uma instância (posição i em ENTRADAS_SELECIONADAS, id global idx_instancia)
    e retorna tudo o que antes era acumulado nas listas globais.
    """
    inst_start = time.time()
    print(
        f"[SHAP][INST-START] local={i} id={idx_instancia} "
        f"at {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    def timed_call(step_name, func, *args, **kwargs):
        t0 = time.time()
        print(
            f"[SHAP][{step_name}][START] inst={idx_instancia} local={i} "
            f"at {time.strftime('%Y-%m-%d %H:%M:%S')}"
        )
        out = func(*args, **kwargs)
        t1 = time.time()
        print(
            f"[SHAP][{step_name}][END]   inst={idx_instancia} local={i} "
            f"at {time.strftime('%Y-%m-%d %H:%M:%S')} dur={t1 - t0:.2f}s"
        )
        return out

    pos_local = map_global_to_local.get(int(idx_instancia))
    if pos_local is None or pos_local >= len(df_shap):
        print(
            f"[SHAP][INST-SKIP] local={i} id={idx_instancia} "
            f"(pos_local inválido) at {time.strftime('%Y-%m-%d %H:%M:%S')}"
        )
        return None  # segurança

    # Probabilidades reais e aproximadas (df_shap)
    y_real = float(df_shap.loc[pos_local, 'prob_modelo'])
    y_shap = float(df_shap.loc[pos_local, 'soma_SHAP'])

    # ---------- Fidelity ----------
    x_row = X_feat_sel.iloc[[pos_local]]  # 1×D coerente
    fidelity_percent, fidelity_abs, det_fid = timed_call(
        "fidelity",
        calcula_fidelity_shapley,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        n_masks=100,            # 2048 se quiser mais robusto
        mask_rate=0.5,
        baseline="background",  # ou 'mean'/'median'/'zero'/dict/np.array
        target="prob",          # manter coerente com TreeExplainer (probability)
        metric="r2",
        random_state=42,
        return_details=True
    )
    print("Fidelity R² (%):", fidelity_percent, " | R²:", fidelity_abs)

    # Diagnóstico visual
    _ = timed_call(
        "plot_fidelity_diagnostics",
        plot_fidelity_shapley_diagnostics,
        det_fid,
        titulo=f"Fidelity SHAP – instância id: {idx_instancia}",
        save_path=BASE_PATH / f"shap_fidelity_diagnostics_inst_{idx_instancia}.png"
    )

    # ---------- Infidelity ----------
    x_i   = X_feat_sel.iloc[i].values
    phi_i = shap_vals[i]  # SHAP da classe positiva para esta instância

    infidelity_percent, infidelity_abs, det_inf = timed_call(
        "infidelity",
        calcula_infidelity,
        model=model,
        x_row=x_i,
        feature_names=feat_cols,
        pos_class=idx_pos,
        target="prob",
        phi=phi_i,
        background=X_feat_sel,   # baseline (median) será calculado aqui
        baseline="median",
        n_masks=2048,
        p_mask=0.5,
        random_state=42
    )

    row_inf = {
        "instancia_local": int(i),
        "pos_glob": int(idx_instancia),               # id real
        "infidelity_abs": round(infidelity_abs, 8),
        "infidelity_%":   round(infidelity_percent, 3)
    }

    # ---------- Faithfulness ----------
    x_series = X_feat_sel.iloc[i]  # 1×D como Series
    faithfulness_percent, faith_abs, corr_raw, det_faith = timed_call(
        "faithfulness",
        calcula_faithfulness_shap,
        x_row=x_series,
        model=model,
        X_ref=X_feat_test,                 # mesmas colunas do treino
        pos_class=idx_pos,
        shap_explainer=shap_tree_explainer,
        background=X_feat_sel,
        target="logit",
        corr="kendall",
        ref_strategy="background",
        n_ref_samples=128,
        use_abs=False,
        return_details=True
    )
    row_faith = {
        "instancia_local": i,
        "faith_%": round(faithfulness_percent, 3),
        "faith_abs": round(faith_abs, 6),
        "corr_raw": round(corr_raw, 6)
    }

    # ---------- Simplicity ----------
    k_tau, simplicity_percent, detalhamento_simp = timed_call(
        "simplicity",
        calcula_simplicity_shap,
        phi_row=shap_vals[i],
        feature_names=feat_cols,
        tau=0.01,
        return_details=True
    )
    row_simplicity = {
        "instancia": int(i),
        "k_tau": int(k_tau),
        "simplicity_%": round(simplicity_percent, 3)
    }

    # ---------- Consistency entre modelos ----------
    consistency_percent = np.nan
    row_consistency = None
    try:
        x1 = X_feat_sel.iloc[[pos_local]]  # 1×D
        _classes = list(getattr(model, "classes_", [0, 1]))
        _pos_class = _classes.index(1) if 1 in _classes else int(np.argmax(_classes))
        consistency_percent, consistency_abs, det_cons = timed_call(
            "consistency",
            calcula_consistency_shap_models,
            x_row_df=x1,
            X_ref=X_feat_train,
            model_f=model,
            model_fp=model,
            pos_class=_pos_class,
            ref_strategy="mean",
            corr="spearman",
            background_f=X_feat_sel,
            background_fp=X_feat_sel,
            rank_by="raw",
            return_details=True
        )
        row_consistency = {"instancia_local": i, "consistency_%": round(consistency_percent, 3)}
    except NameError:
        consistency_percent = np.nan
    except Exception:
        consistency_percent = np.nan

    # ---------- Robustez ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    rob_max, rob_mean, rob_std, det_rob = timed_call(
        "robustness",
        calcula_robustez_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        n_trials=300,
        epsilon=0.05,
        epsilon_mode="relative_range",
        norm="l2",
        strategy="all",
        features_to_test=None,
        cat_flip_prob=0.10,
        random_state=42,
        topk_delta=8,
        score_mode="relative",
        higher_better=True,
    )
    robustez_percent = det_rob.get("robust_score", np.nan)
    row_robustness = {
        "instancia_local": i,
        "robust_max": float(rob_max),
        "robust_mean": float(rob_mean),
        "robust_std": float(rob_std),
        "robust_score": float(robustez_percent) if not np.isnan(robustez_percent) else np.nan
    }

    # ---------- Completeness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    completeness_percentual, completeness_err, completeness_det = timed_call(
        "completeness",
        calcula_completeness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        target="prob",
        return_details=True
    )

    # ---------- Selectivity ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    selectivity_percent, selectivity_abs, selectivity_det = timed_call(
        "selectivity",
        calcula_selectivity_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        K=10,
        baseline="background",
        target="prob",
        clip_negatives=True,
        return_details=True
    )
    row_selectivity = {
        "instancia_local": i,
        "selectivity_%": float(selectivity_percent),
        "selectivity_abs": float(selectivity_abs)
    }

    # ---------- Soundness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    soundness_percent, soundness_abs, soundness_det = timed_call(
        "soundness",
        calcula_soundness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        U_mode="auto_local",
        tau=0.01,
        masking="mean_mode",
        target="prob",
        return_details=True
    )
    row_soundness = {
        "instancia_local": i,
        "soundness_%": float(soundness_percent),
        "soundness_abs": float(soundness_abs)
    }

    # ---------- Directional Soundness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    dir_soundness_percent, dir_soundness_frac, dir_soundness_det = timed_call(
        "directional_soundness",
        calcula_directional_soundness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        R_mode="auto_local",
        tau=0.01,
        tau_mode="rel_fx",
        masking="mean_mode",
        sign_eps_rel=0.01,
        count_zero_matches=True,
        return_details=True
    )
    row_dirsound = {
        "instancia_local": i,
        "directional_soundness_%": float(dir_soundness_percent),
        "directional_soundness_frac": float(dir_soundness_frac)
    }

    # ---------- Stability ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    stability_mean, stability_std, stability_percent, det_stability = timed_call(
        "stability",
        calcula_stability_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        drop_cols=('classe','y'),
        pos_class=idx_pos,
        n_trials=200,
        norm="l2",
        sigma_mode="adaptive",
        alpha=0.02,
        beta=0.10,
        num_bins=4,
        avoid_bucket_crossing=True,
        p_flip=0.05,
        random_state=42
    )
    row_stability = {
        "instancia_local": i,
        "stability_mean": float(stability_mean),
        "stability_std": float(stability_std),
        "stability_score": float(stability_percent)
    }

    # ---------- Sufficiency ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    scores_suff, det_suff = timed_call(
        "sufficiency",
        calcula_sufficiency_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        drop_cols=('classe','y'),
        pos_class=idx_pos,
        percent_list=(0.01, 0.05, 0.10, 0.20),
        ranking="abs",
        masking="median_mode",
        baseline_values=None,
        f_is_proba=True,
        random_state=42
    )
    row_suff = make_sufficiency_row_sufficiency(idx_instancia, scores_suff)
    row_suff_debug = make_sufficiency_debug_row_sufficiency(idx_instancia, det_suff)

    sufficiency_percent_s1, sufficiency_percent_s5, sufficiency_percent_s10, sufficiency_percent_s20 = (
        scores_suff.get(k, np.nan) for k in ('1%', '5%', '10%', '20%')
    )

    # ---------- Tabela por instância ----------
    table_row = {
        'instancia': idx_instancia,
        'fidelity(%)': f"{fidelity_percent:.3f}",
        'infidelity(%)': f"{infidelity_percent:.3f}",
        'faithfulness(%)': f"{faithfulness_percent:.3f}",
        'simplicity(%)': f"{simplicity_percent:.3f}",
        'consistency(%)': f"{consistency_percent:.3f}" if not np.isnan(consistency_percent) else "NA",
        'robustez(%)': f"{robustez_percent:.3f}" if not np.isnan(robustez_percent) else "NA",
        'completeness(%)': f"{completeness_percentual:.3f}",
        'selectivity(%)': f"{selectivity_percent:.3f}",
        'soundness(%)': f"{soundness_percent:.3f}",
        'directional soundness(%)': f"{dir_soundness_percent:.3f}",
        'stability(%)': f"{stability_percent:.3f}",
        'sufficiency-1(%)': f"{sufficiency_percent_s1:.3f}",
        'sufficiency-5(%)': f"{sufficiency_percent_s5:.3f}",
        'sufficiency-10(%)': f"{sufficiency_percent_s10:.3f}",
        'sufficiency-20(%)': f"{sufficiency_percent_s20:.3f}",
    }

    inst_end = time.time()
    print('#===============================================================================================================#')
    print(
        f'Instância {i+1}/{len(ENTRADAS_SELECIONADAS)} (ID={idx_instancia}) processada. '
        f'Tempo total instância: {inst_end - inst_start:.2f}s '
        f'({time.strftime("%Y-%m-%d %H:%M:%S")})'
    )
    print('#===============================================================================================================#')

    return {
        "idx_instancia": idx_instancia,
        "fidelity_percent": float(fidelity_percent),
        "infidelity_percent": float(infidelity_percent),
        "faithfulness_percent": float(faithfulness_percent),
        "simplicity_percent": float(simplicity_percent),
        "consistency_percent": float(consistency_percent) if not np.isnan(consistency_percent) else np.nan,
        "robustez_percent": float(robustez_percent) if not np.isnan(robustez_percent) else np.nan,
        "completeness_percentual": float(completeness_percentual),
        "selectivity_percent": float(selectivity_percent),
        "soundness_percent": float(soundness_percent),
        "dir_soundness_percent": float(dir_soundness_percent),
        "stability_percent": float(stability_percent),
        "suff1": float(sufficiency_percent_s1),
        "suff5": float(sufficiency_percent_s5),
        "suff10": float(sufficiency_percent_s10),
        "suff20": float(sufficiency_percent_s20),
        "table_row": table_row,
        "row_inf": row_inf,
        "row_faith": row_faith,
        "row_simplicity": row_simplicity,
        "row_consistency": row_consistency,
        "row_robustness": row_robustness,
        "row_selectivity": row_selectivity,
        "row_soundness": row_soundness,
        "row_dirsound": row_dirsound,
        "row_stability": row_stability,
        "row_suff": row_suff,
        "row_suff_debug": row_suff_debug,
    }


# ===========================
# Execução em paralelo
# ===========================
RAM_GB   = psutil.virtual_memory().total / 1024**3
MAX_CPU  = multiprocessing.cpu_count()

NOTEBOOKS_RODANDO = 1   # <<< ALTERE AQUI

#N_JOBS = max(1, int((RAM_GB // 4) // NOTEBOOKS_RODANDO))

with parallel_backend("loky", inner_max_num_threads=1):
    results = Parallel(n_jobs=N_JOBS)(
        delayed(_process_one_shap_instance)(i, idx_instancia)
        for i, idx_instancia in enumerate(ENTRADAS_SELECIONADAS)
    )

# ===========================
# Agregação dos resultados
# ===========================
for res in results:
    if res is None:
        continue

    SHAPLE_TABLE_METRICS.append(res["table_row"])

    SHAPLE_FIDELITY_VALUES.append(_to_float_safe(res["fidelity_percent"]))
    SHAPLE_INFIDELITY_VALUES.append(_to_float_safe(res["infidelity_percent"]))
    SHAPLE_FAITHFULLNESS_VALUES.append(_to_float_safe(res["faithfulness_percent"]))
    SHAPLE_SIMPLICITY_VALUES.append(_to_float_safe(res["simplicity_percent"]))
    SHAPLE_CONSISTENCY_VALUES.append(_to_float_safe(res["consistency_percent"]))
    SHAPLE_ROBUSTENESS_VALUES.append(_to_float_safe(res["robustez_percent"]))
    SHAPLE_COMPLETENESS_VALUES.append(_to_float_safe(res["completeness_percentual"]))
    SHAPLE_SELECTIVITY_VALUES.append(_to_float_safe(res["selectivity_percent"]))
    SHAPLE_SOUNDNESS_VALUES.append(_to_float_safe(res["soundness_percent"]))
    SHAPLE_DIR_SOUNDNESS_VALUES.append(_to_float_safe(res["dir_soundness_percent"]))
    SHAPLE_STABILITY_VALUES.append(_to_float_safe(res["stability_percent"]))
    SHAPLE_SUFFICIENCY_1_VALUES.append(_to_float_safe(res["suff1"]))
    SHAPLE_SUFFICIENCY_5_VALUES.append(_to_float_safe(res["suff5"]))
    SHAPLE_SUFFICIENCY_10_VALUES.append(_to_float_safe(res["suff10"]))
    SHAPLE_SUFFICIENCY_20_VALUES.append(_to_float_safe(res["suff20"]))

    rows_inf.append(res["row_inf"])
    rows_faith.append(res["row_faith"])
    rows_simplicity.append(res["row_simplicity"])
    if res["row_consistency"] is not None:
        rows_consistency.append(res["row_consistency"])
    rows_robustness.append(res["row_robustness"])
    rows_selectivity.append(res["row_selectivity"])
    rows_soundness.append(res["row_soundness"])
    rows_dirsound.append(res["row_dirsound"])
    rows_stability.append(res["row_stability"])
    rows_suff.append(res["row_suff"])
    rows_suff_debug.append(res["row_suff_debug"])


# ===========================
# Linha com médias e ICs
# ===========================
SHAPLE_TABLE_METRICS.append({
    'instancia': 'MÉDIA',
    'fidelity(%)':         formatar_ic_percentual(np.nanmean(SHAPLE_FIDELITY_VALUES),        *calcular_ic95_percentual(SHAPLE_FIDELITY_VALUES)),
    'infidelity(%)':       formatar_ic_percentual(np.nanmean(SHAPLE_INFIDELITY_VALUES),      *calcular_ic95_percentual(SHAPLE_INFIDELITY_VALUES)),
    'faithfulness(%)':     formatar_ic_percentual(np.nanmean(SHAPLE_FAITHFULLNESS_VALUES),   *calcular_ic95_percentual(SHAPLE_FAITHFULLNESS_VALUES)),
    'simplicity(%)':       formatar_ic_percentual(np.nanmean(SHAPLE_SIMPLICITY_VALUES),      *calcular_ic95_percentual(SHAPLE_SIMPLICITY_VALUES)),
    'consistency(%)':      formatar_ic_percentual(
        np.nanmean(SHAPLE_CONSISTENCY_VALUES),
        *calcular_ic95_percentual([v for v in SHAPLE_CONSISTENCY_VALUES if not np.isnan(v)])
    ) if any(not np.isnan(v) for v in SHAPLE_CONSISTENCY_VALUES) else "NA",
    'robustez(%)':         formatar_ic_percentual(
        np.nanmean(SHAPLE_ROBUSTENESS_VALUES),
        *calcular_ic95_percentual([v for v in SHAPLE_ROBUSTENESS_VALUES if not np.isnan(v)])
    ) if any(not np.isnan(v) for v in SHAPLE_ROBUSTENESS_VALUES) else "NA",
    'completeness(%)':     formatar_ic_percentual(np.nanmean(SHAPLE_COMPLETENESS_VALUES),    *calcular_ic95_percentual(SHAPLE_COMPLETENESS_VALUES)),
    'selectivity(%)':      formatar_ic_percentual(np.nanmean(SHAPLE_SELECTIVITY_VALUES),     *calcular_ic95_percentual(SHAPLE_SELECTIVITY_VALUES)),
    'soundness(%)':        formatar_ic_percentual(np.nanmean(SHAPLE_SOUNDNESS_VALUES),       *calcular_ic95_percentual(SHAPLE_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.nanmean(SHAPLE_DIR_SOUNDNESS_VALUES), *calcular_ic95_percentual(SHAPLE_DIR_SOUNDNESS_VALUES)),
    'stability(%)':        formatar_ic_percentual(np.nanmean(SHAPLE_STABILITY_VALUES),       *calcular_ic95_percentual(SHAPLE_STABILITY_VALUES)),
    'sufficiency-1(%)':    formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_1_VALUES),   *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)':    formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_5_VALUES),   *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)':   formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_10_VALUES),  *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)':   formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_20_VALUES),  *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_20_VALUES)),
})

fim = time.time()
print(f"Tempo de execução TOTAL SHAP: {fim - inicio:.4f} segundos")

df_shap_metrics = pd.DataFrame(SHAPLE_TABLE_METRICS)
df_shap_metrics.to_csv(f'{BASE_PATH}/SHAP_METRICS.csv', index=False)
df_shap_metrics


In [ ]:
# ===  Shapley Interpretabilidade [Cálculos de Métricas] ===
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
import time
import multiprocessing
from joblib import Parallel, delayed, parallel_backend
import psutil, os, gc
from pathlib import Path  # <<< NOVO

np.random.seed(42)

# --- garantir que BASE_PATH seja Path (corrige erro do "/") ---
if 'BASE_PATH' in globals() and not isinstance(BASE_PATH, Path):
    BASE_PATH = Path(BASE_PATH)

# --- garantir espaço de features numéricas iguais ao treino ---
try:
    feat_cols = list(model.feature_names_in_)
except Exception:
    feat_cols = [c for c in X.columns if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(X[c])]

X_feat_sel   = X_sel[feat_cols].copy()
X_feat_train = X_train[feat_cols].copy() if set(feat_cols).issubset(X_train.columns) else X_train.filter(feat_cols).copy()
X_feat_test  = X[feat_cols].copy()

inicio = time.time()
SHAPLE_TABLE_METRICS         = []
SHAPLE_FIDELITY_VALUES       = []
SHAPLE_INFIDELITY_VALUES     = []
SHAPLE_FAITHFULLNESS_VALUES  = []
SHAPLE_CALIBRATION_VALUES    = []
SHAPLE_SELECTIVITY_VALUES    = []
SHAPLE_SOUNDNESS_VALUES      = []
SHAPLE_RELEVANCE_VALUES      = []
SHAPLE_SIMPLICITY_VALUES     = []
SHAPLE_ROBUSTENESS_VALUES    = []
SHAPLE_ROBUSTENESS_ADV_VALUES= []
SHAPLE_STABILITY_VALUES      = []
SHAPLE_MONOTONIA_VALUES      = []
SHAPLE_COMPLETENESS_VALUES   = []
SHAPLE_SUFFICIENCY_VALUES    = []
SHAPLE_CONSISTENCY_VALUES    = []
SHAPLE_CLASS_CONSISTENCY_VALUES = []
SHAPLE_DISCRIMINABILITY_VALUES =[]
SHAPLE_DISPARITY_VALUES = []
SHAPLE_SUFFICIENCY_1_VALUES  =  []
SHAPLE_SUFFICIENCY_5_VALUES  =  []
SHAPLE_SUFFICIENCY_10_VALUES = []
SHAPLE_SUFFICIENCY_20_VALUES = []
SHAPLE_DIR_SOUNDNESS_VALUES  = []

# --- helper seguro para converter valores para float ---
def _to_float_safe(x):
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace('%', '').replace(',', '.')
    try:
        return float(s)
    except Exception:
        return np.nan

# --- mapeia id (em ENTRADAS_SELECIONADAS) -> índice local (em X_sel/df_shap/shap_mat)
map_global_to_local = {int(id_): li for li, id_ in enumerate(ENTRADAS_SELECIONADAS)}

rows      = []
rows_inf  = []
rows_faith = []
rows_simplicity = []
rows_consistency = []
rows_robustness  = []
rows_selectivity = []
rows_soundness = []
rows_dirsound = []
rows_stability = []
rows_suff = []
rows_suff_debug = []

PERC_LIST = (0.01, 0.05, 0.10, 0.20)  # 1%, 5%, 10%, 20%

# índice da classe positiva (coerente com o modelo)
classes = list(getattr(model, "classes_", [0, 1]))
idx_pos = classes.index(1) if 1 in classes else int(np.argmax(classes))

print(f'Tamanho de entradas selecionadas : {len(ENTRADAS_SELECIONADAS)}')


def _process_one_shap_instance(i, idx_instancia):
    """
    Processa uma instância (posição i em ENTRADAS_SELECIONADAS, id global idx_instancia)
    e retorna tudo o que antes era acumulado nas listas globais.
    """
    inst_start = time.time()
    print(
        f"[SHAP][INST-START] local={i} id={idx_instancia} "
        f"at {time.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    def timed_call(step_name, func, *args, **kwargs):
        t0 = time.time()
        print(
            f"[SHAP][{step_name}][START] inst={idx_instancia} local={i} "
            f"at {time.strftime('%Y-%m-%d %H:%M:%S')}"
        )
        out = func(*args, **kwargs)
        t1 = time.time()
        print(
            f"[SHAP][{step_name}][END]   inst={idx_instancia} local={i} "
            f"at {time.strftime('%Y-%m-%d %H:%M:%S')} dur={t1 - t0:.2f}s"
        )
        return out

    pos_local = map_global_to_local.get(int(idx_instancia))
    if pos_local is None or pos_local >= len(df_shap):
        print(
            f"[SHAP][INST-SKIP] local={i} id={idx_instancia} "
            f"(pos_local inválido) at {time.strftime('%Y-%m-%d %H:%M:%S')}"
        )
        return None  # segurança

    # Probabilidades reais e aproximadas (df_shap)
    y_real = float(df_shap.loc[pos_local, 'prob_modelo'])
    y_shap = float(df_shap.loc[pos_local, 'soma_SHAP'])

    # ---------- Fidelity ----------
    x_row = X_feat_sel.iloc[[pos_local]]  # 1×D coerente
    fidelity_percent, fidelity_abs, det_fid = timed_call(
        "fidelity",
        calcula_fidelity_shapley,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        n_masks=100,
        mask_rate=0.5,
        baseline="background",
        target="prob",
        metric="r2",
        random_state=42,
        return_details=True
    )
    print("Fidelity R² (%):", fidelity_percent, " | R²:", fidelity_abs)

    _ = timed_call(
        "plot_fidelity_diagnostics",
        plot_fidelity_shapley_diagnostics,
        det_fid,
        titulo=f"Fidelity SHAP – instância id: {idx_instancia}",
        save_path=BASE_PATH / f"shap_fidelity_diagnostics_inst_{idx_instancia}.png"
    )

    # ---------- Infidelity ----------
    x_i   = X_feat_sel.iloc[i].values
    phi_i = shap_vals[i]  # SHAP da classe positiva para esta instância

    infidelity_percent, infidelity_abs, det_inf = timed_call(
        "infidelity",
        calcula_infidelity,
        model=model,
        x_row=x_i,
        feature_names=feat_cols,
        pos_class=idx_pos,
        target="prob",
        phi=phi_i,
        background=X_feat_sel,
        baseline="median",
        n_masks=2048,
        p_mask=0.5,
        random_state=42
    )

    row_inf = {
        "instancia_local": int(i),
        "pos_glob": int(idx_instancia),
        "infidelity_abs": round(infidelity_abs, 8),
        "infidelity_%":   round(infidelity_percent, 3)
    }

    # ---------- Faithfulness ----------
    x_series = X_feat_sel.iloc[i]  # 1×D como Series
    faithfulness_percent, faith_abs, corr_raw, det_faith = timed_call(
        "faithfulness",
        calcula_faithfulness_shap,
        x_row=x_series,
        model=model,
        X_ref=X_feat_test,
        pos_class=idx_pos,
        shap_explainer=shap_tree_explainer,
        background=X_feat_sel,
        target="logit",
        corr="kendall",
        ref_strategy="background",
        n_ref_samples=128,
        use_abs=False,
        return_details=True
    )
    row_faith = {
        "instancia_local": i,
        "faith_%": round(faithfulness_percent, 3),
        "faith_abs": round(faith_abs, 6),
        "corr_raw": round(corr_raw, 6)
    }

    # ---------- Simplicity ----------
    k_tau, simplicity_percent, detalhamento_simp = timed_call(
        "simplicity",
        calcula_simplicity_shap,
        phi_row=shap_vals[i],
        feature_names=feat_cols,
        tau=0.01,
        return_details=True
    )
    row_simplicity = {
        "instancia": int(i),
        "k_tau": int(k_tau),
        "simplicity_%": round(simplicity_percent, 3)
    }

    # ---------- Consistency entre modelos ----------
    consistency_percent = np.nan
    row_consistency = None
    try:
        x1 = X_feat_sel.iloc[[pos_local]]  # 1×D
        _classes = list(getattr(model, "classes_", [0, 1]))
        _pos_class = _classes.index(1) if 1 in _classes else int(np.argmax(_classes))
        consistency_percent, consistency_abs, det_cons = timed_call(
            "consistency",
            calcula_consistency_shap_models,
            x_row_df=x1,
            X_ref=X_feat_train,
            model_f=model,
            model_fp=model,
            pos_class=_pos_class,
            ref_strategy="mean",
            corr="spearman",
            background_f=X_feat_sel,
            background_fp=X_feat_sel,
            rank_by="raw",
            return_details=True
        )
        row_consistency = {"instancia_local": i, "consistency_%": round(consistency_percent, 3)}
    except NameError:
        consistency_percent = np.nan
    except Exception:
        consistency_percent = np.nan

    # ---------- Robustez ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    rob_max, rob_mean, rob_std, det_rob = timed_call(
        "robustness",
        calcula_robustez_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        n_trials=50,
        epsilon=0.05,
        epsilon_mode="relative_range",
        norm="l2",
        strategy="all",
        features_to_test=None,
        cat_flip_prob=0.10,
        random_state=42,
        topk_delta=8,
        score_mode="relative",
        higher_better=True,
    )
    robustez_percent = det_rob.get("robust_score", np.nan)
    row_robustness = {
        "instancia_local": i,
        "robust_max": float(rob_max),
        "robust_mean": float(rob_mean),
        "robust_std": float(rob_std),
        "robust_score": float(robustez_percent) if not np.isnan(robustez_percent) else np.nan
    }

    # ---------- Completeness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    completeness_percentual, completeness_err, completeness_det = timed_call(
        "completeness",
        calcula_completeness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        target="prob",
        return_details=True
    )

    # ---------- Selectivity ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    selectivity_percent, selectivity_abs, selectivity_det = timed_call(
        "selectivity",
        calcula_selectivity_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        K=10,
        baseline="background",
        target="prob",
        clip_negatives=True,
        return_details=True
    )
    row_selectivity = {
        "instancia_local": i,
        "selectivity_%": float(selectivity_percent),
        "selectivity_abs": float(selectivity_abs)
    }

    # ---------- Soundness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    soundness_percent, soundness_abs, soundness_det = timed_call(
        "soundness",
        calcula_soundness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        U_mode="auto_local",
        tau=0.01,
        masking="mean_mode",
        target="prob",
        return_details=True
    )
    row_soundness = {
        "instancia_local": i,
        "soundness_%": float(soundness_percent),
        "soundness_abs": float(soundness_abs)
    }

    # ---------- Directional Soundness ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    dir_soundness_percent, dir_soundness_frac, dir_soundness_det = timed_call(
        "directional_soundness",
        calcula_directional_soundness_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        pos_class=idx_pos,
        R_mode="auto_local",
        tau=0.01,
        tau_mode="rel_fx",
        masking="mean_mode",
        sign_eps_rel=0.01,
        count_zero_matches=True,
        return_details=True
    )
    row_dirsound = {
        "instancia_local": i,
        "directional_soundness_%": float(dir_soundness_percent),
        "directional_soundness_frac": float(dir_soundness_frac)
    }

    # ---------- Stability ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    stability_mean, stability_std, stability_percent, det_stability = timed_call(
        "stability",
        calcula_stability_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        drop_cols=('classe','y'),
        pos_class=idx_pos,
        n_trials=50,
        norm="l2",
        sigma_mode="adaptive",
        alpha=0.02,
        beta=0.10,
        num_bins=4,
        avoid_bucket_crossing=True,
        p_flip=0.05,
        random_state=42
    )
    row_stability = {
        "instancia_local": i,
        "stability_mean": float(stability_mean),
        "stability_std": float(stability_std),
        "stability_score": float(stability_percent)
    }

    # ---------- Sufficiency ----------
    x_row = X_feat_sel.iloc[[pos_local]]
    scores_suff, det_suff = timed_call(
        "sufficiency",
        calcula_sufficiency_shap,
        x_row_df=x_row,
        model=model,
        X_ref=X_feat_train,
        background=X_feat_sel,
        drop_cols=('classe','y'),
        pos_class=idx_pos,
        percent_list=(0.01, 0.05, 0.10, 0.20),
        ranking="abs",
        masking="median_mode",
        baseline_values=None,
        f_is_proba=True,
        random_state=42
    )
    row_suff = make_sufficiency_row_sufficiency(idx_instancia, scores_suff)
    row_suff_debug = make_sufficiency_debug_row_sufficiency(idx_instancia, det_suff)

    sufficiency_percent_s1, sufficiency_percent_s5, sufficiency_percent_s10, sufficiency_percent_s20 = (
        scores_suff.get(k, np.nan) for k in ('1%', '5%', '10%', '20%')
    )

    # ---------- Tabela por instância ----------
    table_row = {
        'instancia': idx_instancia,
        'fidelity(%)': f"{fidelity_percent:.3f}",
        'infidelity(%)': f"{infidelity_percent:.3f}",
        'faithfulness(%)': f"{faithfulness_percent:.3f}",
        'simplicity(%)': f"{simplicity_percent:.3f}",
        'consistency(%)': f"{consistency_percent:.3f}" if not np.isnan(consistency_percent) else "NA",
        'robustez(%)': f"{robustez_percent:.3f}" if not np.isnan(robustez_percent) else "NA",
        'completeness(%)': f"{completeness_percentual:.3f}",
        'selectivity(%)': f"{selectivity_percent:.3f}",
        'soundness(%)': f"{soundness_percent:.3f}",
        'directional soundness(%)': f"{dir_soundness_percent:.3f}",
        'stability(%)': f"{stability_percent:.3f}",
        'sufficiency-1(%)': f"{sufficiency_percent_s1:.3f}",
        'sufficiency-5(%)': f"{sufficiency_percent_s5:.3f}",
        'sufficiency-10(%)': f"{sufficiency_percent_s10:.3f}",
        'sufficiency-20(%)': f"{sufficiency_percent_s20:.3f}",
    }

    inst_end = time.time()
    print('#===============================================================================================================#')
    print(
        f'Instância {i+1}/{len(ENTRADAS_SELECIONADAS)} (ID={idx_instancia}) processada. '
        f'Tempo total instância: {inst_end - inst_start:.2f}s '
        f'({time.strftime("%Y-%m-%d %H:%M:%S")})'
    )
    print('#===============================================================================================================#')

    return {
        "idx_instancia": idx_instancia,
        "fidelity_percent": float(fidelity_percent),
        "infidelity_percent": float(infidelity_percent),
        "faithfulness_percent": float(faithfulness_percent),
        "simplicity_percent": float(simplicity_percent),
        "consistency_percent": float(consistency_percent) if not np.isnan(consistency_percent) else np.nan,
        "robustez_percent": float(robustez_percent) if not np.isnan(robustez_percent) else np.nan,
        "completeness_percentual": float(completeness_percentual),
        "selectivity_percent": float(selectivity_percent),
        "soundness_percent": float(soundness_percent),
        "dir_soundness_percent": float(dir_soundness_percent),
        "stability_percent": float(stability_percent),
        "suff1": float(sufficiency_percent_s1),
        "suff5": float(sufficiency_percent_s5),
        "suff10": float(sufficiency_percent_s10),
        "suff20": float(sufficiency_percent_s20),
        "table_row": table_row,
        "row_inf": row_inf,
        "row_faith": row_faith,
        "row_simplicity": row_simplicity,
        "row_consistency": row_consistency,
        "row_robustness": row_robustness,
        "row_selectivity": row_selectivity,
        "row_soundness": row_soundness,
        "row_dirsound": row_dirsound,
        "row_stability": row_stability,
        "row_suff": row_suff,
        "row_suff_debug": row_suff_debug,
    }


# ===========================
# Execução em paralelo (com BATCH de instâncias)
# ===========================
RAM_GB   = psutil.virtual_memory().total / 1024**3
MAX_CPU  = multiprocessing.cpu_count()

NOTEBOOKS_RODANDO = 1   # <<< ajuste se tiver vários notebooks pesados abertos

# Limite máximo de processos em paralelo (para não "explodir" o i9)
#MAX_PARALLEL_JOBS = 4   # por ex.: 4, 6 ou 8, conforme seu gosto

#N_JOBS_AUTO = max(1, int((RAM_GB // 4) // NOTEBOOKS_RODANDO))
#N_JOBS = max(1, min(MAX_PARALLEL_JOBS, N_JOBS_AUTO))
# Batch de instâncias: controla quantas instâncias são processadas por "onda" de Parallel

N_JOBS = 4
BATCH_SIZE_INST = 4   # por ex.: 4, 8, 10; você pode testar

all_results = []
n_inst = len(ENTRADAS_SELECIONADAS)
indices = list(range(n_inst))

batches = [
    indices[start:min(start + BATCH_SIZE_INST, n_inst)]
    for start in range(0, n_inst, BATCH_SIZE_INST)
]

print(f"[SHAP] N_JOBS={N_JOBS} | BATCH_SIZE_INST={BATCH_SIZE_INST} | n_batches={len(batches)}")

with parallel_backend("loky", inner_max_num_threads=1):
    for b_idx, batch_idx in enumerate(batches):
        print(f"[SHAP][BATCH] {b_idx+1}/{len(batches)} -> instâncias locais {batch_idx[0]}..{batch_idx[-1]}")
        batch_results = Parallel(
            n_jobs=N_JOBS,
            batch_size=1    # 1 instância por tarefa, deixa o controle para o nosso BATCH_SIZE_INST
        )(
            delayed(_process_one_shap_instance)(
                i, int(ENTRADAS_SELECIONADAS[i])
            )
            for i in batch_idx
        )
        all_results.extend(batch_results)
        gc.collect()

results = all_results

# ===========================
# Agregação dos resultados
# ===========================
for res in results:
    if res is None:
        continue

    SHAPLE_TABLE_METRICS.append(res["table_row"])

    SHAPLE_FIDELITY_VALUES.append(_to_float_safe(res["fidelity_percent"]))
    SHAPLE_INFIDELITY_VALUES.append(_to_float_safe(res["infidelity_percent"]))
    SHAPLE_FAITHFULLNESS_VALUES.append(_to_float_safe(res["faithfulness_percent"]))
    SHAPLE_SIMPLICITY_VALUES.append(_to_float_safe(res["simplicity_percent"]))
    SHAPLE_CONSISTENCY_VALUES.append(_to_float_safe(res["consistency_percent"]))
    SHAPLE_ROBUSTENESS_VALUES.append(_to_float_safe(res["robustez_percent"]))
    SHAPLE_COMPLETENESS_VALUES.append(_to_float_safe(res["completeness_percentual"]))
    SHAPLE_SELECTIVITY_VALUES.append(_to_float_safe(res["selectivity_percent"]))
    SHAPLE_SOUNDNESS_VALUES.append(_to_float_safe(res["soundness_percent"]))
    SHAPLE_DIR_SOUNDNESS_VALUES.append(_to_float_safe(res["dir_soundness_percent"]))
    SHAPLE_STABILITY_VALUES.append(_to_float_safe(res["stability_percent"]))
    SHAPLE_SUFFICIENCY_1_VALUES.append(_to_float_safe(res["suff1"]))
    SHAPLE_SUFFICIENCY_5_VALUES.append(_to_float_safe(res["suff5"]))
    SHAPLE_SUFFICIENCY_10_VALUES.append(_to_float_safe(res["suff10"]))
    SHAPLE_SUFFICIENCY_20_VALUES.append(_to_float_safe(res["suff20"]))

    rows_inf.append(res["row_inf"])
    rows_faith.append(res["row_faith"])
    rows_simplicity.append(res["row_simplicity"])
    if res["row_consistency"] is not None:
        rows_consistency.append(res["row_consistency"])
    rows_robustness.append(res["row_robustness"])
    rows_selectivity.append(res["row_selectivity"])
    rows_soundness.append(res["row_soundness"])
    rows_dirsound.append(res["row_dirsound"])
    rows_stability.append(res["row_stability"])
    rows_suff.append(res["row_suff"])
    rows_suff_debug.append(res["row_suff_debug"])


# ===========================
# Linha com médias e ICs
# ===========================
SHAPLE_TABLE_METRICS.append({
    'instancia': 'MÉDIA',
    'fidelity(%)':         formatar_ic_percentual(np.nanmean(SHAPLE_FIDELITY_VALUES),        *calcular_ic95_percentual(SHAPLE_FIDELITY_VALUES)),
    'infidelity(%)':       formatar_ic_percentual(np.nanmean(SHAPLE_INFIDELITY_VALUES),      *calcular_ic95_percentual(SHAPLE_INFIDELITY_VALUES)),
    'faithfulness(%)':     formatar_ic_percentual(np.nanmean(SHAPLE_FAITHFULLNESS_VALUES),   *calcular_ic95_percentual(SHAPLE_FAITHFULLNESS_VALUES)),
    'simplicity(%)':       formatar_ic_percentual(np.nanmean(SHAPLE_SIMPLICITY_VALUES),      *calcular_ic95_percentual(SHAPLE_SIMPLICITY_VALUES)),
    'consistency(%)':      formatar_ic_percentual(
        np.nanmean(SHAPLE_CONSISTENCY_VALUES),
        *calcular_ic95_percentual([v for v in SHAPLE_CONSISTENCY_VALUES if not np.isnan(v)])
    ) if any(not np.isnan(v) for v in SHAPLE_CONSISTENCY_VALUES) else "NA",
    'robustez(%)':         formatar_ic_percentual(
        np.nanmean(SHAPLE_ROBUSTENESS_VALUES),
        *calcular_ic95_percentual([v for v in SHAPLE_ROBUSTENESS_VALUES if not np.isnan(v)])
    ) if any(not np.isnan(v) for v in SHAPLE_ROBUSTENESS_VALUES) else "NA",
    'completeness(%)':     formatar_ic_percentual(np.nanmean(SHAPLE_COMPLETENESS_VALUES),    *calcular_ic95_percentual(SHAPLE_COMPLETENESS_VALUES)),
    'selectivity(%)':      formatar_ic_percentual(np.nanmean(SHAPLE_SELECTIVITY_VALUES),     *calcular_ic95_percentual(SHAPLE_SELECTIVITY_VALUES)),
    'soundness(%)':        formatar_ic_percentual(np.nanmean(SHAPLE_SOUNDNESS_VALUES),       *calcular_ic95_percentual(SHAPLE_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.nanmean(SHAPLE_DIR_SOUNDNESS_VALUES), *calcular_ic95_percentual(SHAPLE_DIR_SOUNDNESS_VALUES)),
    'stability(%)':        formatar_ic_percentual(np.nanmean(SHAPLE_STABILITY_VALUES),       *calcular_ic95_percentual(SHAPLE_STABILITY_VALUES)),
    'sufficiency-1(%)':    formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_1_VALUES),   *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)':    formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_5_VALUES),   *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)':   formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_10_VALUES),  *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)':   formatar_ic_percentual(np.nanmean(SHAPLE_SUFFICIENCY_20_VALUES),  *calcular_ic95_percentual(SHAPLE_SUFFICIENCY_20_VALUES)),
})

fim = time.time()
print(f"Tempo de execução TOTAL SHAP: {fim - inicio:.4f} segundos")

df_shap_metrics = pd.DataFrame(SHAPLE_TABLE_METRICS)
df_shap_metrics.to_csv(f'{BASE_PATH}/SHAP_METRICS.csv', index=False)
df_shap_metrics


In [ ]:
ENTRADAS_SELECIONADAS_bkp = ENTRADAS_SELECIONADAS.copy()

In [ ]:
ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS_bkp.copy()
ENTRADAS_SELECIONADAS

In [ ]:
# ===================== contribuições SHAP + Tabela de métricas (instância / média / mediana) =====================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# --------------------------------- utils básicos ---------------------------------
def _coerce_binary_series(s):
    s = pd.Series(s).copy()
    map_true  = {"1", 1, True, "true", "True", "sim", "Sim", "yes", "Yes", "y", "Y", "positivo", "Positivo"}
    map_false = {"0", 0, False, "false", "False", "nao", "não", "Nao", "Não", "no", "No", "n", "N", "negativo", "Negativo"}
    def _m(v):
        sv = str(v).strip()
        if sv in map_true:  return 1
        if sv in map_false: return 0
        return v
    s = s.map(_m)
    s = pd.to_numeric(s, errors="coerce")
    return s

THRESHOLD   = 0.5   # caso derive y a partir de probabilidade
TOP_K_SHAP  = 20    # número de features a exibir no plot/tabelas

def _infer_y_from_X(Xdf, threshold=0.5):
    """
    Extrai y de colunas em X:
      (1) alvo explícito (y/target/label/classe/...),
      (2) probabilidade única (y_proba/proba_1/score_pos/...),
      (3) fallback: coluna ~[0,1] vira proba->y (>= threshold).
    Retorna (y_series, cols_to_drop).
    """
    TARGET_CANDIDATES = ['y','classe','target','label','y_true','diagnosis','outcome']
    PROBA_CANDIDATES  = ['y_proba','proba','proba_1','p1','prob_1','prob_pos','score','score_pos','pred_proba_1']

    # 1) alvo explícito
    for col in TARGET_CANDIDATES:
        if col in Xdf.columns:
            y_raw = Xdf[col]
            y_bin = _coerce_binary_series(y_raw).rename('y')
            return y_bin, [col]

    # 2) probabilidade em única coluna
    for col in PROBA_CANDIDATES:
        if col in Xdf.columns:
            y_bin = (pd.to_numeric(Xdf[col], errors='coerce') >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    # 3) fallback: heurística p/ coluna [0,1]
    for col in Xdf.columns:
        s = pd.to_numeric(Xdf[col], errors='coerce')
        if s.notna().mean() > 0.9 and (s.between(0, 1, inclusive='both').mean() > 0.9):
            y_bin = (s >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    raise ValueError("Não encontrei y nem y_proba dentro de X para alinhar o alvo.")

# --------------------------------- preparo do X (remove alvos e probas) ---------------------------------
try:
    EXCLUDE_COLS
except NameError:
    EXCLUDE_COLS = []

y_inferido, y_cols_to_drop = _infer_y_from_X(X)

TARGET_LIKE = ['y','classe','target','label','y_true','septic','outcome']
PROBA_LIKE  = ['y_proba','proba','proba_0','proba_1','prob_0','prob_1','p0','p1',
               'score','score_pos','score_neg','pred_proba_0','pred_proba_1']

cols_drop   = [c for c in (EXCLUDE_COLS + TARGET_LIKE + PROBA_LIKE + y_cols_to_drop) if c in X.columns]
X_feat_full = X.drop(columns=list(set(cols_drop)), errors='ignore').copy()

# --- alinhamento com o modelo: mesmas colunas e ordem que o modelo usou no treino ---
if hasattr(model, "feature_names_in_"):
    feat_cols_eff = [c for c in model.feature_names_in_.tolist() if c in X_feat_full.columns]
    missing = [c for c in model.feature_names_in_.tolist() if c not in X_feat_full.columns]
    if len(missing) > 0:
        raise ValueError(f"As seguintes colunas esperadas pelo modelo não estão em X: {missing}")
    X_feat = X_feat_full.loc[:, feat_cols_eff].copy()
else:
    if hasattr(model, "n_features_in_"):
        n_expected = int(model.n_features_in_)
        if X_feat_full.shape[1] > n_expected:
            X_feat = X_feat_full.iloc[:, :n_expected].copy()
        elif X_feat_full.shape[1] == n_expected:
            X_feat = X_feat_full.copy()
        else:
            raise ValueError(
                f"X tem {X_feat_full.shape[1]} features, mas o modelo espera {n_expected}. "
                f"Adicione as colunas faltantes ou informe feature_names_in_."
            )
    else:
        X_feat = X_feat_full.copy()

# y alinhado
y_alinhado = pd.Series(y_inferido).reindex(X_feat.index).astype('Int64').rename('y')

# --------------------------------- split 8/1/1 consistente ---------------------------------
# Se já vieram do pipeline 8/1/1, apenas reutilize e garanta as colunas corretas:
if all(name in globals() for name in ["X_train", "X_test", "y_train", "y_test"]):
    X_train = X_train.loc[:, X_feat.columns]
    X_test  = X_test.loc[:,  X_feat.columns]
    X_train_real, X_test_real = X_train, X_test
else:
    idx_trval, idx_tst = train_test_split(
        X_feat.index, test_size=0.10, stratify=y_alinhado, random_state=42
    )
    y_trval = y_alinhado.loc[idx_trval]
    idx_tr, idx_val = train_test_split(
        idx_trval, test_size=(1/9), stratify=y_trval, random_state=42
    )

    X_train = X_feat.loc[idx_tr].copy()          # ~80%
    y_train = y_alinhado.loc[idx_tr].copy()
    X_val   = X_feat.loc[idx_val].copy()         # ~10% (não usado aqui, mas preservado)
    y_val   = y_alinhado.loc[idx_val].copy()
    X_test  = X_feat.loc[idx_tst].copy()         # ~10%
    y_test  = y_alinhado.loc[idx_tst].copy()

    X_train_real, X_test_real = X_train, X_test

# --------------------------------- nomes de classes ---------------------------------
unique_classes = sorted(pd.unique(y_alinhado.dropna()))
if len(unique_classes) == 2 and set(unique_classes) <= {0,1}:
    class_names = ['Risco Baixo','Risco Alto']
else:
    class_names = [f'Classe {c}' for c in unique_classes]

# --------------------------------- SHAP explainer (treinado com 100% treino) ---------------------------------
if 'shap_explainer' not in globals():
    shap_explainer = shap.TreeExplainer(
        model,
        data=X_train,
        model_output="probability"
    )

FEATURES_SHAP = list(X_train.columns)

# --------------------------------- métricas: helpers ---------------------------------
METRIC_COLS = [
    'fidelity(%)','infidelity(%)','faithfulness(%)','simplicity(%)','consistency(%)',
    'robustez(%)','stability(%)','selectivity(%)','soundness(%)','directional soundness(%)',
    'sufficiency-1(%)','sufficiency-5(%)','sufficiency-10(%)','sufficiency-20(%)','completeness(%)'
]

_num_regex = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def _to_float_or_nan(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.number)): return float(x)
    s = str(x)
    m = _num_regex.search(s)
    return float(m.group(0)) if m else np.nan

def get_metrics_instance_mean_median(df_metrics, inst_idx):
    row_inst = df_metrics.loc[df_metrics['instancia'] == inst_idx]
    row_mean = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MÉDIA')]
    row_medi = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MEDIANA')]

    if row_mean.empty or row_medi.empty:
        tmp = df_metrics.copy()
        for c in METRIC_COLS:
            if c in tmp.columns:
                tmp[c] = tmp[c].map(_to_float_or_nan)
        only_num = tmp.loc[pd.to_numeric(tmp['instancia'], errors='coerce').notna(), METRIC_COLS]
        mean_vals = only_num.mean(numeric_only=True)
        medi_vals = only_num.median(numeric_only=True)
    else:
        mean_vals = row_mean.iloc[0][METRIC_COLS].map(_to_float_or_nan)
        medi_vals = row_medi.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    inst_vals = pd.Series({c: np.nan for c in METRIC_COLS})
    if not row_inst.empty:
        inst_vals = row_inst.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    out = pd.DataFrame({
        "metric": METRIC_COLS,
        "inst":  [inst_vals.get(c, np.nan) for c in METRIC_COLS],
        "mean":  [mean_vals.get(c, np.nan) for c in METRIC_COLS],
        "median":[medi_vals.get(c, np.nan) for c in METRIC_COLS],
    })
    return out

# ------------ helper para garantir vetor 1D de SHAP para 1 instância ------------
def _extract_shap_1d(shap_vals, pos_idx: int):
    """
    Recebe a saída de shap_explainer.shap_values(X_inst) e devolve um vetor 1D
    com as contribuições para a classe pos_idx (ou única saída).
    """
    if isinstance(shap_vals, list):
        arr = np.array(shap_vals[pos_idx])
    else:
        arr = np.array(shap_vals)

    if arr.ndim == 1:
        return arr.astype(float)

    if arr.ndim == 2:          # (n_samples, n_features)
        return arr[0].astype(float)

    if arr.ndim == 3:
        if arr.shape[0] == 1:          # (1, n_feat, n_out)
            return arr[0, :, 0].astype(float)
        if arr.shape[1] == 1:          # (n_classes, 1, n_feat)
            return arr[pos_idx, 0, :].astype(float)
        return arr[0, :, 0].astype(float)

    raise ValueError(f"Formato inesperado de shap_vals: ndim={arr.ndim}, shape={arr.shape}")

# --------------------------------- PLOT PRINCIPAL: SHAP + tabelas ---------------------------------
def plot_shap_with_real_values_and_metrics(inst_id, df_metrics):
    """
    Versão SHAP do seu plot_lime_with_real_values_and_metrics, mantendo o mesmo layout.
    Coluna 1: waterfall SHAP
    Coluna 2: tabela de valores reais
    Coluna 3: tabela de métricas
    """

    # --- localizar o rótulo de índice real via X['id'] ---
    if 'id' not in X.columns:
        raise KeyError("Coluna 'id' não encontrada em X.")
    idx_candidates = X.index[X['id'] == inst_id]
    if len(idx_candidates) == 0:
        print(f"Não achei instância com id={inst_id} em X['id'].")
        return
    idx_label = idx_candidates[0]

    # --- usar SEMPRE os features efetivos (mesmo espaço do modelo/SHAP) ---
    if idx_label not in X_feat.index:
        raise ValueError(f"O índice {idx_label} (id={inst_id}) não está presente em X_feat.")

    # pega a linha completa no espaço alinhado ao modelo
    x_row_full = X_feat.loc[idx_label]
    x_row      = x_row_full[FEATURES_SHAP]
    row_real   = x_row

    # usa as MESMAS colunas/ordem na predição do modelo
    probs = model.predict_proba(
        pd.DataFrame([x_row], columns=FEATURES_SHAP)
    )[0]

    classes  = list(model.classes_)
    pos_idx  = classes.index(1) if 1 in classes else int(np.argmax(probs))
    neg_idx  = classes.index(0) if 0 in classes else (1 - pos_idx if len(classes) >= 2 else 0)
    p_pos, p_neg = float(probs[pos_idx]), float(probs[neg_idx])
 
    # -------- SHAP local --------
    shap_vals = shap_explainer.shap_values(
        pd.DataFrame([x_row], columns=FEATURES_SHAP)
    )
    contrib_full = _extract_shap_1d(shap_vals, pos_idx=pos_idx)

    # base value para a classe positiva
    base_val = shap_explainer.expected_value
    if isinstance(base_val, (list, np.ndarray)) and np.ndim(base_val) > 0:
        base_pos = float(base_val[pos_idx])
    else:
        base_pos = float(base_val)

    # Explanation para o waterfall
    exp = shap.Explanation(
        values=contrib_full,
        base_values=base_pos,
        data=row_real.values,
        feature_names=FEATURES_SHAP
    )

    # Para tabelas: ordenar por |SHAP| e limitar em TOP_K_SHAP
    pairs_idx = sorted(
        list(enumerate(contrib_full)),
        key=lambda t: -abs(t[1])
    )
    m = min(len(pairs_idx), TOP_K_SHAP)
    pairs_idx = pairs_idx[:m]

    feat_names     = FEATURES_SHAP
    feat_idx_order = [i for i, _ in pairs_idx]
    weights        = np.array([w for _, w in pairs_idx], dtype=float)
    real_values    = [row_real[feat_names[i]] for i in feat_idx_order]




    # ---- detectar o ponto onde a SOMA ACUMULADA muda de sinal ----
    # A waterfall do SHAP é exibida de baixo para cima.
    # Portanto, acumulamos de baixo → cima, usando a mesma ordem usada pelo SHAP.
    #ordered_weights = weights[::-1]  # inverte para simular a ordem visual do waterfall
    #cumsum_vals = base_pos + np.cumsum(ordered_weights)

    #sign_change_idx = None
    #for k in range(1, len(cumsum_vals)):
    #    if np.sign(cumsum_vals[k-1]) != np.sign(cumsum_vals[k]):
    #        # k é o índice na ordem bottom→top
    #        # Converter para índice da ordem top→bottom (que é o eixo Y do plot)
    #        sign_change_idx = m - k
    #        break



    # --- métricas por id externo ---
    mt = get_metrics_instance_mean_median(df_metrics, inst_id)

    # ================= FIGURA PRINCIPAL (layout idêntico ao LIME) =================
    fig = plt.figure(figsize=(12.5, 3.0), dpi=120)

    # 3 colunas: waterfall SHAP | valores reais | métricas
    gs = fig.add_gridspec(1, 3, width_ratios=[3.0, 2.0, 2.2], wspace=0.24)

    # ---- título geral à esquerda ("Explicações – Instância id: X") ----
    fig.suptitle(
        f"Explicações – Instância id: {inst_id}",
        fontsize=22,
        fontweight="bold",
        x=0.02      ,
        ha="left",
        y=0.96
    )

    # ---- blocos de risco no topo ----
    txt_p_neg = f"{p_neg:.2f}".replace(".", ",")
    txt_p_pos = f"{p_pos:.2f}".replace(".", ",")

    fig.text(0.60, 0.94, "Risco Baixo",
             fontsize=15, fontweight="bold",
             color="#1f77ff", ha="center", va="bottom")
    fig.text(0.60, 0.89, txt_p_neg,
             fontsize=20, fontweight="bold",
             color="#1f77ff", ha="center", va="top")

    fig.text(0.82, 0.94, "Risco Alto",
             fontsize=15, fontweight="bold",
             color="#ff7f0e", ha="center", va="bottom")
    fig.text(0.82, 0.89, txt_p_pos,
             fontsize=20, fontweight="bold",
             color="#ff7f0e", ha="center", va="top")

    # ------------------------------------------------------------------
    # 1) WATERFALL SHAP (coluna esquerda) – fontes próximas ao LIME
    # ------------------------------------------------------------------
    ax = fig.add_subplot(gs[0, 0])

    with plt.rc_context({'font.size': 9}):
        plt.sca(ax)
        shap.plots.waterfall(exp, max_display=TOP_K_SHAP, show=False)




    # ============================
    #    LINHA DE MARCAÇÃO
    # ============================
    #if sign_change_idx is not None:
    #    y_mark = sign_change_idx + 0.5  # linha entre a feature k e k+1

        #ax.axhline(
         #   y=y_mark,
          #  color="black",
           # linewidth=1.2,
            #linestyle="--",
            #alpha=0.75
        #)

        #ax.text(
        #    0.02, y_mark + 0.2,
        #    "Mudança de direção do f(x)",
        #    transform=ax.get_yaxis_transform(),
        #    fontsize=8,
        #    color="black",
        #    ha="left"
        #)







    # deixar os textos do waterfall com fonte 9 (reforço)
    for txt in ax.texts:
        txt.set_fontsize(9)
    ax.tick_params(labelsize=9)

    # ------------------------------------------------------------------
    # 2) Tabela de valores reais (coluna central)
    # ------------------------------------------------------------------
    axr = fig.add_subplot(gs[0, 1])
    axr.axis("off")

    axr.text(0.00, 0.98, "Feature", fontsize=11, fontweight="bold",
             ha="left", va="top")
    axr.text(1.33, 0.98, "Valor (real)", fontsize=11, fontweight="bold",
             ha="right", va="top")

    def _fmt_val(v):
        try:
            return f"{float(v):.2f}".replace(".", ",")
        except Exception:
            return str(v)

    y0, dy = 0.92, 0.055
    for k, idx_f in enumerate(feat_idx_order):
        if y0 - k*dy < 0.05:
            break
        fname = feat_names[idx_f]
        fval  = real_values[k]

        axr.text(0.00, y0 - k*dy, fname,           ha="left",  va="top", fontsize=9)
        axr.text(1.33, y0 - k*dy, _fmt_val(fval),  ha="right", va="top", fontsize=9)

    # ------------------------------------------------------------------
    # 3) Tabela de métricas (coluna direita, igual ao LIME)
    # ------------------------------------------------------------------
    axm = fig.add_subplot(gs[0, 2])
    axm.axis("off")

    from matplotlib.lines import Line2D

    # linha vertical separando "Valor (real)" das métricas
    bbox_val = axr.get_position()
    bbox_met = axm.get_position()
    x_sep = (bbox_val.x1 + bbox_met.x0) / 2.0

    linha_sep = Line2D(
        [x_sep+0.039, x_sep+0.039],       # x0, x1
        [0.10, 0.90],         # y0, y1
        transform=fig.transFigure,
        color="#cccccc",
        linewidth=1.0,
        alpha=0.8
    )
    fig.add_artist(linha_sep)

    # x mais espaçados para evitar qualquer sobreposição
    x_feat   = 0.15
    x_inst   = 1.00
    x_media  = 1.38
    x_mediana= 1.78

    axm.text(x_feat,    0.98, "Métrica",   fontsize=11, fontweight="bold", ha="left",  va="top")
    axm.text(x_inst,    0.98, "Valor",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_media,   0.98, "Média",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_mediana, 0.98, "Mediana",   fontsize=11, fontweight="bold", ha="right", va="top")

    def _fmt_metric(v):
        return "0,00" if pd.isna(v) else f"{float(v):.3f}".replace(".", ",")

    y0, dy = 0.92, 0.055
    for i, row in mt.iterrows():
        if y0 - i*dy < 0.05:
            break

        axm.text(x_feat,  y0 - i*dy, row['metric'],             ha="left",  va="top", fontsize=9)
        axm.text(x_inst,  y0 - i*dy, _fmt_metric(row['inst']),   ha="right", va="top", fontsize=9)
        axm.text(x_media, y0 - i*dy, _fmt_metric(row['mean']),   ha="right", va="top", fontsize=9)
        axm.text(x_mediana, y0 - i*dy, _fmt_metric(row['median']), ha="right", va="top", fontsize=9)

    plt.subplots_adjust(top=0.82)
    fig.set_size_inches(12.5, 6.0, forward=True)
    plt.show()

# ----------------- rodar para as suas instâncias -----------------
try:
    ENTRADAS_SELECIONADAS
except NameError:
    ENTRADAS_SELECIONADAS = [0]

ENTRADAS_SELECIONADAS = np.asarray(list(ENTRADAS_SELECIONADAS), dtype=int)

for idx_pos in ENTRADAS_SELECIONADAS:
    plot_shap_with_real_values_and_metrics(int(idx_pos), df_shap_metrics)


In [ ]:
# ===  Shap Interpretabilidade [Heatmap das Métricas]===

# Filtra as instâncias, removendo a linha de média
df_shap_metrics_filter = df_shap_metrics[df_shap_metrics["instancia"] != "MÉDIA"].copy()
df_shap_metrics_filter = df_shap_metrics_filter[df_shap_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_shap_metrics_filter.columns
                  if df_shap_metrics[col].nunique() == 1 and (df_shap_metrics[col].iloc[0] == '100.0%' or df_shap_metrics[col].iloc[0] == '0.0%')]

# df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# Calcular correlação de Pearson
corr_matrix = df_shap_metrics_filter.corr(method='pearson')

# Máscara para ocultar correlações abaixo de 0.3
# mask_low_corr = np.abs(corr_matrix) < 0.3

# manter só correlações > 0.3 ou < -0.3
keep = (corr_matrix > 0.3) | (corr_matrix < -0.3)

# máscara para o seaborn (True = esconder)
mask = ~keep
np.fill_diagonal(mask.values, True)  # (opcional) esconde a diagonal

# Plotar heatmap
# ordered = corr_matrix.loc[order, order]
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Heatmap de Correlação (Pearson) entre Métricas - Apenas valores com |r| > 0.3")
plt.tight_layout()
plt.savefig("heatmap_metricas_filtradas.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

# --- escolha: usar correlação absoluta (ignora o sinal) ou assinada
use_abs = True   # True => distância = 1 - |corr| ; False => 1 - corr

# MIN: converter para float, substituir NaN/Inf e garantir simetria/limites
C = corr_matrix.to_numpy(dtype=float)
C = np.nan_to_num(C, nan=0.0, posinf=1.0, neginf=-1.0)     # MIN
C = np.clip(C, -1.0, 1.0)                                   # MIN
C = 0.5 * (C + C.T)                                         # MIN (simetriza)
np.fill_diagonal(C, 1.0)                                     # MIN (autocorr = 1)

C = np.abs(C) if use_abs else C

# distância (0=correlação perfeita; 1=sem correlação se use_abs=True)
D = 1.0 - C
np.fill_diagonal(D, 0.0)
D = np.maximum(D, 0.0)                                      # MIN: evita negativos por ruído numérico

# linkage precisa do formato "condensed"
condensed_D = squareform(D, checks=False)

# método de ligação; "average" é estável para distâncias de correlação
Z = hierarchy.linkage(condensed_D, method='average')

# dendrograma
plt.figure(figsize=(12, 6))
dn = hierarchy.dendrogram(Z, labels=corr_matrix.columns.tolist(), leaf_rotation=90)
plt.title(f"Dendrograma das métricas (distância = 1 - {'|corr|' if use_abs else 'corr'})")
plt.ylabel("Distância")
plt.tight_layout()
plt.savefig("dendrogram_metricas.png", dpi=150)
plt.show()

# (opcional) ordem das métricas segundo o dendrograma
order = [corr_matrix.columns[i] for i in dn["leaves"]]
print(order)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial.distance import squareform

def venn_from_corr_mds_labels(
    corr_matrix,
    radius=1.0,               # raio (mesmo para todos)
    epsilon_frac=0.06,        # ε aplicado SÓ quando r>0: d <- d + ε*R
    sign_mode="both",         # "both" | "positive" | "negative"
    use_abs=False,            # se True, ignora sinais (sign_mode fica sem efeito)
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,         # fonte 8
    label_weight='normal',    # sem negrito
    label_jitter=0.03,        # jitter vertical inicial (fração do raio)
    repel_iters=150,          # iterações de repulsão dos rótulos
    repel_step=0.02,          # passo da repulsão (fração do raio)
    inside_margin=0.12,       # margem p/ manter texto dentro do círculo
    title="Venn adaptado (d = (1 − r)·R, ε>0 só para r>0, filtro de sinal)"
):
    """
    Distância alvo entre centros: d(r) = (1 − r)*R  (+ ε*R se r>0).
    - sign_mode:
        * "both": usa r como está;
        * "positive": se r<=0, força r_eff=-1 (distância máxima 2R);
        * "negative": se r>=0, força r_eff=-1 (distância máxima 2R).
    - use_abs=True ignora sinais e, portanto, sign_mode não tem efeito.
    """

    # ---------- 1) correlação -> distância alvo (com epsilon e filtro de sinal) ----------
    C_raw = corr_matrix.to_numpy(dtype=float)
    C_raw = np.clip(C_raw, -1.0, 1.0)
    n = C_raw.shape[0]

    # máscara de r>0 (para epsilon)
    mask_pos = (C_raw > 0)

    if use_abs:
        C_eff = np.abs(C_raw)
    else:
        # aplica filtro de sinal
        if sign_mode == "positive":
            # mantém só r>0; demais vão para -1 (distância máxima 2R)
            C_eff = np.where(C_raw > 0, C_raw, -1.0)
        elif sign_mode == "negative":
            # mantém só r<0; demais vão para -1
            C_eff = np.where(C_raw < 0, C_raw, -1.0)
        else:
            C_eff = C_raw

    R = float(radius)
    T = (1.0 - C_eff) * R  # d(r) base
    # aplica ε apenas para r>0 (com base no sinal original, como pedido)
    if epsilon_frac and epsilon_frac > 0:
        T = T + (epsilon_frac * R) * mask_pos.astype(float)

    # simetriza e zera diagonal
    T = 0.5 * (T + T.T)
    np.fill_diagonal(T, 0.0)

    # ---------- 2) embedding por MDS (SMACOF; fallback p/ MDS clássico) ----------
    try:
        from sklearn.manifold import MDS
        mds = MDS(
            n_components=2, metric=True, dissimilarity='precomputed',
            n_init=n_init, max_iter=max_iter, random_state=random_state
        )
        X = mds.fit_transform(T)
    except Exception:
        # MDS clássico (Torgerson)
        J = np.eye(n) - np.ones((n, n))/n
        B = -0.5 * J @ (T**2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        L = np.clip(eigvals[idx][:2], 0, None)
        V = eigvecs[:, idx][:, :2]
        X = V * np.sqrt(L + 1e-12)

    # reescala global para casar melhor T
    DX = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)  # centraliza

    labels = corr_matrix.columns.tolist()

    # ---------- 3) paleta de cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed: k_auto += 1

    # ---------- 4) desenha círculos (mesmo raio) ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col,
                      alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 5) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=n)) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(n):
            for j in range(i+1, n):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # limites e estética
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

# ===== Exemplos =====
# 1) ambos os sinais, com epsilon:
# venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=4, sign_mode="both", use_abs=False)

# 2) só correlações positivas:
venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=0.5, sign_mode="positive", use_abs=False)

# 3) só correlações negativas:
# venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=4, sign_mode="negative", use_abs=False)


In [ ]:
import re
import unicodedata
import difflib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial.distance import squareform

# --- normalizador de nomes (case/acentos/espacos/simbolos) ---
def _normalize_name(s: str) -> str:
    s = str(s)
    # remove acentos
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    # remove marcadores como "(%)", "%", "()", etc
    s = re.sub(r'\(\s*%\s*\)', '', s)
    s = s.replace('%', '')
    # remove espaços, underscores e hifens
    s = re.sub(r'[\s_\-]+', '', s)
    # mantém apenas alfa-num
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def _resolve_metric_names(corr_columns, requested_names):
    """Mapeia nomes pedidos para colunas reais da corr_matrix via normalização + fuzzy."""
    # dicionário: normalizado -> nome original da coluna
    norm_to_col = {}
    for col in corr_columns:
        norm_to_col.setdefault(_normalize_name(col), col)

    resolved = []
    missing  = []
    suggestions = {}

    for name in requested_names:
        key = _normalize_name(name)
        if key in norm_to_col:
            resolved.append(norm_to_col[key])
        else:
            # tenta fuzzy contra as chaves normalizadas
            choices = list(norm_to_col.keys())
            match = difflib.get_close_matches(key, choices, n=1, cutoff=0.6)
            if match:
                resolved.append(norm_to_col[match[0]])
                suggestions[name] = norm_to_col[match[0]]
            else:
                missing.append(name)

    return resolved, missing, suggestions

def venn_from_corr_subset(
    corr_matrix,
    metrics=None,            # lista com os nomes desejados (em qualquer formato)
    radius=1.0,              # raio R (igual p/ todos)
    epsilon_pos_frac=0.06,   # ε (fração de R) só p/ pares com r>0
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,
    label_weight='normal',
    label_jitter=0.03,
    repel_iters=150,
    repel_step=0.02,
    inside_margin=0.12,
    title="Venn adaptado (subset; d = (1 − r)·R; ε>0 só para r>0)"
):
    # ---------- 0) resolver nomes do subset ----------
    if metrics is not None:
        resolved, missing, sugg = _resolve_metric_names(corr_matrix.columns, metrics)
        if missing and not resolved:
            cols_list = ', '.join(map(str, corr_matrix.columns))
            raise ValueError(
                "Nenhuma métrica do subset encontrada nas colunas da corr_matrix.\n"
                f"Pedidas: {metrics}\n"
                f"Colunas disponíveis: {cols_list}"
            )
        if missing:  # avisa mas segue com as resolvidas
            print("[aviso] Métricas não encontradas e ignoradas:", missing)
            if sugg:
                print("[aviso] Sugestões aplicadas:", sugg)
        if not resolved:
            raise ValueError("Após normalização, não sobrou nenhuma métrica válida para plotar.")
        labels = resolved
        C_raw = corr_matrix.loc[labels, labels].to_numpy(dtype=float)
    else:
        labels = corr_matrix.columns.tolist()
        C_raw = corr_matrix.to_numpy(dtype=float)

    # ---------- 1) correlação → distância-alvo ----------
    C_raw = np.clip(C_raw, -1.0, 1.0)
    R = float(radius)

    # d(r) base
    T = (1.0 - C_raw) * R

    # aplica ε apenas quando r>0 (não aplica em r<=0)
    if epsilon_pos_frac and epsilon_pos_frac > 0:
        mask_pos = (C_raw > 0).astype(float)
        T += (epsilon_pos_frac * R) * mask_pos

    # simetriza e zera diagonal
    np.fill_diagonal(T, 0.0)
    T = 0.5 * (T + T.T)

    # ---------- 2) embedding (MDS SMACOF; fallback p/ clássico) ----------
    try:
        from sklearn.manifold import MDS
        mds = MDS(n_components=2, metric=True, dissimilarity='precomputed',
                  n_init=n_init, max_iter=max_iter, random_state=random_state)
        X = mds.fit_transform(T)
    except Exception:
        n = T.shape[0]
        J = np.eye(n) - np.ones((n, n))/n
        B = -0.5 * J @ (T**2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        L = np.clip(eigvals[idx][:2], 0, None)
        V = eigvecs[:, idx][:, :2]
        X = V * np.sqrt(L + 1e-12)

    # reescala global para aproximar T
    DX = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)

    # ---------- 3) cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed: k_auto += 1

    # ---------- 4) círculos ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col, alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 5) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=len(labels))) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(len(labels)):
            for j in range(i+1, len(labels)):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # ---------- 6) estética ----------
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

# ===== Exemplo de uso =====
subset = ["Robustness","Fidelity", "Infidelity"]
venn_from_corr_subset(corr_matrix,
                      metrics=subset,
                      radius=1.0,
                      epsilon_pos_frac=0.06,
                      figsize=(8,8),
                      label_fontsize=8)


In [ ]:
# ===  Shape Interpretabilidade [Gráfico de Radar das Instâncias]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil

# Filtra as instâncias, removendo a linha de média
# Filtra as instâncias, removendo a linha de média
df_shap_metrics_filter = df_shap_metrics[df_shap_metrics["instancia"] != "MÉDIA"].copy()
df_shap_metrics_filter = df_shap_metrics_filter[df_shap_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_shap_metrics_filter.columns
                  if df_shap_metrics[col].nunique() == 1 and (df_shap_metrics[col].iloc[0] == '100.0%' or df_shap_metrics[col].iloc[0] == '0.0%')]

# df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')


# Lista de métricas e ângulos
metrics = [col for col in df_shap_metrics_filter.columns if col != 'instancia']
num_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]  # fecha o polígono

# Layout do PDF
plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_shap_metrics_filter)
pages = ceil(total_instances / plots_per_page)

# Nome do arquivo de saída
pdf_path = "graficos_radar_angular_limpo.pdf"
pdf = PdfPages(pdf_path)

# Loop de páginas
for page in range(pages):
    fig, axs = plt.subplots(rows_per_page, plots_per_row, figsize=(8.27, 11.69), subplot_kw=dict(polar=True))
    fig.suptitle("Análise de Métricas XAI por Instâncias", fontsize=14, y=0.96, color='navy')
    axs = axs.flatten()

    for i in range(plots_per_page):
        idx = page * plots_per_page + i
        if idx >= total_instances:
            axs[i].axis('off')
            continue

        row = df_shap_metrics_filter.iloc[idx]
        values = [float(str(row[m]).replace('', '')) / 100 for m in metrics]
        values += values[:1]  # fecha o polígono

        ax = axs[i]

        # --- radar da instância ---
        ax.plot(angles, values, linewidth=1.25, linestyle='solid', color='navy', zorder=3)
        ax.fill(angles, values, color='cornflowerblue', alpha=0.32, zorder=2)

        # --- remover grades e valores radiais ---
        ax.grid(False)
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)
        ax.set_yticks([])                       # sem círculos/valores radiais
        ax.spines['polar'].set_visible(False)
        ax.set_rlabel_position(0)

        # --- limites (um pouco acima de 1 para dar espaço aos rótulos) ---
        ax.set_ylim(0, 1.05)

        # --- rótulos das métricas (negrito + mais espaçados) ---
        # escondemos os ticks padrão e desenhamos rótulos manualmente levemente fora do raio 1
        ax.set_xticks([])
        label_r = 1.03
        for theta, label in zip(angles[:-1], metrics):
            ax.text(theta, label_r, label,
                    ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    clip_on=False)

        # --- polígono externo (mesmo nº de lados que as métricas) ---
        outer_r = [1.0] * len(angles)           # angles já está fechado
        ax.fill(angles, outer_r, color='#f1f1f1', zorder=0)
        ax.plot(angles, outer_r, linewidth=1.0, color='#444', zorder=1)

        # --- título ---
        ax.set_title(f"Instância {int(row['instancia'])}", size=9, pad=8)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()
    # pdf.savefig(fig)
    # plt.close()

pdf.close()
print(f"✅ PDF salvo com sucesso em: {pdf_path}")


In [ ]:
# ===  Shap Interpretabilidade [Gráfico de Radar Global]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===== Atualização: gráfico único com todas as instâncias sobrepostas =====
df_shap_metrics_filter = df_shap_metrics[df_shap_metrics["instancia"] != "MÉDIA"].copy()
df_shap_metrics_filter = df_shap_metrics_filter[df_shap_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_shap_metrics_filter.columns
                  if df_shap_metrics[col].nunique() == 1 and (df_shap_metrics[col].iloc[0] == '100.0%' or df_shap_metrics[col].iloc[0] == '0.0%')]

# df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# Métricas e ângulos
metrics = [col for col in df_shap_metrics_filter.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# Gráfico único
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Todas as Instâncias sobrepostas", fontsize=16, color='darkorange')

# Rótulos e grades
rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# Plotar cada instância
for _, row in df_shap_metrics_filter.iterrows():
    values = [float(str(row[m]).replace('%', '')) for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.2, linestyle='solid', alpha=0.5, label=f"Instância {int(row['instancia'])}")
    ax.fill(angles, values, alpha=0.05, color='orange')

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ===  Shapley Interpretabilidade [Gráfico de Radar por Intrancias e Grupos de Métricas]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil

# ===== 1. Filtrar e preparar =====
df_shap_metrics_filter = df_shap_metrics[df_shap_metrics["instancia"] != "MÉDIA"].copy()
df_shap_metrics_filter = df_shap_metrics_filter[df_shap_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_shap_metrics_filter.columns
                  if df_shap_metrics[col].nunique() == 1 and (df_shap_metrics[col].iloc[0] == '100.0%' or df_shap_metrics[col].iloc[0] == '0.0%')]

# df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# ===== 2. Definir grupos =====
grupos = {
    "Fidelidade": [
        "fidelity(%)",
        "infidelity(%)",
        "faithfulness(%)",
        "soundness(%)",
        "directional soundness(%)",
    ],
    "Relevância": [
        "selectivity(%)",  # 'relevance(%)' não aparece no seu dataset
    ],
    "Simplicidade": [
        "simplicity(%)",
    ],
    "Robustez": [
        "robustez(%)",
        "stability(%)",    # 'robustez_adversarial(%)' não disponível
    ],
    "Generalização": [
        # "completeness(%)",
        "consistency(%)",  # no seu dataset vem com '%'
        "sufficiency-1(%)",
        "sufficiency-5(%)",
        "sufficiency-10(%)",
        "sufficiency-20(%)",
    ],
}

# Verificar quais colunas estão no DataFrame
for k, v in grupos.items():
    grupos[k] = [col for col in v if col in df_shap_metrics_filter.columns]

# ===== 3. Calcular médias por grupo =====
df_grupos = pd.DataFrame()
df_grupos['instancia'] = df_shap_metrics_filter['instancia']

for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_shap_metrics_filter[cols].replace('%', '', regex=True).astype(float).mean(axis=1)

# ===== 4. Gerar gráfico radar com médias por grupo =====

# Lista de métricas agregadas e ângulos
metrics = [col for col in df_grupos.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# Layout
plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_grupos)
pages = ceil(total_instances / plots_per_page)
rgrid_values = [10, 30, 50, 70, 90]

# Saída PDF
pdf_path = "graficos_teia_laranja_por_grupo.pdf"
pdf = PdfPages(pdf_path)

for page in range(pages):
    fig, axs = plt.subplots(rows_per_page, plots_per_row, figsize=(8.27, 11.69), subplot_kw=dict(polar=True))
    fig.suptitle("Análise de Grupos de Métricas XAI por Instância", fontsize=14, y=0.96, color='darkorange')
    axs = axs.flatten()

    for i in range(plots_per_page):
        idx = page * plots_per_page + i
        if idx >= total_instances:
            axs[i].axis('off')
            continue

        row = df_grupos.iloc[idx]
        raw_values = [row[m] for m in metrics]
        values = raw_values + raw_values[:1]

        ax = axs[i]
        ax.plot(angles, values, linewidth=2, linestyle='solid', color='darkorange')
        ax.fill(angles, values, color='orange', alpha=0.3)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics, fontsize=7)

        # Melhor espaçamento dos labels angulares
        for label in ax.get_xticklabels():
            label.set_horizontalalignment('center')
            label.set_fontsize(6)
            label.set_rotation(15)

        ax.set_yticks(rgrid_values)
        ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=5)
        ax.set_rlabel_position(0)
        ax.set_title(f"Instância {row['instancia']}", size=9, pad=10, color='darkorange')
        ax.spines['polar'].set_visible(True)
        ax.spines['polar'].set_color('darkorange')
        ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()
    # pdf.savefig(fig)
    # plt.close()

pdf.close()
print(f"PDF salvo em: {pdf_path}")


In [ ]:
# ===  Shapley Interpretabilidade [Gráfico de Radar por Intrancias e Global por Grupos de Métricas]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_shap_metrics_filter = df_shap_metrics[df_shap_metrics["instancia"] != "MÉDIA"].copy()
df_shap_metrics_filter = df_shap_metrics_filter[df_shap_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')


cols_to_remove = [col for col in df_shap_metrics_filter.columns
                  if df_shap_metrics[col].nunique() == 1 and (df_shap_metrics[col].iloc[0] == '100.0%' or df_shap_metrics[col].iloc[0] == '0.0%')]

# df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + ['instancia', 'completeness(%)', 'monotony(%)', 'relevance(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.drop(columns=cols_to_remove + [ 'completeness(%)'])
df_shap_metrics_filter = df_shap_metrics_filter.replace('%', '', regex=True).apply(pd.to_numeric, errors='ignore')

# ===== 2. Definir grupos =====
# ===== 2. Definir grupos =====
grupos = {
    "Fidelidade": [
        "fidelity(%)",
        "infidelity(%)",
        "faithfulness(%)",
        "soundness(%)",
        "directional soundness(%)",
    ],
    "Relevância": [
        "selectivity(%)",  # 'relevance(%)' não aparece no seu dataset
    ],
    "Simplicidade": [
        "simplicity(%)",
    ],
    "Robustez": [
        "robustez(%)",
        "stability(%)",    # 'robustez_adversarial(%)' não disponível
    ],
    "Generalização": [
        # "completeness(%)",
        "consistency(%)",  # no seu dataset vem com '%'
        "sufficiency-1(%)",
        "sufficiency-5(%)",
        "sufficiency-10(%)",
        "sufficiency-20(%)",
    ],
}

# Verificar colunas válidas
for k, v in grupos.items():
    grupos[k] = [col for col in v if col in df_shap_metrics_filter.columns]

# ===== 3. Calcular médias por grupo =====
df_grupos = pd.DataFrame()
df_grupos['instancia'] = df_shap_metrics_filter['instancia']

for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_shap_metrics_filter[cols].replace('%', '', regex=True).astype(float).mean(axis=1)

# ===== 4. Gráfico único com todas as instâncias =====
metrics = [col for col in df_grupos.columns if col != 'instancia']
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Todas as Instâncias sobrepostas (Agrupadas por Métricas)", fontsize=16, color='darkorange')

rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# Plotar todas as instâncias agrupadas
for _, row in df_grupos.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, linestyle='solid', alpha=0.5, label=f"Instância {int(row['instancia'])}")
    ax.fill(angles, values, alpha=0.05, color='orange')

ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# === UpSet plot das métricas shap (sobreposição por instância) ===
# Usa df_shap_metrics já existente. Salva em BASE_PATH / "upset_shap_metricas.png"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_indicators
except Exception:
    raise ImportError("Instale o pacote 'upsetplot': pip install upsetplot")

# ----------------- CONFIG -----------------
THRESHOLD_ABS = 60.0       # limiar absoluto em %
USE_PERCENTILE = True     # se True, usa percentil por métrica
PERCENTILE_Q = 75

# BASE_PATH seguro (aceita str ou Path; se não existir, usa cwd)
SAFE_BASE = Path(BASE_PATH) if 'BASE_PATH' in globals() else Path.cwd()
OUTFILE   = SAFE_BASE / "upset_shap_metricas.png"

# ----------------- PREPARO -----------------
# preserva média/mediana numa cópia separada (para referência)
agg_rows = df_shap_metrics[
    df_shap_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# trabalha numa cópia sem média/mediana para o UpSet
dfu = df_shap_metrics[
    ~df_shap_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# pega colunas de métricas "(%)"
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)'.")

# “xx.x%” -> float
M = (
    dfu[metric_cols]
    .replace('%', '', regex=True)
    .replace('N/A', np.nan)
    .apply(pd.to_numeric, errors='coerce')
)

# limiares por métrica
if USE_PERCENTILE:
    thresholds = M.quantile(PERCENTILE_Q / 100.0, numeric_only=True)
else:
    thresholds = pd.Series(THRESHOLD_ABS, index=M.columns, dtype=float)

# indicadores booleanos: métrica >= limiar
indicators = M.ge(thresholds, axis=1)

# constrói dados para o UpSet
data_upset = from_indicators(indicators=indicators.columns.tolist(), data=indicators)

# ----------------- PLOT -----------------
plt.figure(figsize=(12, 7), dpi=150)
up = UpSet(
    data_upset,
    show_counts=True,
    sort_by="cardinality",
    sort_categories_by="cardinality",
    element_size=28
)
up.plot()

title = (f"UpSet shap — interseções de métricas ≥ P{PERCENTILE_Q}"
         if USE_PERCENTILE else
         f"UpSet shap — interseções de métricas ≥ {THRESHOLD_ABS:.0f}%")
plt.suptitle(title, y=1.02)

plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
plt.show()
print(f"✅ UpSet salvo em: {OUTFILE}")
# opcional: acessar agg_rows se quiser consultar média/mediana preservadas


In [ ]:
# === UpSet de Correlação entre Métricas shap ===
# Nomes das métricas à esquerda (padrão) + replicados na base (vertical)
# Requer: df_shap_metrics e (opcional) BASE_PATH (str ou Path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    raise ImportError("Instale: pip install upsetplot")

# ----------------- CONFIG -----------------
TAU = 0.30        # limiar de |correlação| p/ considerar “alta correlação”
MIN_GROUP = 2     # grupos com pelo menos 2 métricas
SAFE_BASE = Path(BASE_PATH) if 'BASE_PATH' in globals() else Path.cwd()
OUTFILE = SAFE_BASE / "upset_shap_corr.png"

# ----------------- PREPARO -----------------
# remove média/mediana
dfu = df_shap_metrics[~df_shap_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA","MEDIANA"])].copy()

# seleciona métricas "(%)" e converte p/ float
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)'.")

M = (dfu[metric_cols]
     .replace('%', '', regex=True)
     .replace('N/A', np.nan)
     .apply(pd.to_numeric, errors='coerce'))

# correlação coluna-coluna
corr = M.corr(method="pearson")

# memberships: para cada métrica i, conjunto das que têm |r| ≥ TAU com i
metrics = corr.columns.tolist()
memberships, labels_anchor = [], []
for i in metrics:
    group = [j for j in metrics if j != i and np.isfinite(corr.loc[i, j]) and abs(corr.loc[i, j]) >= TAU]
    if len(group) >= MIN_GROUP:
        memberships.append(group)
        labels_anchor.append(i)

if not memberships:
    print(f"Nenhum grupo de métricas com |r| ≥ {TAU:.2f}. Tente reduzir TAU.")
else:
    data = from_memberships(memberships)

    # ----------------- PLOT (horizontal: nomes à esquerda) -----------------
    fig = plt.figure(figsize=(22, 10), dpi=300)
    up = UpSet(
        data,
        show_counts=True,
        sort_by="cardinality",
        sort_categories_by="cardinality",
        element_size=30,
        orientation="horizontal"  # mantém nomes à esquerda
    )
    up.plot()

    # --------- Faixa auxiliar com nomes das métricas na base ----------
    # Encontrar, de forma robusta, o eixo que contém os rótulos Y das métricas
    fig = plt.gcf()
    metric_set = set(metrics)
    left_axis = None
    for ax in fig.axes:
        ytxt = [t.get_text() for t in ax.get_yticklabels()]
        match_count = sum(1 for t in ytxt if t in metric_set)
        if match_count >= max(3, len(metric_set) // 2):  # heurística robusta
            left_axis = ax
            break
    if left_axis is None:
        # fallback: usa o eixo com mais rótulos de métricas
        best_ax, best_score = None, -1
        for ax in fig.axes:
            score = sum(1 for t in ax.get_yticklabels() if t.get_text() in metric_set)
            if score > best_score:
                best_ax, best_score = ax, score
        left_axis = best_ax

    # Se ainda assim não achou, evita crash
    if left_axis is not None:
        pos = left_axis.get_position()  # bbox do eixo da matriz

        # >>> AJUSTES APENAS NOS LABELS DA PARTE INFERIOR <<<
        gutter_h = 0.10  # (↑) aumenta a altura da faixa inferior para não sobrepor
        bottom = max(0.02, pos.y0 - gutter_h - 0.02)  # (↓) desce um pouco mais a faixa
        ax_bottom = fig.add_axes([pos.x0, bottom, pos.width, gutter_h])  # largura 100% da matriz
        ax_bottom.set_xlim(0, 1)
        ax_bottom.set_ylim(-0.2, 1.0)  # (↑) permite texto abaixo de 0
        ax_bottom.axis("off")

        # ordem dos nomes conforme aparecem à esquerda
        ylabels = [lab.get_text() for lab in left_axis.get_yticklabels() if lab.get_text()]
        n = len(ylabels)
        if n > 0:
            xs = np.linspace(0.0, 1.0, n, endpoint=True)
            for x, name in zip(xs, ylabels):
                # y=-0.12 coloca o texto abaixo da linha, evitando sobreposição
                ax_bottom.text(x, -0.12, name, rotation=90, ha="center", va="top", fontsize=10)

    plt.suptitle(f"UpSet shap — grupos de métricas com |corr| ≥ {TAU:.2f}", y=0.98)
    plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
    plt.show()
    print(f"✅ UpSet (correlação) salvo em: {OUTFILE}")
